In [1]:
from pathlib import Path

# AthletiQ project root
PROJECT_ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

SPORTSMOT = PROJECT_ROOT / "datasets/football/benchmarks/sportsmot/extracted"

print("SportsMOT path:")
print(SPORTSMOT)

print("\nExists:", SPORTSMOT.exists())

if not SPORTSMOT.exists():
    raise FileNotFoundError(f"SportsMOT dataset not found: {SPORTSMOT}")

print("\nSplits:")
for split in ["train", "val", "test"]:
    split_dir = SPORTSMOT / split

    if split_dir.exists():
        sequences = [p for p in split_dir.iterdir() if p.is_dir()]
        print(f"{split:5s}: {len(sequences):3d} sequences")
    else:
        print(f"{split:5s}: NOT FOUND")

SportsMOT path:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted

Exists: True

Splits:
train:  15 sequences
val  :  15 sequences
test :  50 sequences


In [2]:
from pathlib import Path

VAL_DIR = SPORTSMOT / "val"

# Pick the first sequence
seq = sorted([p for p in VAL_DIR.iterdir() if p.is_dir()])[0]

print("Selected sequence:", seq.name)
print("\nContents:")

for item in sorted(seq.iterdir()):
    if item.is_dir():
        print(f"📁 {item.name}/")
    else:
        print(f"📄 {item.name}")

Selected sequence: v_2QhNRucNC7E_c017

Contents:
📁 gt/
📁 img1/
📄 seqinfo.ini


In [3]:
SEQINFO = seq / "seqinfo.ini"
GT_FILE = seq / "gt" / "gt.txt"
IMG_DIR = seq / "img1"

print("\nImage directory:", IMG_DIR)
print("Images:", len(list(IMG_DIR.glob("*.jpg"))))

print("\n--- seqinfo.ini ---")
print(SEQINFO.read_text())

print("\n--- First 10 GT lines ---")
with open(GT_FILE, "r") as f:
    for _ in range(10):
        print(f.readline().strip())


Image directory: /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017/img1
Images: 450

--- seqinfo.ini ---
[Sequence]
name=v_2QhNRucNC7E_c017
imDir=img1
frameRate=25
seqLength=450
imWidth=1280
imHeight=720
imExt=.jpg

--- First 10 GT lines ---
1, 0, 509, 458, 36, 92, 1, 1, 1
1, 1, 874, 398, 32, 81, 1, 1, 1
1, 2, 931, 208, 26, 58, 1, 1, 1
1, 3, 197, 313, 51, 65, 1, 1, 1
1, 4, 400, 223, 23, 55, 1, 1, 1
1, 5, 358, 207, 26, 53, 1, 1, 1
1, 6, 8, 470, 38, 104, 1, 1, 1
1, 7, 292, 195, 32, 55, 1, 1, 1
1, 8, 0, 231, 27, 66, 1, 1, 1
1, 10, 778, 386, 39, 82, 1, 1, 1


In [4]:
from ultralytics import YOLO
import torch

MODEL_PATH = PROJECT_ROOT / "ai-engine/models/football_player_detector.pt"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

player_model = YOLO(str(MODEL_PATH))

print("\nModel loaded successfully.")
print("Task:", player_model.task)
print("Classes:", player_model.names)

PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU

Model loaded successfully.
Task: detect
Classes: {0: 'player'}


In [5]:
# Read the first frame
first_frame = sorted(IMG_DIR.glob("*.jpg"))[0]

result = player_model.predict(
    source=str(first_frame),
    conf=0.25,
    imgsz=1280,
    device=0,
    verbose=False
)[0]

print("Frame:", first_frame.name)
print("Detections:", len(result.boxes))

if len(result.boxes) > 0:
    print("\nFirst 5 detections:")
    for box in result.boxes[:5]:
        xyxy = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cls = int(box.cls[0])

        print(
            f"bbox={xyxy.astype(int)}, "
            f"conf={conf:.3f}, "
            f"class={cls}"
        )

Frame: 000001.jpg
Detections: 22

First 5 detections:
bbox=[776 385 816 467], conf=0.959, class=0
bbox=[953 240 992 307], conf=0.955, class=0
bbox=[1073  303 1100  376], conf=0.952, class=0
bbox=[509 459 544 549], conf=0.950, class=0
bbox=[505 378 536 465], conf=0.944, class=0


In [6]:
from types import SimpleNamespace
from ultralytics.trackers.byte_tracker import BYTETracker

tracker_args = SimpleNamespace(
    track_high_thresh=0.5,
    track_low_thresh=0.1,
    new_track_thresh=0.5,
    track_buffer=30,
    match_thresh=0.8,
    fuse_score=True
)

tracker = BYTETracker(tracker_args)

print("ByteTrack initialized successfully.")

ByteTrack initialized successfully.


In [7]:
# Run ByteTrack through Ultralytics' built-in tracking pipeline
track_result = player_model.track(
    source=str(first_frame),
    tracker="bytetrack.yaml",
    conf=0.25,
    imgsz=1280,
    device=0,
    persist=True,
    verbose=False
)[0]

print("Frame:", first_frame.name)
print("Detections:", len(track_result.boxes))

if track_result.boxes.id is not None:
    track_ids = track_result.boxes.id.cpu().numpy().astype(int)
    print("Track IDs:", track_ids)
    print("Number of tracks:", len(track_ids))
else:
    print("No track IDs returned.")

Frame: 000001.jpg
Detections: 22
Track IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
Number of tracks: 22


In [8]:
from collections import defaultdict

# Fresh tracker state through Ultralytics
track_history = defaultdict(list)

results = player_model.track(
    source=[str(p) for p in sorted(IMG_DIR.glob("*.jpg"))[:30]],
    tracker="bytetrack.yaml",
    conf=0.25,
    imgsz=1280,
    device=0,
    persist=True,
    verbose=False
)

for frame_idx, r in enumerate(results, start=1):
    if r.boxes.id is not None:
        ids = r.boxes.id.cpu().numpy().astype(int)
        track_history[frame_idx] = ids.tolist()

        print(
            f"Frame {frame_idx:02d}: "
            f"{len(ids):2d} detections | "
            f"IDs: {ids.tolist()}"
        )
    else:
        print(f"Frame {frame_idx:02d}: 0 tracks")

print("\nUnique track IDs:", len(set(
    tid for ids in track_history.values() for tid in ids
)))

Frame 01: 22 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Frame 02: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 19, 20, 21]
Frame 03: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 04: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 05: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 06: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 07: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 08: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 09: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11]
Frame 10: 19 detections | IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 16, 17, 19, 20, 21, 11

In [9]:
import gc
import torch

# Remove current model/tracking objects from GPU
del player_model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU allocated after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved after cleanup:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU allocated after cleanup: 0.03 GB
GPU reserved after cleanup:  0.06 GB


In [10]:
from pathlib import Path
import torch

PROJECT_ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")
SPORTSMOT = PROJECT_ROOT / "datasets/football/benchmarks/sportsmot/extracted"

VAL_DIR = SPORTSMOT / "val"
seq = VAL_DIR / "v_2QhNRucNC7E_c017"

IMG_DIR = seq / "img1"
GT_FILE = seq / "gt" / "gt.txt"

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

print("\nSequence:", seq.name)
print("Frames:", len(list(IMG_DIR.glob("*.jpg"))))

GPU available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory allocated: 0.03 GB

Sequence: v_2QhNRucNC7E_c017
Frames: 450


In [11]:
from ultralytics import YOLO

MODEL_PATH = PROJECT_ROOT / "ai-engine/models/football_player_detector.pt"

player_model = YOLO(str(MODEL_PATH))

print("Model loaded successfully.")
print("Task:", player_model.task)
print("Classes:", player_model.names)
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Model loaded successfully.
Task: detect
Classes: {0: 'player'}
GPU memory allocated: 0.03 GB


In [12]:
import time
import pandas as pd

start_time = time.time()
tracking_rows = []

results = player_model.track(
    source=str(IMG_DIR),       # directory, NOT list of images
    tracker="bytetrack.yaml",
    conf=0.25,
    imgsz=1280,
    device=0,
    persist=True,
    stream=True,
    verbose=False
)

for frame_idx, r in enumerate(results, start=1):

    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        ids = r.boxes.id.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for bbox, track_id, conf in zip(boxes, ids, confs):
            x1, y1, x2, y2 = bbox

            tracking_rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "confidence": conf
            })

    if frame_idx % 50 == 0:
        print(f"Processed {frame_idx}/450 frames")

elapsed = time.time() - start_time

tracks_df = pd.DataFrame(tracking_rows)

print("\n========== TRACKING COMPLETE ==========")
print(f"Frames processed : {frame_idx}")
print(f"Tracking rows    : {len(tracks_df):,}")
print(f"Unique track IDs : {tracks_df['track_id'].nunique()}")
print(f"Time             : {elapsed:.2f} sec")
print(f"FPS              : {frame_idx / elapsed:.2f}")

Processed 50/450 frames
Processed 100/450 frames
Processed 150/450 frames
Processed 200/450 frames
Processed 250/450 frames
Processed 300/450 frames
Processed 350/450 frames
Processed 400/450 frames
Processed 450/450 frames

========== TRACKING COMPLETE ==========
Frames processed : 450
Tracking rows    : 6,757
Unique track IDs : 69
Time             : 48.04 sec
FPS              : 9.37


In [13]:
track_stats = (
    tracks_df
    .groupby("track_id")
    .agg(
        frames_tracked=("frame", "nunique"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        avg_confidence=("confidence", "mean")
    )
    .sort_values("frames_tracked", ascending=False)
)

print("Total unique tracks:", len(track_stats))

print("\nTrack duration statistics:")
print(track_stats["frames_tracked"].describe())

print("\nTop 20 longest tracks:")
display(track_stats.head(20))

print("\nShort tracks (<= 10 frames):",
      (track_stats["frames_tracked"] <= 10).sum())

print("Tracks lasting > 100 frames:",
      (track_stats["frames_tracked"] > 100).sum())

print("Tracks lasting > 300 frames:",
      (track_stats["frames_tracked"] > 300).sum())

Total unique tracks: 69

Track duration statistics:
count     69.000000
mean      97.927536
std      129.860029
min        1.000000
25%        3.000000
50%       48.000000
75%      139.000000
max      419.000000
Name: frames_tracked, dtype: float64

Top 20 longest tracks:


,frames_tracked,first_frame,last_frame,avg_confidence
track_id,,,,
19,419,1,450,0.874056
1,403,1,403,0.946143
13,379,1,389,0.899606
8,375,1,381,0.922716
17,373,1,375,0.891045
21,366,1,391,0.907755
12,364,1,378,0.875920
20,358,1,379,0.833786
4,357,1,358,0.946149



Short tracks (<= 10 frames): 29
Tracks lasting > 100 frames: 20
Tracks lasting > 300 frames: 9


In [14]:
# Load SportsMOT ground truth
gt = pd.read_csv(
    GT_FILE,
    header=None,
    names=[
        "frame",
        "track_id",
        "x",
        "y",
        "width",
        "height",
        "conf",
        "class",
        "visibility"
    ]
)

print("GT rows:", len(gt))
print("GT unique IDs:", gt["track_id"].nunique())
print("GT frames:", gt["frame"].nunique())

print("\nGT players per frame:")
print(
    gt.groupby("frame")["track_id"]
    .nunique()
    .describe()
)

print("\nFirst 10 GT rows:")
display(gt.head(10))

GT rows: 6622
GT unique IDs: 21
GT frames: 450

GT players per frame:
count    450.000000
mean      14.715556
std        1.907430
min       11.000000
25%       13.000000
50%       14.500000
75%       16.000000
max       19.000000
Name: track_id, dtype: float64

First 10 GT rows:


,frame,track_id,x,y,width,height,conf,class,visibility
0,1,0,509,458,36,92,1,1,1
1,1,1,874,398,32,81,1,1,1
2,1,2,931,208,26,58,1,1,1
3,1,3,197,313,51,65,1,1,1
4,1,4,400,223,23,55,1,1,1
5,1,5,358,207,26,53,1,1,1
6,1,6,8,470,38,104,1,1,1
7,1,7,292,195,32,55,1,1,1
8,1,8,0,231,27,66,1,1,1
9,1,10,778,386,39,82,1,1,1


In [15]:
import numpy as np

def calculate_iou(box_a, box_b):
    """
    box format: [x1, y1, x2, y2]
    """
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


# Convert GT boxes to x1,y1,x2,y2
gt_eval = gt.copy()

gt_eval["x1"] = gt_eval["x"]
gt_eval["y1"] = gt_eval["y"]
gt_eval["x2"] = gt_eval["x"] + gt_eval["width"]
gt_eval["y2"] = gt_eval["y"] + gt_eval["height"]

IOU_THRESHOLD = 0.5

matches = []

for frame in range(1, 451):

    gt_frame = gt_eval[gt_eval["frame"] == frame]
    pred_frame = tracks_df[tracks_df["frame"] == frame]

    used_preds = set()

    # Match each GT player to the best unused prediction
    for _, gt_row in gt_frame.iterrows():

        gt_box = [
            gt_row["x1"],
            gt_row["y1"],
            gt_row["x2"],
            gt_row["y2"]
        ]

        best_iou = 0
        best_pred_idx = None

        for pred_idx, pred_row in pred_frame.iterrows():

            if pred_idx in used_preds:
                continue

            pred_box = [
                pred_row["x1"],
                pred_row["y1"],
                pred_row["x2"],
                pred_row["y2"]
            ]

            iou = calculate_iou(gt_box, pred_box)

            if iou > best_iou:
                best_iou = iou
                best_pred_idx = pred_idx

        if best_iou >= IOU_THRESHOLD:
            used_preds.add(best_pred_idx)

            matches.append({
                "frame": frame,
                "gt_id": int(gt_row["track_id"]),
                "pred_id": int(
                    pred_frame.loc[best_pred_idx, "track_id"]
                ),
                "iou": best_iou
            })

matches_df = pd.DataFrame(matches)

print("GT annotations:", len(gt))
print("Predicted tracks:", len(tracks_df))
print("Successful matches:", len(matches_df))

print(
    f"Match rate: "
    f"{len(matches_df) / len(gt) * 100:.2f}%"
)

print(
    f"Mean matched IoU: "
    f"{matches_df['iou'].mean():.3f}"
)

print("\nFirst 10 matches:")
display(matches_df.head(10))

GT annotations: 6622
Predicted tracks: 6757
Successful matches: 6249
Match rate: 94.37%
Mean matched IoU: 0.848

First 10 matches:


,frame,gt_id,pred_id,iou
0,1,0,4,0.963602
1,1,1,6,0.911591
2,1,2,7,0.871807
3,1,3,9,0.981912
4,1,4,14,0.972763
5,1,5,20,0.926516
6,1,6,10,0.954642
7,1,7,12,0.923101
8,1,8,16,0.982965
9,1,10,1,0.946463


In [16]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

IOU_THRESHOLD = 0.5
matches = []

for frame in range(1, 451):

    gt_frame = gt_eval[gt_eval["frame"] == frame].reset_index(drop=True)
    pred_frame = tracks_df[tracks_df["frame"] == frame].reset_index(drop=True)

    if len(gt_frame) == 0 or len(pred_frame) == 0:
        continue

    # Build IoU matrix
    iou_matrix = np.zeros((len(gt_frame), len(pred_frame)))

    for i, gt_row in gt_frame.iterrows():

        gt_box = [
            gt_row["x1"], gt_row["y1"],
            gt_row["x2"], gt_row["y2"]
        ]

        for j, pred_row in pred_frame.iterrows():

            pred_box = [
                pred_row["x1"], pred_row["y1"],
                pred_row["x2"], pred_row["y2"]
            ]

            iou_matrix[i, j] = calculate_iou(gt_box, pred_box)

    # Hungarian assignment maximizes IoU
    gt_indices, pred_indices = linear_sum_assignment(
        -iou_matrix
    )

    for gt_idx, pred_idx in zip(gt_indices, pred_indices):

        iou = iou_matrix[gt_idx, pred_idx]

        if iou >= IOU_THRESHOLD:

            matches.append({
                "frame": frame,
                "gt_id": int(gt_frame.loc[gt_idx, "track_id"]),
                "pred_id": int(pred_frame.loc[pred_idx, "track_id"]),
                "iou": float(iou)
            })

matches_df = pd.DataFrame(matches)

print("========== HUNGARIAN MATCHING ==========")
print("GT annotations:", len(gt))
print("Predicted detections:", len(tracks_df))
print("Successful matches:", len(matches_df))
print(f"Match rate: {len(matches_df) / len(gt) * 100:.2f}%")
print(f"Mean matched IoU: {matches_df['iou'].mean():.3f}")

print("\nMatches per frame:")
print(
    matches_df.groupby("frame")
    .size()
    .describe()
)

display(matches_df.head(10))

========== HUNGARIAN MATCHING ==========
GT annotations: 6622
Predicted detections: 6757
Successful matches: 6249
Match rate: 94.37%
Mean matched IoU: 0.848

Matches per frame:
count    450.000000
mean      13.886667
std        2.536688
min        5.000000
25%       13.000000
50%       14.000000
75%       16.000000
max       19.000000
dtype: float64


,frame,gt_id,pred_id,iou
0,1,0,4,0.963602
1,1,1,6,0.911591
2,1,2,7,0.871807
3,1,3,9,0.981912
4,1,4,14,0.972763
5,1,5,20,0.926516
6,1,6,10,0.954642
7,1,7,12,0.923101
8,1,8,16,0.982965
9,1,10,1,0.946463


In [17]:
# Sort matches chronologically
matches_sorted = matches_df.sort_values(
    ["gt_id", "frame"]
).reset_index(drop=True)

id_switches = 0
switch_details = []

for gt_id, group in matches_sorted.groupby("gt_id"):

    group = group.sort_values("frame")

    previous_pred = None
    previous_frame = None

    for _, row in group.iterrows():

        current_pred = int(row["pred_id"])
        current_frame = int(row["frame"])

        if previous_pred is not None:

            if current_pred != previous_pred:
                id_switches += 1

                switch_details.append({
                    "gt_id": int(gt_id),
                    "frame": current_frame,
                    "previous_pred_id": previous_pred,
                    "new_pred_id": current_pred
                })

        previous_pred = current_pred
        previous_frame = current_frame

switch_df = pd.DataFrame(switch_details)

print("========== ID SWITCH ANALYSIS ==========")
print("Total ID switches:", id_switches)
print("GT identities:", gt["track_id"].nunique())

if len(switch_df) > 0:
    print("\nFirst 20 ID switches:")
    display(switch_df.head(20))
else:
    print("\nNo ID switches found.")

========== ID SWITCH ANALYSIS ==========
Total ID switches: 86
GT identities: 21

First 20 ID switches:


,gt_id,frame,previous_pred_id,new_pred_id
0,0,371,4,155
1,0,389,155,296
2,1,191,6,106
3,2,155,7,89
4,2,351,89,146
5,2,377,146,127
6,2,378,127,106
7,2,381,106,189
8,2,382,189,196
9,2,384,196,225


In [18]:
print("========== FRAMES 360–400 ==========")

region = matches_df[
    matches_df["frame"].between(360, 400)
].copy()

summary = (
    region
    .groupby("frame")
    .agg(
        matched_gt=("gt_id", "nunique"),
        unique_pred=("pred_id", "nunique"),
        mean_iou=("iou", "mean")
    )
    .reset_index()
)

display(summary)

print("\nID changes in this region:")

region = region.sort_values(["gt_id", "frame"])

for gt_id, group in region.groupby("gt_id"):
    group = group.sort_values("frame")
    ids = group["pred_id"].tolist()

    changes = []
    for i in range(1, len(ids)):
        if ids[i] != ids[i-1]:
            changes.append(
                f"frame {group.iloc[i]['frame']}: "
                f"{ids[i-1]} → {ids[i]}"
            )

    if changes:
        print(f"\nGT player {gt_id}:")
        for change in changes:
            print(" ", change)

========== FRAMES 360–400 ==========


,frame,matched_gt,unique_pred,mean_iou
0,360,13,13,0.866177
1,361,13,13,0.877640
2,362,14,14,0.860621
3,363,14,14,0.839941
4,364,13,13,0.841636
5,365,13,13,0.847570
6,366,13,13,0.837645
7,367,13,13,0.873394
8,368,13,13,0.841310
9,369,13,13,0.830858



ID changes in this region:

GT player 0:
  frame 389.0: 155 → 296

GT player 2:
  frame 377.0: 146 → 127
  frame 378.0: 127 → 106
  frame 381.0: 106 → 189
  frame 382.0: 189 → 196
  frame 384.0: 196 → 225
  frame 388.0: 225 → 261

GT player 3:
  frame 381.0: 21 → 19
  frame 391.0: 19 → 156
  frame 395.0: 156 → 343

GT player 4:
  frame 381.0: 55 → 156
  frame 390.0: 156 → 309
  frame 399.0: 309 → 350
  frame 400.0: 350 → 309

GT player 5:
  frame 371.0: 20 → 156
  frame 383.0: 156 → 146
  frame 385.0: 146 → 127
  frame 386.0: 127 → 106
  frame 387.0: 106 → 13
  frame 395.0: 13 → 344

GT player 7:
  frame 380.0: 12 → 13
  frame 381.0: 13 → 8
  frame 385.0: 8 → 189
  frame 386.0: 189 → 196
  frame 388.0: 196 → 225
  frame 391.0: 225 → 318
  frame 398.0: 318 → 320

GT player 8:
  frame 395.0: 334 → 342
  frame 396.0: 342 → 346
  frame 400.0: 346 → 348

GT player 9:
  frame 371.0: 151 → 154

GT player 11:
  frame 377.0: 138 → 185

GT player 12:
  frame 397.0: 21 → 348

GT player 13:
  fra

In [19]:
from pathlib import Path

tracker_config = PROJECT_ROOT / "ai-engine/notebooks/bytetrack_sportsmot.yaml"

tracker_config.write_text("""
tracker_type: bytetrack

track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 60
match_thresh: 0.8
fuse_score: True
""")

print("Tracker config created:")
print(tracker_config)
print()
print(tracker_config.read_text())

Tracker config created:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/bytetrack_sportsmot.yaml


tracker_type: bytetrack

track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 60
match_thresh: 0.8
fuse_score: True



In [20]:
import time
import pandas as pd

start_time = time.time()
tracking_rows_60 = []

results_60 = player_model.track(
    source=str(IMG_DIR),
    tracker=str(tracker_config),
    conf=0.25,
    imgsz=1280,
    device=0,
    persist=True,
    stream=True,
    verbose=False
)

for frame_idx, r in enumerate(results_60, start=1):

    if r.boxes.id is None:
        continue

    boxes = r.boxes.xyxy.cpu().numpy()
    ids = r.boxes.id.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()

    for bbox, track_id, conf in zip(boxes, ids, confs):
        x1, y1, x2, y2 = bbox

        tracking_rows_60.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

    if frame_idx % 50 == 0:
        print(f"Processed {frame_idx}/450 frames")

elapsed_60 = time.time() - start_time

tracks_df_60 = pd.DataFrame(tracking_rows_60)

print("\n========== TRACKING BUFFER 60 ==========")
print(f"Frames processed : {frame_idx}")
print(f"Tracking rows    : {len(tracks_df_60):,}")
print(f"Unique track IDs : {tracks_df_60['track_id'].nunique()}")
print(f"Time             : {elapsed_60:.2f} sec")
print(f"FPS              : {frame_idx / elapsed_60:.2f}")

Processed 50/450 frames
Processed 100/450 frames
Processed 150/450 frames
Processed 200/450 frames
Processed 250/450 frames
Processed 300/450 frames
Processed 350/450 frames
Processed 400/450 frames
Processed 450/450 frames

========== TRACKING BUFFER 60 ==========
Frames processed : 450
Tracking rows    : 6,735
Unique track IDs : 69
Time             : 38.51 sec
FPS              : 11.69


In [21]:
matches_60 = []

for frame in range(1, 451):

    gt_frame = gt_eval[gt_eval["frame"] == frame].reset_index(drop=True)
    pred_frame = tracks_df_60[
        tracks_df_60["frame"] == frame
    ].reset_index(drop=True)

    if len(gt_frame) == 0 or len(pred_frame) == 0:
        continue

    iou_matrix = np.zeros(
        (len(gt_frame), len(pred_frame))
    )

    for i, gt_row in gt_frame.iterrows():

        gt_box = [
            gt_row["x1"], gt_row["y1"],
            gt_row["x2"], gt_row["y2"]
        ]

        for j, pred_row in pred_frame.iterrows():

            pred_box = [
                pred_row["x1"], pred_row["y1"],
                pred_row["x2"], pred_row["y2"]
            ]

            iou_matrix[i, j] = calculate_iou(
                gt_box,
                pred_box
            )

    gt_indices, pred_indices = linear_sum_assignment(
        -iou_matrix
    )

    for gt_idx, pred_idx in zip(gt_indices, pred_indices):

        iou = iou_matrix[gt_idx, pred_idx]

        if iou >= 0.5:
            matches_60.append({
                "frame": frame,
                "gt_id": int(gt_frame.loc[gt_idx, "track_id"]),
                "pred_id": int(
                    pred_frame.loc[pred_idx, "track_id"]
                ),
                "iou": float(iou)
            })

matches_df_60 = pd.DataFrame(matches_60)

print("========== BUFFER 60 EVALUATION ==========")
print("Successful matches:", len(matches_df_60))
print(
    f"Match rate: "
    f"{len(matches_df_60) / len(gt) * 100:.2f}%"
)
print(
    f"Mean IoU: "
    f"{matches_df_60['iou'].mean():.3f}"
)

========== BUFFER 60 EVALUATION ==========
Successful matches: 6229
Match rate: 94.07%
Mean IoU: 0.847


In [22]:
# Build per-GT-player matched prediction history
# Only count an ID switch when the GT player is matched
# in consecutive frames.

matches_sorted = matches_df.sort_values(
    ["gt_id", "frame"]
).reset_index(drop=True)

true_id_switches = 0
switch_events = []

for gt_id, group in matches_sorted.groupby("gt_id"):

    group = group.sort_values("frame").reset_index(drop=True)

    for i in range(1, len(group)):

        prev = group.iloc[i - 1]
        curr = group.iloc[i]

        # Only consecutive frames
        if curr["frame"] == prev["frame"] + 1:

            if curr["pred_id"] != prev["pred_id"]:

                true_id_switches += 1

                switch_events.append({
                    "gt_id": int(gt_id),
                    "frame": int(curr["frame"]),
                    "previous_pred_id": int(prev["pred_id"]),
                    "new_pred_id": int(curr["pred_id"])
                })

true_switch_df = pd.DataFrame(switch_events)

print("========== CONSECUTIVE-FRAME ID SWITCHES ==========")
print("ID switches:", true_id_switches)

if len(true_switch_df):
    display(true_switch_df.head(20))

========== CONSECUTIVE-FRAME ID SWITCHES ==========
ID switches: 41


,gt_id,frame,previous_pred_id,new_pred_id
0,2,378,127,106
1,2,382,189,196
2,4,71,14,55
3,4,72,55,14
4,4,78,14,55
5,4,399,309,350
6,4,400,350,309
7,5,386,127,106
8,5,387,106,13
9,7,381,13,8


In [23]:
TP = len(matches_df)
FP = len(tracks_df) - TP
FN = len(gt) - TP
IDSW = true_id_switches

MOTA = 1 - (FN + FP + IDSW) / len(gt)

print("========== PRELIMINARY TRACKING METRICS ==========")
print(f"True Positives : {TP}")
print(f"False Positives: {FP}")
print(f"False Negatives: {FN}")
print(f"ID Switches    : {IDSW}")
print(f"MOTA           : {MOTA:.4f} ({MOTA * 100:.2f}%)")

========== PRELIMINARY TRACKING METRICS ==========
True Positives : 6249
False Positives: 508
False Negatives: 373
ID Switches    : 41
MOTA           : 0.8608 (86.08%)


In [24]:
!pip install motmetrics


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [25]:
# Check whether motmetrics is installed

try:
    import motmetrics as mm
    print("motmetrics version:", mm.__version__)
except ImportError:
    print("motmetrics is NOT installed.")

motmetrics version: 1.4.0


In [26]:
import motmetrics as mm

acc = mm.MOTAccumulator(auto_id=True)

# Add frame-by-frame GT ↔ prediction distances
for frame in sorted(set(gt_eval["frame"]) | set(tracks_df["frame"])):
    gt_frame = gt_eval[gt_eval["frame"] == frame]
    pred_frame = tracks_df[tracks_df["frame"] == frame]

    gt_ids = gt_frame["track_id"].astype(int).tolist()
    pred_ids = pred_frame["track_id"].astype(int).tolist()

    gt_boxes = gt_frame[["x1", "y1", "x2", "y2"]].values
    pred_boxes = pred_frame[["x1", "y1", "x2", "y2"]].values

    distances = []

    for gt_box in gt_boxes:
        row = []
        for pred_box in pred_boxes:
            row.append(1 - calculate_iou(gt_box, pred_box))
        distances.append(row)

    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        distances = None
    else:
        distances = np.array(distances)
        distances[distances > 0.5] = np.nan

    acc.update(
        gt_ids,
        pred_ids,
        distances
    )

print("MOT accumulator created successfully.")

MOT accumulator created successfully.


In [27]:
mh = mm.metrics.create()

summary = mh.compute(
    acc,
    metrics=[
        "num_frames",
        "num_objects",
        "num_matches",
        "num_switches",
        "num_false_positives",
        "num_misses",
        "mota",
        "motp",
        "idf1",
    ],
    name="SportsMOT_v_2QhNRucNC7E_c017"
)

display(summary.T)

,SportsMOT_v_2QhNRucNC7E_c017
num_frames,450.000000
num_objects,6622.000000
num_matches,6182.000000
num_switches,67.000000
num_false_positives,508.000000
num_misses,373.000000
mota,0.856841
motp,0.152122
idf1,0.759399


In [28]:
VAL_DIR = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/"
    "datasets/football/benchmarks/sportsmot/extracted/val"
)

val_sequences = sorted([
    p for p in VAL_DIR.iterdir()
    if p.is_dir()
])

print("Total validation sequences:", len(val_sequences))

# Select 5 sequences spread through the validation set
selected_sequences = [
    val_sequences[i]
    for i in np.linspace(0, len(val_sequences) - 1, 5, dtype=int)
]

print("\nSelected sequences:")
for seq in selected_sequences:
    print("-", seq.name)

Total validation sequences: 15

Selected sequences:
- v_2QhNRucNC7E_c017
- v_G-vNjfx1GGc_c600
- v_ITo3sCnpw_k_c011
- v_dw7LOz17Omg_c067
- v_i2_L4qquVg0_c010


In [29]:
print("Detector:", player_model.ckpt_path)
print("Tracker config:", "default bytetrack.yaml")
print("Sequences:", len(selected_sequences))

for seq in selected_sequences:
    print(f"{seq.name}: {len(list((seq / 'img1').glob('*.jpg')))} frames")

Detector: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/models/football_player_detector.pt
Tracker config: default bytetrack.yaml
Sequences: 5
v_2QhNRucNC7E_c017: 450 frames
v_G-vNjfx1GGc_c600: 500 frames
v_ITo3sCnpw_k_c011: 776 frames
v_dw7LOz17Omg_c067: 450 frames
v_i2_L4qquVg0_c010: 758 frames


In [30]:
all_tracking_results = {}

for seq in selected_sequences:
    seq_name = seq.name
    img_dir = seq / "img1"

    print(f"\n{'='*60}")
    print(f"TRACKING: {seq_name}")
    print(f"{'='*60}")

    results = player_model.track(
        source=str(img_dir),
        tracker="bytetrack.yaml",
        persist=False,
        stream=True,
        verbose=False,
        conf=0.25,
        imgsz=1280,
        device=0
    )

    rows = []

    for frame_idx, result in enumerate(results, start=1):
        if result.boxes is None or result.boxes.id is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, track_id, confidence in zip(boxes, ids, confs):
            x1, y1, x2, y2 = box

            rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "confidence": float(confidence)
            })

    seq_tracks = pd.DataFrame(rows)
    all_tracking_results[seq_name] = seq_tracks

    print(f"Frames processed : {frame_idx}")
    print(f"Tracking rows    : {len(seq_tracks):,}")
    print(f"Unique track IDs : {seq_tracks['track_id'].nunique()}")


TRACKING: v_2QhNRucNC7E_c017
Frames processed : 450
Tracking rows    : 7,113
Unique track IDs : 23

TRACKING: v_G-vNjfx1GGc_c600
Frames processed : 500
Tracking rows    : 5,686
Unique track IDs : 17

TRACKING: v_ITo3sCnpw_k_c011
Frames processed : 776
Tracking rows    : 10,795
Unique track IDs : 22

TRACKING: v_dw7LOz17Omg_c067
Frames processed : 450
Tracking rows    : 7,534
Unique track IDs : 25

TRACKING: v_i2_L4qquVg0_c010
Frames processed : 758
Tracking rows    : 8,726
Unique track IDs : 20


In [31]:
for seq_name, df in all_tracking_results.items():
    print(f"\n{seq_name}")
    print(f"  Frames with tracks : {df['frame'].nunique()}")
    print(f"  Tracking rows      : {len(df):,}")
    print(f"  Unique IDs         : {df['track_id'].nunique()}")
    print(f"  Avg tracks/frame   : {len(df) / df['frame'].nunique():.2f}")
    print(f"  Max tracks/frame   : {df.groupby('frame').size().max()}")


v_2QhNRucNC7E_c017
  Frames with tracks : 450
  Tracking rows      : 7,113
  Unique IDs         : 23
  Avg tracks/frame   : 15.81
  Max tracks/frame   : 23

v_G-vNjfx1GGc_c600
  Frames with tracks : 500
  Tracking rows      : 5,686
  Unique IDs         : 17
  Avg tracks/frame   : 11.37
  Max tracks/frame   : 17

v_ITo3sCnpw_k_c011
  Frames with tracks : 776
  Tracking rows      : 10,795
  Unique IDs         : 22
  Avg tracks/frame   : 13.91
  Max tracks/frame   : 22

v_dw7LOz17Omg_c067
  Frames with tracks : 450
  Tracking rows      : 7,534
  Unique IDs         : 25
  Avg tracks/frame   : 16.74
  Max tracks/frame   : 25

v_i2_L4qquVg0_c010
  Frames with tracks : 758
  Tracking rows      : 8,726
  Unique IDs         : 20
  Avg tracks/frame   : 11.51
  Max tracks/frame   : 20


In [32]:
sequence_metrics = []

for seq_name, pred_df in all_tracking_results.items():

    seq_dir = VAL_DIR / seq_name
    gt_file = seq_dir / "gt" / "gt.txt"

    gt_seq = pd.read_csv(
        gt_file,
        header=None,
        names=[
            "frame", "track_id", "x", "y",
            "width", "height", "conf", "class", "visibility"
        ]
    )

    gt_seq["x1"] = gt_seq["x"]
    gt_seq["y1"] = gt_seq["y"]
    gt_seq["x2"] = gt_seq["x"] + gt_seq["width"]
    gt_seq["y2"] = gt_seq["y"] + gt_seq["height"]

    acc = mm.MOTAccumulator(auto_id=True)

    for frame in sorted(
        set(gt_seq["frame"]) | set(pred_df["frame"])
    ):
        gt_frame = gt_seq[gt_seq["frame"] == frame]
        pred_frame = pred_df[pred_df["frame"] == frame]

        gt_ids = gt_frame["track_id"].astype(int).tolist()
        pred_ids = pred_frame["track_id"].astype(int).tolist()

        gt_boxes = gt_frame[
            ["x1", "y1", "x2", "y2"]
        ].values

        pred_boxes = pred_frame[
            ["x1", "y1", "x2", "y2"]
        ].values

        if len(gt_boxes) == 0 or len(pred_boxes) == 0:
            distances = None
        else:
            distances = np.array([
                [
                    1 - calculate_iou(gt_box, pred_box)
                    for pred_box in pred_boxes
                ]
                for gt_box in gt_boxes
            ])

            distances[distances > 0.5] = np.nan

        acc.update(
            gt_ids,
            pred_ids,
            distances
        )

    summary = mh.compute(
        acc,
        metrics=[
            "num_frames",
            "num_objects",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "motp",
            "idf1"
        ],
        name=seq_name
    )

    sequence_metrics.append(summary)

metrics_df = pd.concat(sequence_metrics)

display(
    metrics_df[
        [
            "num_frames",
            "num_objects",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "idf1"
        ]
    ]
)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,idf1
v_2QhNRucNC7E_c017,450,6622,1151,5333,629,138,0.078828,0.120131
v_G-vNjfx1GGc_c600,500,5462,1156,3848,682,458,0.086781,0.128095
v_ITo3sCnpw_k_c011,776,9736,2278,7313,1204,145,0.110312,0.115727
v_dw7LOz17Omg_c067,450,6959,1068,5735,731,156,0.048426,0.100048
v_i2_L4qquVg0_c010,758,8262,1899,6196,631,167,0.153474,0.111726


In [33]:
seq_name = "v_2QhNRucNC7E_c017"

pred_df = all_tracking_results[seq_name]

gt_file = VAL_DIR / seq_name / "gt" / "gt.txt"

gt_test = pd.read_csv(
    gt_file,
    header=None,
    names=[
        "frame", "track_id", "x", "y",
        "width", "height", "conf", "class", "visibility"
    ]
)

gt_test["x1"] = gt_test["x"]
gt_test["y1"] = gt_test["y"]
gt_test["x2"] = gt_test["x"] + gt_test["width"]
gt_test["y2"] = gt_test["y"] + gt_test["height"]

print("GT rows:", len(gt_test))
print("Prediction rows:", len(pred_df))

print("GT frames:", gt_test["frame"].min(), "→", gt_test["frame"].max())
print("Pred frames:", pred_df["frame"].min(), "→", pred_df["frame"].max())

print("GT unique IDs:", gt_test["track_id"].nunique())
print("Pred unique IDs:", pred_df["track_id"].nunique())

# Check one frame manually
frame = 1

g = gt_test[gt_test["frame"] == frame]
p = pred_df[pred_df["frame"] == frame]

print("\nFrame 1")
print("GT boxes:", len(g))
print("Pred boxes:", len(p))

if len(g) and len(p):
    ious = []

    for _, gt_row in g.iterrows():
        gt_box = [
            gt_row["x1"], gt_row["y1"],
            gt_row["x2"], gt_row["y2"]
        ]

        best_iou = max(
            calculate_iou(
                gt_box,
                [
                    r["x1"], r["y1"],
                    r["x2"], r["y2"]
                ]
            )
            for _, r in p.iterrows()
        )

        ious.append(best_iou)

    print("Best IoU per GT box:")
    print(np.round(ious, 3))

    print("GT boxes with IoU >= 0.5:",
          sum(i >= 0.5 for i in ious))

GT rows: 6622
Prediction rows: 7113
GT frames: 1 → 450
Pred frames: 1 → 450
GT unique IDs: 21
Pred unique IDs: 23

Frame 1
GT boxes: 19
Pred boxes: 22
Best IoU per GT box:
[      0.964       0.912       0.872       0.982       0.973       0.927       0.955       0.923       0.983       0.946       0.967       0.959       0.977       0.904       0.952        0.95       0.934       0.782       0.921]
GT boxes with IoU >= 0.5: 19


In [34]:
# Test the MOT accumulator on a single frame

acc_test = mm.MOTAccumulator(auto_id=True)

frame = 1

g = gt_test[gt_test["frame"] == frame]
p = pred_df[pred_df["frame"] == frame]

gt_ids = g["track_id"].astype(int).tolist()
pred_ids = p["track_id"].astype(int).tolist()

gt_boxes = g[["x1", "y1", "x2", "y2"]].values
pred_boxes = p[["x1", "y1", "x2", "y2"]].values

distances = np.array([
    [
        1 - calculate_iou(gt_box, pred_box)
        for pred_box in pred_boxes
    ]
    for gt_box in gt_boxes
])

# Reject IoU < 0.5
distances[distances > 0.5] = np.nan

acc_test.update(
    gt_ids,
    pred_ids,
    distances
)

test_summary = mh.compute(
    acc_test,
    metrics=[
        "num_matches",
        "num_false_positives",
        "num_misses",
        "num_switches",
        "mota",
        "idf1"
    ],
    name="frame_1"
)

display(test_summary.T)

,frame_1
num_matches,19.000000
num_false_positives,3.000000
num_misses,0.000000
num_switches,0.000000
mota,0.842105
idf1,0.926829


In [35]:
print("Prediction ID summary:")
print(
    pred_df.groupby("track_id")["frame"]
    .agg(["min", "max", "count"])
    .sort_values("count", ascending=False)
    .head(15)
)

print("\nNumber of prediction IDs:", pred_df["track_id"].nunique())
print("Prediction ID range:",
      pred_df["track_id"].min(),
      "→",
      pred_df["track_id"].max())

print("\nGT ID range:",
      gt_test["track_id"].min(),
      "→",
      gt_test["track_id"].max())

Prediction ID summary:
          min  max  count
track_id                 
1           1  450    450
2           1  450    450
3           1  450    450
4           1  450    450
5           1  450    450
6           1  450    450
7           1  450    450
8           1  450    450
9           1  450    450
10          1  450    450
11          1  450    450
12          1  450    447
13          1  449    433
14          1  448    393
15          1  417    306

Number of prediction IDs: 23
Prediction ID range: 1 → 23

GT ID range: 0 → 20


In [36]:
print("GT confidence values:")
print(gt_test["conf"].value_counts().sort_index())

print("\nGT class values:")
print(gt_test["class"].value_counts().sort_index())

print("\nGT visibility statistics:")
print(gt_test["visibility"].describe())

print("\nGT rows by class:")
print(gt_test.groupby("class").size())

GT confidence values:
conf
1    6622
Name: count, dtype: int64

GT class values:
class
1    6622
Name: count, dtype: int64

GT visibility statistics:
count    6622.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: visibility, dtype: float64

GT rows by class:
class
1    6622
dtype: int64


In [37]:
# Recalculate frame-level Hungarian matches for this exact run

all_matches = []

for frame in sorted(gt_test["frame"].unique()):

    g = gt_test[gt_test["frame"] == frame]
    p = pred_df[pred_df["frame"] == frame]

    gt_boxes = g[["x1", "y1", "x2", "y2"]].values
    pred_boxes = p[["x1", "y1", "x2", "y2"]].values

    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        continue

    iou_matrix = np.array([
        [
            calculate_iou(gt_box, pred_box)
            for pred_box in pred_boxes
        ]
        for gt_box in gt_boxes
    ])

    cost = 1 - iou_matrix

    row_ind, col_ind = linear_sum_assignment(cost)

    for r, c in zip(row_ind, col_ind):
        if iou_matrix[r, c] >= 0.5:
            all_matches.append({
                "frame": frame,
                "gt_id": int(g.iloc[r]["track_id"]),
                "pred_id": int(p.iloc[c]["track_id"]),
                "iou": float(iou_matrix[r, c])
            })

matches_current = pd.DataFrame(all_matches)

print("========== HUNGARIAN MATCHING ==========")
print("Matches:", len(matches_current))
print("Match rate:", len(matches_current) / len(gt_test))
print("Mean IoU:", matches_current["iou"].mean())

========== HUNGARIAN MATCHING ==========
Matches: 6484
Match rate: 0.9791603745092117
Mean IoU: 0.9311683026294354


In [38]:
matches_sorted = matches_current.sort_values(
    ["gt_id", "frame"]
).reset_index(drop=True)

switch_events = []

for gt_id, group in matches_sorted.groupby("gt_id"):

    group = group.sort_values("frame").reset_index(drop=True)

    for i in range(1, len(group)):

        prev = group.iloc[i - 1]
        curr = group.iloc[i]

        # Only count a switch when the same GT is matched
        # in consecutive frames.
        if curr["frame"] == prev["frame"] + 1:

            if curr["pred_id"] != prev["pred_id"]:

                switch_events.append({
                    "gt_id": int(gt_id),
                    "frame": int(curr["frame"]),
                    "previous_pred_id": int(prev["pred_id"]),
                    "new_pred_id": int(curr["pred_id"])
                })

switch_df_current = pd.DataFrame(switch_events)

print("========== CURRENT TRACKING DIAGNOSTIC ==========")
print("Matches:", len(matches_current))
print("Match rate:", f"{len(matches_current) / len(gt_test) * 100:.2f}%")
print("Mean IoU:", f"{matches_current['iou'].mean():.4f}")
print("Consecutive-frame ID switches:", len(switch_df_current))

if len(switch_df_current):
    display(switch_df_current.head(20))

========== CURRENT TRACKING DIAGNOSTIC ==========
Matches: 6484
Match rate: 97.92%
Mean IoU: 0.9312
Consecutive-frame ID switches: 5269


,gt_id,frame,previous_pred_id,new_pred_id
0,0,3,4,5
1,0,4,5,7
2,0,5,7,6
3,0,6,6,3
4,0,7,3,5
5,0,8,5,3
6,0,9,3,15
7,0,10,15,3
8,0,13,3,1
9,0,15,1,2


In [39]:
# Rebuild MOTAccumulator for only the first 10 frames
# so we can inspect the identity events directly.

acc_debug = mm.MOTAccumulator(auto_id=True)

for frame in range(1, 11):

    g = gt_test[gt_test["frame"] == frame]
    p = pred_df[pred_df["frame"] == frame]

    gt_ids = g["track_id"].astype(int).tolist()
    pred_ids = p["track_id"].astype(int).tolist()

    gt_boxes = g[["x1", "y1", "x2", "y2"]].values
    pred_boxes = p[["x1", "y1", "x2", "y2"]].values

    distances = np.array([
        [
            1 - calculate_iou(gt_box, pred_box)
            for pred_box in pred_boxes
        ]
        for gt_box in gt_boxes
    ])

    distances[distances > 0.5] = np.nan

    acc_debug.update(
        gt_ids,
        pred_ids,
        distances
    )

events = acc_debug.events

display(events.head(50))

Type   OId   HId         D
FrameId Event                             
0       0        RAW   NaN   NaN       NaN
        1        RAW   0.0   4.0  0.036398
        2        RAW   1.0   6.0  0.088409
        3        RAW   2.0   7.0  0.128193
        4        RAW   3.0   9.0  0.018088
        5        RAW   4.0  14.0  0.027237
        6        RAW   5.0  20.0  0.073484
        7        RAW   6.0  10.0  0.045358
        8        RAW   7.0  12.0  0.076899
        9        RAW   8.0  16.0  0.017035
        10       RAW  10.0   1.0  0.053537
        11       RAW  11.0   3.0  0.033107
        12       RAW  12.0  11.0  0.040721
        13       RAW  13.0  13.0  0.022779
        14       RAW  14.0  21.0  0.098144
        15       RAW  14.0  22.0  0.096355
        16       RAW  15.0   2.0  0.048246
        17       RAW  16.0   5.0  0.049710
        18       RAW  17.0   8.0  0.066217
        19       RAW  18.0  17.0  0.217740
        20       RAW  20.0  15.0  0.078618
        21       RAW   NaN  18.0       NaN
        22       RAW   NaN  19.0       NaN
        23     MATCH   0.0   4.0  0.036398
        24     MATCH   1.0   6.0  0.088409
        25     MATCH   2.0   7.0  0.128193
        26     MATCH   3.0   9.0  0.018088
        27     MATCH   4.0  14.0  0.027237
        28     MATCH   5.0  20.0  0.073484
        29     MATCH   6.0  10.0  0.045358
        30     MATCH   7.0  12.0  0.076899
        31     MATCH   8.0  16.0  0.017035
        32     MATCH  10.0   1.0  0.053537
        33     MATCH  11.0   3.0  0.033107
        34     MATCH  12.0  11.0  0.040721
        35     MATCH  13.0  13.0  0.022779
        36     MATCH  14.0  22.0  0.096355
        37     MATCH  15.0   2.0  0.048246
        38     MATCH  16.0   5.0  0.049710
        39     MATCH  17.0   8.0  0.066217
        40     MATCH  18.0  17.0  0.217740
        41     MATCH  20.0  15.0  0.078618
        42        FP   NaN  18.0       NaN
        43        FP   NaN  19.0       NaN
        44        FP   NaN  21.0       NaN
1       0        RAW   NaN   NaN       NaN
        1        RAW   0.0   4.0  0.056095
        2        RAW   1.0  19.0  0.034839
        3        RAW   2.0   6.0  0.058423
        4        RAW   3.0  15.0  0.052785

In [40]:
track_id = 4

track_4 = pred_df[
    pred_df["track_id"] == track_id
].sort_values("frame").copy()

track_4["cx"] = (track_4["x1"] + track_4["x2"]) / 2
track_4["cy"] = (track_4["y1"] + track_4["y2"]) / 2

print("Track ID:", track_id)
print("Frames:", len(track_4))
print("Frame range:", track_4["frame"].min(), "→", track_4["frame"].max())

print("\nFirst 20 positions:")
display(
    track_4[
        ["frame", "cx", "cy", "confidence"]
    ].head(20)
)

print("\nLargest frame-to-frame position jumps:")

track_4["jump"] = np.sqrt(
    track_4["cx"].diff() ** 2 +
    track_4["cy"].diff() ** 2
)

display(
    track_4.nlargest(
        15,
        "jump"
    )[
        ["frame", "cx", "cy", "jump"]
    ]
)

Track ID: 4
Frames: 450
Frame range: 1 → 450

First 20 positions:


,frame,cx,cy,confidence
3,1,527.026718,504.680450,0.950326
25,2,534.277222,505.706360,0.952642
44,3,808.423462,427.286591,0.956991
63,4,811.854248,427.314240,0.956056
83,5,541.809570,426.793396,0.959769
102,6,946.127930,437.112915,0.952424
121,7,77.234896,519.758301,0.957423
140,8,560.230225,429.895874,0.949734
161,9,977.275635,435.316956,0.952475
180,10,271.446953,352.089661,0.948595



Largest frame-to-frame position jumps:


,frame,cx,cy,jump
2874,165,31.001871,539.715759,1237.313089
951,50,101.886955,506.007904,1133.816292
932,49,1225.755981,356.148651,1126.184798
2184,121,124.270218,526.482040,1085.864752
5431,334,1247.355225,464.630402,1081.057251
3838,224,1157.412354,328.365967,989.491915
2908,167,1219.467041,311.275635,967.691710
2169,120,1174.459106,250.428497,959.002873
816,43,1207.447754,355.141937,937.429381
4704,281,245.541687,290.158325,936.294173


In [41]:
strict_tracker_config = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/"
    "ai-engine/notebooks/bytetrack_strict.yaml"
)

strict_tracker_config.write_text("""
tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 30
match_thresh: 0.9
fuse_score: True
""")

print(strict_tracker_config)
print(strict_tracker_config.read_text())

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/bytetrack_strict.yaml

tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 30
match_thresh: 0.9
fuse_score: True



In [42]:
test_seq = VAL_DIR / "v_2QhNRucNC7E_c017"
img_dir = test_seq / "img1"

strict_results = player_model.track(
    source=str(img_dir),
    tracker=str(strict_tracker_config),
    persist=False,
    stream=True,
    verbose=False,
    conf=0.25,
    imgsz=1280,
    device=0
)

strict_rows = []

for frame_idx, result in enumerate(strict_results, start=1):

    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, confidence in zip(boxes, ids, confs):
        x1, y1, x2, y2 = box

        strict_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "confidence": float(confidence)
        })

strict_df = pd.DataFrame(strict_rows)

print("========== STRICT BYTETRACK ==========")
print("Frames:", strict_df["frame"].nunique())
print("Tracking rows:", len(strict_df))
print("Unique IDs:", strict_df["track_id"].nunique())
print("Avg tracks/frame:",
      len(strict_df) / strict_df["frame"].nunique())

========== STRICT BYTETRACK ==========
Frames: 450
Tracking rows: 6862
Unique IDs: 21
Avg tracks/frame: 15.248888888888889


In [43]:
track_id = 4

track_strict = strict_df[
    strict_df["track_id"] == track_id
].sort_values("frame").copy()

track_strict["cx"] = (
    track_strict["x1"] + track_strict["x2"]
) / 2

track_strict["cy"] = (
    track_strict["y1"] + track_strict["y2"]
) / 2

track_strict["jump"] = np.sqrt(
    track_strict["cx"].diff() ** 2 +
    track_strict["cy"].diff() ** 2
)

print("Track ID:", track_id)
print("Frames:", len(track_strict))
print("Frame range:",
      track_strict["frame"].min(),
      "→",
      track_strict["frame"].max())

print("\nLargest position jumps:")
display(
    track_strict.nlargest(
        15, "jump"
    )[[
        "frame",
        "cx",
        "cy",
        "jump"
    ]]
)

Track ID: 4
Frames: 450
Frame range: 1 → 450

Largest position jumps:


,frame,cx,cy,jump
2765,165,31.001871,539.715759,1237.313089
926,50,101.886955,506.007904,1133.816292
908,49,1225.755981,356.148651,1126.184798
2096,121,124.270218,526.482040,1085.864752
5258,334,1247.355225,464.630402,1081.057251
3696,224,1157.412354,328.365967,989.491915
2798,167,1219.467041,311.275635,967.691710
2081,120,1174.459106,250.428497,959.002873
795,43,1207.447754,355.141937,937.429381
4547,281,245.541687,290.158325,936.294173


In [44]:
print("========== FIRST 10 FRAMES ==========")

for frame in range(1, 11):
    f = strict_df[strict_df["frame"] == frame]

    print(
        f"Frame {frame:3d}: "
        f"{len(f):2d} detections | "
        f"IDs = {sorted(f['track_id'].tolist())}"
    )

========== FIRST 10 FRAMES ==========
Frame   1: 21 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Frame   2: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   3: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   4: 20 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Frame   5: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   6: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   7: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   8: 19 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Frame   9: 18 detections | IDs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Frame  10: 19 detections | IDs = [1, 2, 3, 4,

In [45]:
# Build a GT ↔ prediction association table
# using the existing Hungarian matches.

association = (
    matches_current
    .groupby(["pred_id", "gt_id"])
    .size()
    .reset_index(name="frames")
    .sort_values(["pred_id", "frames"], ascending=[True, False])
)

print("========== PREDICTION ID → GT ID ASSOCIATIONS ==========")

for pred_id in sorted(association["pred_id"].unique()):
    rows = association[association["pred_id"] == pred_id]

    print(
        f"\nPred ID {pred_id}:"
    )

    display(
        rows.head(5)[
            ["pred_id", "gt_id", "frames"]
        ]
    )

========== PREDICTION ID → GT ID ASSOCIATIONS ==========

Pred ID 1:


,pred_id,gt_id,frames
0,1,0,92
10,1,10,80
18,1,19,52
11,1,11,41
6,1,6,39



Pred ID 2:


,pred_id,gt_id,frames
29,2,10,90
19,2,0,66
35,2,16,63
30,2,11,47
34,2,15,28



Pred ID 3:


...

,pred_id,gt_id,frames
364,21,11,1
365,21,14,1
366,21,15,1



Pred ID 22:


,pred_id,gt_id,frames
367,22,14,1
368,22,17,1



Pred ID 23:


,pred_id,gt_id,frames
369,23,4,1


In [46]:
import motmetrics as mm

print("motmetrics version:", mm.__version__)
print("Available metrics:", len(mm.metrics.motchallenge_metrics))
print(mm.metrics.motchallenge_metrics)

motmetrics version: 1.4.0
Available metrics: 18
['idf1', 'idp', 'idr', 'recall', 'precision', 'num_unique_objects', 'mostly_tracked', 'partially_tracked', 'mostly_lost', 'num_false_positives', 'num_misses', 'num_switches', 'num_fragmentations', 'mota', 'motp', 'num_transfer', 'num_ascend', 'num_migrate']


In [47]:
continuity_stats = []

for track_id, group in strict_df.groupby("track_id"):

    group = group.sort_values("frame").copy()

    cx = (group["x1"] + group["x2"]) / 2
    cy = (group["y1"] + group["y2"]) / 2

    jumps = np.sqrt(
        cx.diff() ** 2 +
        cy.diff() ** 2
    ).dropna()

    continuity_stats.append({
        "track_id": int(track_id),
        "frames": len(group),
        "start": int(group["frame"].min()),
        "end": int(group["frame"].max()),
        "max_jump_px": float(jumps.max()) if len(jumps) else 0,
        "mean_jump_px": float(jumps.mean()) if len(jumps) else 0,
        "jumps_over_300": int((jumps > 300).sum()),
        "jumps_over_500": int((jumps > 500).sum()),
    })

continuity_df = pd.DataFrame(continuity_stats)

display(
    continuity_df.sort_values(
        "max_jump_px",
        ascending=False
    )
)

,track_id,frames,start,end,max_jump_px,mean_jump_px,jumps_over_300,jumps_over_500
14,15,259,1,415,1251.995867,300.461438,117,59
2,3,450,1,450,1245.890221,374.037321,249,140
5,6,450,1,450,1244.033036,378.859647,247,138
3,4,450,1,450,1237.313089,345.726069,230,105
1,2,450,1,450,1214.794212,338.418610,222,129
9,10,450,1,450,1172.825892,334.529185,221,124
11,12,442,1,450,1170.468542,281.249661,184,92
17,18,71,1,389,1167.440140,465.636463,50,30
6,7,450,1,450,1162.941092,377.480000,243,146
8,9,450,1,450,1152.749365,342.577510,219,116


In [48]:
continuity_stats = []

for track_id, group in strict_df.groupby("track_id"):

    # IMPORTANT: sort chronologically
    group = group.sort_values("frame").copy()

    cx = (group["x1"] + group["x2"]) / 2
    cy = (group["y1"] + group["y2"]) / 2

    jumps = np.sqrt(
        cx.diff() ** 2 +
        cy.diff() ** 2
    ).dropna()

    continuity_stats.append({
        "track_id": int(track_id),
        "frames": len(group),
        "start": int(group["frame"].min()),
        "end": int(group["frame"].max()),
        "max_jump_px": float(jumps.max()) if len(jumps) else 0,
        "mean_jump_px": float(jumps.mean()) if len(jumps) else 0,
        "jumps_over_300": int((jumps > 300).sum()),
        "jumps_over_500": int((jumps > 500).sum()),
    })

continuity_df = pd.DataFrame(continuity_stats)

display(
    continuity_df.sort_values(
        "max_jump_px",
        ascending=False
    ).reset_index(drop=True)
)

,track_id,frames,start,end,max_jump_px,mean_jump_px,jumps_over_300,jumps_over_500
0,15,259,1,415,1251.995867,300.461438,117,59
1,3,450,1,450,1245.890221,374.037321,249,140
2,6,450,1,450,1244.033036,378.859647,247,138
3,4,450,1,450,1237.313089,345.726069,230,105
4,2,450,1,450,1214.794212,338.418610,222,129
5,10,450,1,450,1172.825892,334.529185,221,124
6,12,442,1,450,1170.468542,281.249661,184,92
7,18,71,1,389,1167.440140,465.636463,50,30
8,7,450,1,450,1162.941092,377.480000,243,146
9,9,450,1,450,1152.749365,342.577510,219,116


In [49]:
print("Columns:")
print(strict_df.columns.tolist())

print("\nShape:")
print(strict_df.shape)

print("\nFirst 10 rows:")
display(strict_df.head(10))

print("\nLast 10 rows:")
display(strict_df.tail(10))

print("\nFrame statistics:")
print(strict_df["frame"].describe())

print("\nTrack ID statistics:")
print(strict_df["track_id"].describe())

Columns:
['frame', 'track_id', 'x1', 'y1', 'x2', 'y2', 'confidence']

Shape:
(6862, 7)

First 10 rows:


,frame,track_id,x1,y1,x2,y2,confidence
0,1,1,776.609619,385.219666,816.669189,467.823547,0.958812
1,1,2,953.606995,240.375244,992.615417,307.178589,0.955291
2,1,3,1073.547852,303.178772,1100.982422,376.013000,0.951863
3,1,4,509.419830,459.365112,544.633606,549.995789,0.950326
4,1,5,505.393768,378.801636,536.201477,465.070984,0.943560
5,1,6,873.090149,398.265869,907.199646,481.103210,0.940528
6,1,7,928.057434,207.583313,956.454041,266.102753,0.936946
7,1,8,877.174988,208.466522,902.128479,271.671265,0.934530
8,1,9,197.156128,312.695312,247.974945,378.660950,0.934529
9,1,10,7.413300,469.603912,46.784611,574.750854,0.934207



Last 10 rows:


,frame,track_id,x1,y1,x2,y2,confidence
6852,450,3,944.676147,446.710144,982.937866,536.862305,0.956555
6853,450,4,424.891937,287.842834,472.261505,359.166077,0.955474
6854,450,5,988.925903,358.883484,1035.437378,437.231079,0.950609
6855,450,6,1080.456421,244.413193,1108.844116,315.322998,0.938196
6856,450,7,506.662079,215.084351,554.186584,278.006104,0.927266
6857,450,8,714.116150,293.553375,750.391418,364.821930,0.923574
6858,450,9,822.670044,246.934555,856.195068,310.239380,0.921279
6859,450,10,633.494568,221.817688,661.512024,283.290222,0.894812
6860,450,11,1127.622192,208.592590,1172.544067,268.261749,0.793734
6861,450,12,1141.762817,242.714386,1173.812866,308.396576,0.711225



Frame statistics:
count    6862.000000
mean      212.120519
std       130.526209
min         1.000000
25%        96.000000
50%       208.000000
75%       326.000000
max       450.000000
Name: frame, dtype: float64

Track ID statistics:
count    6862.000000
mean        8.268289
std         4.641452
min         1.000000
25%         4.000000
50%         8.000000
75%        12.000000
max        21.000000
Name: track_id, dtype: float64


In [50]:
# Check chronological continuity for ONE track
test_track_id = 1

g = (
    strict_df[strict_df["track_id"] == test_track_id]
    .sort_values("frame")
    .reset_index(drop=True)
)

cx = (g["x1"] + g["x2"]) / 2
cy = (g["y1"] + g["y2"]) / 2

g["cx"] = cx
g["cy"] = cy
g["jump_px"] = np.sqrt(
    g["cx"].diff() ** 2 +
    g["cy"].diff() ** 2
)

print(f"Track ID: {test_track_id}")
print(f"Number of detections: {len(g)}")
print(f"Start frame: {g['frame'].min()}")
print(f"End frame: {g['frame'].max()}")

display(
    g[
        ["frame", "track_id", "cx", "cy", "jump_px"]
    ].head(20)
)

print("\nLargest jumps:")
display(
    g[
        ["frame", "cx", "cy", "jump_px"]
    ]
    .sort_values("jump_px", ascending=False)
    .head(10)
)

Track ID: 1
Number of detections: 450
Start frame: 1
End frame: 450


,frame,track_id,cx,cy,jump_px
0,1,1,796.639404,426.521606,NaN
1,2,1,804.894409,426.814697,8.260206
2,3,1,511.038971,304.369507,318.345478
3,4,1,1001.233154,275.729828,491.030110
4,5,1,1120.937378,343.111450,137.365877
5,6,1,1018.129883,276.520081,122.489965
6,7,1,1139.062012,344.109894,138.538668
7,8,1,82.735535,519.300476,1070.755512
8,9,1,1152.762939,345.570007,1084.039170
9,10,1,572.546509,429.782837,586.295921



Largest jumps:


,frame,cx,cy,jump_px
286,287,116.819565,262.018608,1114.234811
244,245,43.921311,377.266785,1107.994138
126,127,1206.228882,251.928894,1102.895888
8,9,1152.762939,345.570007,1084.039170
7,8,82.735535,519.300476,1070.755512
110,111,1135.527344,251.982758,1065.171748
319,320,228.973145,537.474670,1002.605576
317,318,245.320892,535.331421,1002.546214
214,215,215.145767,351.443970,1001.794736
220,221,180.220688,353.961243,997.489284


In [51]:
for frame_id in range(1, 6):

    g = (
        strict_df[strict_df["frame"] == frame_id]
        .copy()
    )

    g["cx"] = (g["x1"] + g["x2"]) / 2
    g["cy"] = (g["y1"] + g["y2"]) / 2

    print(f"\n===== FRAME {frame_id} =====")

    display(
        g[
            ["track_id", "cx", "cy", "confidence"]
        ]
        .sort_values("track_id")
        .reset_index(drop=True)
    )


===== FRAME 1 =====


,track_id,cx,cy,confidence
0,1,796.639404,426.521606,0.958812
1,2,973.111206,273.776917,0.955291
2,3,1087.265137,339.595886,0.951863
3,4,527.026718,504.680450,0.950326
4,5,520.797623,421.936310,0.943560
5,6,890.144897,439.684540,0.940528
6,7,942.255737,236.843033,0.936946
7,8,889.651733,240.068893,0.934530
8,9,222.565536,345.678131,0.934529
9,10,27.098955,522.177383,0.934207



===== FRAME 2 =====


,track_id,cx,cy,confidence
0,1,804.894409,426.814697,0.959645
1,2,982.333740,274.608948,0.958364
2,3,1095.218994,340.373230,0.957256
3,4,534.277222,505.706360,0.952642
4,5,528.082153,421.979584,0.951447
5,6,951.914795,236.906586,0.934695
6,7,897.078613,240.755981,0.933730
7,8,317.345062,220.495514,0.933498
8,9,675.416260,206.938858,0.925753
9,10,649.647705,279.989799,0.918691



===== FRAME 3 =====


,track_id,cx,cy,confidence
0,1,511.038971,304.369507,0.962025
1,2,1102.445312,341.640320,0.958585
2,3,991.504517,275.176849,0.957834
3,4,808.423462,427.286591,0.956991
4,5,541.622559,506.876831,0.947068
5,6,533.638794,422.644775,0.946507
6,7,433.330048,252.413788,0.938974
7,8,797.243225,269.856079,0.937979
8,9,43.578144,520.508789,0.935168
9,10,904.817505,241.027756,0.930814



===== FRAME 4 =====


,track_id,cx,cy,confidence
0,1,1001.233154,275.729828,0.962101
1,2,1110.680176,342.368652,0.958899
2,3,518.518051,304.817261,0.957363
3,4,811.854248,427.314240,0.956056
4,5,536.380615,424.732910,0.953809
5,6,924.435059,439.055420,0.945540
6,7,549.665771,507.594635,0.943795
7,8,237.588867,345.032959,0.942916
8,9,911.757324,242.082306,0.933131
9,10,443.309509,253.200165,0.929681



===== FRAME 5 =====


,track_id,cx,cy,confidence
0,1,1120.937378,343.111450,0.964449
1,2,527.268311,304.476013,0.961592
2,3,1010.467468,275.889954,0.960360
3,4,541.809570,426.793396,0.959769
4,5,817.707031,427.375214,0.956755
5,6,557.683960,507.800781,0.947952
6,7,242.995377,345.531921,0.947821
7,8,916.945068,243.028900,0.940500
8,9,852.661377,222.284363,0.931093
9,10,453.005249,254.098206,0.930313


In [52]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import time

PLAYER_MODEL_PATH = "/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/models/football_player_detector.pt"

# Fresh model instance — clears any previous tracker state
fresh_model = YOLO(PLAYER_MODEL_PATH)

fresh_results = []

start_time = time.time()

results = fresh_model.track(
    source=str(IMG_DIR),
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):

    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):

        x1, y1, x2, y2 = box

        fresh_results.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

fresh_df = pd.DataFrame(fresh_results)

elapsed = time.time() - start_time

print(f"Frames: {fresh_df['frame'].nunique()}")
print(f"Rows: {len(fresh_df)}")
print(f"Unique IDs: {fresh_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {450 / elapsed:.2f}")

Frames: 450
Rows: 6757
Unique IDs: 69
Time: 33.81s
FPS: 13.31


In [53]:
for frame_id in range(1, 6):

    g = fresh_df[fresh_df["frame"] == frame_id].copy()

    g["cx"] = (g["x1"] + g["x2"]) / 2
    g["cy"] = (g["y1"] + g["y2"]) / 2

    print(f"\n===== FRAME {frame_id} =====")

    display(
        g[
            ["track_id", "cx", "cy", "confidence"]
        ]
        .sort_values("track_id")
        .reset_index(drop=True)
    )


===== FRAME 1 =====


,track_id,cx,cy,confidence
0,1,796.639404,426.521606,0.958812
1,2,973.111206,273.776917,0.955291
2,3,1087.265137,339.595886,0.951863
3,4,527.026733,504.680450,0.950326
4,5,520.797607,421.936310,0.943560
5,6,890.144897,439.684540,0.940528
6,7,942.255737,236.843033,0.936946
7,8,889.651733,240.068893,0.934530
8,9,222.565536,345.678131,0.934529
9,10,27.098955,522.177368,0.934207



===== FRAME 2 =====


,track_id,cx,cy,confidence
0,1,803.802856,426.775940,0.959645
1,2,981.114258,274.498932,0.958364
2,3,1094.167236,340.270447,0.957256
3,4,533.318481,505.570679,0.952642
4,5,527.118896,421.973877,0.951447
5,6,900.083374,439.338531,0.743323
6,7,950.637573,236.898193,0.934695
7,8,896.096558,240.665131,0.933730
8,9,226.111237,345.347778,0.870899
9,10,34.494125,521.173401,0.852789



===== FRAME 3 =====


,track_id,cx,cy,confidence
0,1,807.786804,427.188293,0.956991
1,2,989.672546,275.067596,0.957834
2,3,1101.002197,341.377319,0.958585
3,4,540.110229,506.633728,0.947068
4,5,532.531616,422.498596,0.946507
5,6,910.564575,439.671753,0.891889
6,7,959.545166,236.949326,0.930750
7,8,903.208862,240.978363,0.930814
8,9,230.406860,345.087036,0.918900
9,10,41.962208,520.602661,0.935168



===== FRAME 4 =====


,track_id,cx,cy,confidence
0,1,811.529053,427.332733,0.956056
1,2,999.634155,275.654327,0.962101
2,3,1109.309570,342.266602,0.958899
3,4,548.255981,507.499084,0.943795
4,5,536.205078,424.248749,0.953809
5,6,922.525024,439.221466,0.945540
6,7,968.712585,236.983566,0.922602
7,8,910.647705,241.872772,0.933131
8,9,236.383240,345.008667,0.942916
9,10,50.829391,520.448608,0.912619



===== FRAME 5 =====


,track_id,cx,cy,confidence
0,1,816.892700,427.412048,0.956755
1,2,1009.362183,275.938812,0.960360
2,3,1119.336060,343.067047,0.964449
3,4,556.571899,507.897461,0.947952
4,5,541.187988,426.307098,0.959769
5,6,933.696899,438.506836,0.894302
6,7,977.231689,237.232788,0.922214
7,8,916.638367,242.850647,0.940500
8,9,242.146744,345.352875,0.947821
9,10,59.381325,520.609436,0.924944


In [54]:
continuity_stats = []

for track_id, group in fresh_df.groupby("track_id"):

    group = group.sort_values("frame").reset_index(drop=True)

    cx = (group["x1"] + group["x2"]) / 2
    cy = (group["y1"] + group["y2"]) / 2

    jumps = np.sqrt(
        cx.diff() ** 2 +
        cy.diff() ** 2
    ).dropna()

    continuity_stats.append({
        "track_id": int(track_id),
        "frames": len(group),
        "start": int(group["frame"].min()),
        "end": int(group["frame"].max()),
        "max_jump_px": float(jumps.max()) if len(jumps) else 0,
        "mean_jump_px": float(jumps.mean()) if len(jumps) else 0,
        "jumps_over_300": int((jumps > 300).sum()),
        "jumps_over_500": int((jumps > 500).sum()),
    })

continuity_df = pd.DataFrame(continuity_stats)

display(
    continuity_df
    .sort_values("max_jump_px", ascending=False)
    .reset_index(drop=True)
)

,track_id,frames,start,end,max_jump_px,mean_jump_px,jumps_over_300,jumps_over_500
0,13,379,1,389,67.714256,3.076351,0,0
1,189,3,377,385,55.648067,45.831589,0,0
2,225,4,381,389,52.292587,34.553848,0,0
3,196,4,378,387,52.022346,34.855747,0,0
4,106,233,191,450,50.963165,4.261807,0,0
...,...,...,...,...,...,...,...,...
64,213,1,380,380,0.000000,0.000000,0,0
65,334,1,393,393,0.000000,0.000000,0,0
66,342,1,395,395,0.000000,0.000000,0,0
67,346,1,396,396,0.000000,0.000000,0,0


In [55]:
top10 = continuity_df.sort_values(
    "max_jump_px",
    ascending=False
).head(10)

for _, row in top10.iterrows():
    print(
        f"ID={int(row['track_id'])} | "
        f"frames={int(row['frames'])} | "
        f"start={int(row['start'])} | "
        f"end={int(row['end'])} | "
        f"max_jump={row['max_jump_px']:.2f}px | "
        f"mean_jump={row['mean_jump_px']:.2f}px | "
        f">300={int(row['jumps_over_300'])} | "
        f">500={int(row['jumps_over_500'])}"
    )

ID=13 | frames=379 | start=1 | end=389 | max_jump=67.71px | mean_jump=3.08px | >300=0 | >500=0
ID=189 | frames=3 | start=377 | end=385 | max_jump=55.65px | mean_jump=45.83px | >300=0 | >500=0
ID=225 | frames=4 | start=381 | end=389 | max_jump=52.29px | mean_jump=34.55px | >300=0 | >500=0
ID=196 | frames=4 | start=378 | end=387 | max_jump=52.02px | mean_jump=34.86px | >300=0 | >500=0
ID=106 | frames=233 | start=191 | end=450 | max_jump=50.96px | mean_jump=4.26px | >300=0 | >500=0
ID=154 | frames=78 | start=371 | end=450 | max_jump=50.39px | mean_jump=7.97px | >300=0 | >500=0
ID=146 | frames=24 | start=351 | end=383 | max_jump=47.69px | mean_jump=6.82px | >300=0 | >500=0
ID=347 | frames=45 | start=397 | end=450 | max_jump=47.45px | mean_jump=5.58px | >300=0 | >500=0
ID=127 | frames=133 | start=244 | end=401 | max_jump=46.11px | mean_jump=5.45px | >300=0 | >500=0
ID=19 | frames=419 | start=1 | end=450 | max_jump=45.70px | mean_jump=4.38px | >300=0 | >500=0


In [56]:
import motmetrics as mm
import numpy as np

# Fresh MOT accumulator
acc_fresh = mm.MOTAccumulator(auto_id=True)

# Process every frame
for frame_id in range(1, 451):

    gt_frame = gt_eval[gt_eval["frame"] == frame_id]
    pred_frame = fresh_df[fresh_df["frame"] == frame_id]

    gt_ids = gt_frame["track_id"].astype(int).to_numpy()
    pred_ids = pred_frame["track_id"].astype(int).to_numpy()

    distances = np.ones(
        (len(gt_ids), len(pred_ids)),
        dtype=float
    )

    # IoU distance
    for i, gt_row in enumerate(gt_frame.itertuples()):
        gt_box = (
            gt_row.x1,
            gt_row.y1,
            gt_row.x2,
            gt_row.y2
        )

        for j, pred_row in enumerate(pred_frame.itertuples()):
            pred_box = (
                pred_row.x1,
                pred_row.y1,
                pred_row.x2,
                pred_row.y2
            )

            distances[i, j] = 1 - calculate_iou(
                gt_box,
                pred_box
            )

    # Only allow IoU >= 0.5
    distances[distances > 0.5] = np.nan

    acc_fresh.update(
        gt_ids,
        pred_ids,
        distances
    )

# MOTChallenge metrics
mh = mm.metrics.create()

summary = mh.compute(
    acc_fresh,
    metrics=[
        "num_frames",
        "num_objects",
        "num_matches",
        "num_switches",
        "num_false_positives",
        "num_misses",
        "mota",
        "motp",
        "idf1",
        "idp",
        "idr"
    ],
    name="fresh_bytetrack"
)

display(summary)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
fresh_bytetrack,450,6622,6182,67,508,373,0.856841,0.152122,0.759399,0.751813,0.76714


In [57]:
from pathlib import Path

SPORTS_MOT_VAL_DIR = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/"
    "datasets/football/benchmarks/sportsmot/extracted/val"
)

print("Validation directory:", SPORTS_MOT_VAL_DIR)
print("Exists:", SPORTS_MOT_VAL_DIR.exists())

Validation directory: /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val
Exists: True


In [58]:
from ultralytics import YOLO
import pandas as pd
import time

remaining_sequences = [
    "v_G-vNjfx1GGc_c600",
    "v_ITo3sCnpw_k_c011",
    "v_dw7LOz17Omg_c067",
    "v_i2_L4qquVg0_c010",
]

multi_tracking_results = {}

for seq_name in remaining_sequences:

    print(f"\n{'='*60}")
    print(f"Running: {seq_name}")
    print(f"{'='*60}")

    seq_img_dir = SPORTS_MOT_VAL_DIR / seq_name / "img1"

    # Fresh model for every sequence
    model = YOLO(PLAYER_MODEL_PATH)

    rows = []
    start_time = time.time()

    results = model.track(
        source=str(seq_img_dir),
        tracker="bytetrack.yaml",
        persist=True,
        stream=True,
        conf=0.25,
        imgsz=1280,
        verbose=False
    )

    for frame_idx, result in enumerate(results, start=1):

        if result.boxes is None or result.boxes.id is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, track_id, conf in zip(boxes, ids, confs):

            x1, y1, x2, y2 = box

            rows.append({
                "frame": frame_idx,
                "track_id": track_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "confidence": conf
            })

    elapsed = time.time() - start_time

    seq_df = pd.DataFrame(rows)
    multi_tracking_results[seq_name] = seq_df

    print(f"Frames: {seq_df['frame'].nunique()}")
    print(f"Rows: {len(seq_df)}")
    print(f"Unique IDs: {seq_df['track_id'].nunique()}")
    print(f"Time: {elapsed:.2f}s")
    print(f"FPS: {seq_df['frame'].nunique() / elapsed:.2f}")


Running: v_G-vNjfx1GGc_c600
Frames: 500
Rows: 5222
Unique IDs: 89
Time: 46.74s
FPS: 10.70

Running: v_ITo3sCnpw_k_c011
Frames: 776
Rows: 10205
Unique IDs: 105
Time: 69.48s
FPS: 11.17

Running: v_dw7LOz17Omg_c067
Frames: 450
Rows: 7138
Unique IDs: 97
Time: 41.68s
FPS: 10.80

Running: v_i2_L4qquVg0_c010
Frames: 758
Rows: 7682
Unique IDs: 154
Time: 60.72s
FPS: 12.48


In [59]:
def evaluate_mot_sequence(seq_name, pred_df, gt_file):
    
    gt = pd.read_csv(
        gt_file,
        header=None,
        names=[
            "frame",
            "track_id",
            "x",
            "y",
            "width",
            "height",
            "conf",
            "class",
            "visibility"
        ]
    )

    gt["x1"] = gt["x"]
    gt["y1"] = gt["y"]
    gt["x2"] = gt["x"] + gt["width"]
    gt["y2"] = gt["y"] + gt["height"]

    acc = mm.MOTAccumulator(auto_id=True)

    max_frame = max(
        int(gt["frame"].max()),
        int(pred_df["frame"].max())
    )

    for frame_id in range(1, max_frame + 1):

        gt_frame = gt[gt["frame"] == frame_id]
        pred_frame = pred_df[pred_df["frame"] == frame_id]

        gt_ids = gt_frame["track_id"].astype(int).to_numpy()
        pred_ids = pred_frame["track_id"].astype(int).to_numpy()

        distances = np.ones(
            (len(gt_ids), len(pred_ids)),
            dtype=float
        )

        for i, gt_row in enumerate(gt_frame.itertuples()):

            gt_box = (
                gt_row.x1,
                gt_row.y1,
                gt_row.x2,
                gt_row.y2
            )

            for j, pred_row in enumerate(pred_frame.itertuples()):

                pred_box = (
                    pred_row.x1,
                    pred_row.y1,
                    pred_row.x2,
                    pred_row.y2
                )

                distances[i, j] = 1 - calculate_iou(
                    gt_box,
                    pred_box
                )

        distances[distances > 0.5] = np.nan

        acc.update(
            gt_ids,
            pred_ids,
            distances
        )

    mh = mm.metrics.create()

    summary = mh.compute(
        acc,
        metrics=[
            "num_frames",
            "num_objects",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "motp",
            "idf1",
            "idp",
            "idr"
        ],
        name=seq_name
    )

    return summary


c600_gt = (
    SPORTS_MOT_VAL_DIR /
    "v_G-vNjfx1GGc_c600" /
    "gt" /
    "gt.txt"
)

c600_metrics = evaluate_mot_sequence(
    "c600",
    multi_tracking_results["v_G-vNjfx1GGc_c600"],
    c600_gt
)

display(c600_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c600,500,5462,4635,67,520,760,0.753387,0.190715,0.558405,0.571237,0.546137


In [60]:
c011_gt = (
    SPORTS_MOT_VAL_DIR /
    "v_ITo3sCnpw_k_c011" /
    "gt" /
    "gt.txt"
)

c011_metrics = evaluate_mot_sequence(
    "c011",
    multi_tracking_results["v_ITo3sCnpw_k_c011"],
    c011_gt
)

display(c011_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c011,776,9736,9254,107,844,375,0.863804,0.169892,0.586631,0.57315,0.60076


In [61]:
c067_gt = (
    SPORTS_MOT_VAL_DIR /
    "v_dw7LOz17Omg_c067" /
    "gt" /
    "gt.txt"
)

c067_metrics = evaluate_mot_sequence(
    "c067",
    multi_tracking_results["v_dw7LOz17Omg_c067"],
    c067_gt
)

display(c067_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c067,450,6959,6649,53,436,257,0.892801,0.160411,0.693623,0.684926,0.702543


In [62]:
c010_gt = (
    SPORTS_MOT_VAL_DIR /
    "v_i2_L4qquVg0_c010" /
    "gt" /
    "gt.txt"
)

c010_metrics = evaluate_mot_sequence(
    "c010",
    multi_tracking_results["v_i2_L4qquVg0_c010"],
    c010_gt
)

display(c010_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c010,758,8262,7271,125,286,866,0.845437,0.167643,0.44995,0.466936,0.434156


In [63]:
all_acc = mm.MOTAccumulator(auto_id=True)

sequence_pairs = [
    ("c017", "v_2QhNRucNC7E_c017", fresh_df),
    ("c600", "v_G-vNjfx1GGc_c600", multi_tracking_results["v_G-vNjfx1GGc_c600"]),
    ("c011", "v_ITo3sCnpw_k_c011", multi_tracking_results["v_ITo3sCnpw_k_c011"]),
    ("c067", "v_dw7LOz17Omg_c067", multi_tracking_results["v_dw7LOz17Omg_c067"]),
    ("c010", "v_i2_L4qquVg0_c010", multi_tracking_results["v_i2_L4qquVg0_c010"]),
]

for short_name, seq_name, pred_df in sequence_pairs:

    gt_file = (
        SPORTS_MOT_VAL_DIR /
        seq_name /
        "gt" /
        "gt.txt"
    )

    gt = pd.read_csv(
        gt_file,
        header=None,
        names=[
            "frame", "track_id", "x", "y",
            "width", "height", "conf",
            "class", "visibility"
        ]
    )

    gt["x1"] = gt["x"]
    gt["y1"] = gt["y"]
    gt["x2"] = gt["x"] + gt["width"]
    gt["y2"] = gt["y"] + gt["height"]

    for frame_id in range(
        1,
        max(
            int(gt["frame"].max()),
            int(pred_df["frame"].max())
        ) + 1
    ):

        gt_frame = gt[gt["frame"] == frame_id]
        pred_frame = pred_df[pred_df["frame"] == frame_id]

        gt_ids = gt_frame["track_id"].astype(int).to_numpy()
        pred_ids = pred_frame["track_id"].astype(int).to_numpy()

        distances = np.ones(
            (len(gt_ids), len(pred_ids)),
            dtype=float
        )

        for i, gt_row in enumerate(gt_frame.itertuples()):

            gt_box = (
                gt_row.x1,
                gt_row.y1,
                gt_row.x2,
                gt_row.y2
            )

            for j, pred_row in enumerate(pred_frame.itertuples()):

                pred_box = (
                    pred_row.x1,
                    pred_row.y1,
                    pred_row.x2,
                    pred_row.y2
                )

                distances[i, j] = 1 - calculate_iou(
                    gt_box,
                    pred_box
                )

        distances[distances > 0.5] = np.nan

        all_acc.update(
            gt_ids,
            pred_ids,
            distances
        )

overall_metrics = mm.metrics.create().compute(
    all_acc,
    metrics=[
        "num_frames",
        "num_objects",
        "num_matches",
        "num_switches",
        "num_false_positives",
        "num_misses",
        "mota",
        "motp",
        "idf1",
        "idp",
        "idr"
    ],
    name="5_sequence_baseline"
)

display(overall_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
5_sequence_baseline,2934,37041,33909,501,2594,2631,0.845415,0.16718,0.217381,0.21749,0.217273


In [64]:
# Recompute aggregate MOT metrics with globally unique prediction IDs

global_acc = mm.MOTAccumulator(auto_id=True)

sequence_pairs = [
    ("c017", "v_2QhNRucNC7E_c017", fresh_df),
    ("c600", "v_G-vNjfx1GGc_c600", multi_tracking_results["v_G-vNjfx1GGc_c600"]),
    ("c011", "v_ITo3sCnpw_k_c011", multi_tracking_results["v_ITo3sCnpw_k_c011"]),
    ("c067", "v_dw7LOz17Omg_c067", multi_tracking_results["v_dw7LOz17Omg_c067"]),
    ("c010", "v_i2_L4qquVg0_c010", multi_tracking_results["v_i2_L4qquVg0_c010"]),
]

for seq_idx, (short_name, seq_name, pred_df) in enumerate(sequence_pairs):

    gt_file = (
        SPORTS_MOT_VAL_DIR / seq_name / "gt" / "gt.txt"
    )

    gt = pd.read_csv(
        gt_file,
        header=None,
        names=[
            "frame", "track_id", "x", "y",
            "width", "height", "conf",
            "class", "visibility"
        ]
    )

    gt["x1"] = gt["x"]
    gt["y1"] = gt["y"]
    gt["x2"] = gt["x"] + gt["width"]
    gt["y2"] = gt["y"] + gt["height"]

    # Make prediction IDs globally unique across sequences
    pred_df = pred_df.copy()
    pred_df["global_track_id"] = (
        pred_df["track_id"].astype(int)
        + (seq_idx + 1) * 100000
    )

    for frame_id in range(
        1,
        max(
            int(gt["frame"].max()),
            int(pred_df["frame"].max())
        ) + 1
    ):

        gt_frame = gt[gt["frame"] == frame_id]
        pred_frame = pred_df[pred_df["frame"] == frame_id]

        gt_ids = (
            gt_frame["track_id"]
            .astype(int)
            .to_numpy()
        )

        pred_ids = (
            pred_frame["global_track_id"]
            .astype(int)
            .to_numpy()
        )

        distances = np.ones(
            (len(gt_ids), len(pred_ids)),
            dtype=float
        )

        for i, gt_row in enumerate(gt_frame.itertuples()):

            gt_box = (
                gt_row.x1,
                gt_row.y1,
                gt_row.x2,
                gt_row.y2
            )

            for j, pred_row in enumerate(pred_frame.itertuples()):

                pred_box = (
                    pred_row.x1,
                    pred_row.y1,
                    pred_row.x2,
                    pred_row.y2
                )

                distances[i, j] = 1 - calculate_iou(
                    gt_box,
                    pred_box
                )

        distances[distances > 0.5] = np.nan

        global_acc.update(
            gt_ids,
            pred_ids,
            distances
        )

global_metrics = mm.metrics.create().compute(
    global_acc,
    metrics=[
        "num_frames",
        "num_objects",
        "num_matches",
        "num_switches",
        "num_false_positives",
        "num_misses",
        "mota",
        "motp",
        "idf1",
        "idp",
        "idr"
    ],
    name="5_sequence_global_id_baseline"
)

display(global_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
5_sequence_global_id_baseline,2934,37041,33908,502,2594,2631,0.845388,0.16718,0.214923,0.215031,0.214816


In [65]:
from pathlib import Path

BYTE_TRACK_09 = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/"
    "ai-engine/notebooks/bytetrack_09.yaml"
)

BYTE_TRACK_09.write_text("""
tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 30
match_thresh: 0.9
fuse_score: True
""")

print(BYTE_TRACK_09)
print(BYTE_TRACK_09.exists())

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/bytetrack_09.yaml
True


In [66]:
# Experiment 1: ByteTrack match_thresh = 0.9
# Sequence: c017

from ultralytics import YOLO
import pandas as pd
import time

exp_model = YOLO(PLAYER_MODEL_PATH)

exp_rows = []
start_time = time.time()

results = exp_model.track(
    source=str(IMG_DIR),
    tracker=str(BYTE_TRACK_09),
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):

    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):

        x1, y1, x2, y2 = box

        exp_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

exp_df = pd.DataFrame(exp_rows)

elapsed = time.time() - start_time

print(f"Frames: {exp_df['frame'].nunique()}")
print(f"Rows: {len(exp_df)}")
print(f"Unique IDs: {exp_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {exp_df['frame'].nunique() / elapsed:.2f}")

Frames: 450
Rows: 6791
Unique IDs: 44
Time: 39.70s
FPS: 11.33


In [67]:
exp09_metrics = evaluate_mot_sequence(
    "c017_match09",
    exp_df,
    SPORTS_MOT_VAL_DIR /
    "v_2QhNRucNC7E_c017" /
    "gt" /
    "gt.txt"
)

display(exp09_metrics)

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c017_match09,450,6622,6236,39,516,347,0.863787,0.153781,0.786401,0.776616,0.796436


In [68]:
exp09_c600_model = YOLO(PLAYER_MODEL_PATH)

exp09_c600_rows = []
start_time = time.time()

c600_img_dir = (
    SPORTS_MOT_VAL_DIR /
    "v_G-vNjfx1GGc_c600" /
    "img1"
)

results = exp09_c600_model.track(
    source=str(c600_img_dir),
    tracker=str(BYTE_TRACK_09),
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):

    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):

        x1, y1, x2, y2 = box

        exp09_c600_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

exp09_c600_df = pd.DataFrame(exp09_c600_rows)

elapsed = time.time() - start_time

print(f"Frames: {exp09_c600_df['frame'].nunique()}")
print(f"Rows: {len(exp09_c600_df)}")
print(f"Unique IDs: {exp09_c600_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {exp09_c600_df['frame'].nunique() / elapsed:.2f}")

Frames: 500
Rows: 5213
Unique IDs: 68
Time: 42.99s
FPS: 11.63


In [69]:
exp09_c600_metrics = evaluate_mot_sequence(
    "c600_match_0.9",
    exp09_c600_df,
    SPORTS_MOT_VAL_DIR / "v_G-vNjfx1GGc_c600" / "gt" / "gt.txt"
)

exp09_c600_metrics

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c600_match_0.9,500,5462,4703,51,459,708,0.777005,0.190167,0.576487,0.590255,0.563347


In [70]:
exp09_c011_model = YOLO(PLAYER_MODEL_PATH)

exp09_c011_rows = []
start_time = time.time()

c011_img_dir = (
    SPORTS_MOT_VAL_DIR /
    "v_ITo3sCnpw_k_c011" /
    "img1"
)

results = exp09_c011_model.track(
    source=str(c011_img_dir),
    tracker=str(BYTE_TRACK_09),
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):
        x1, y1, x2, y2 = box

        exp09_c011_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

exp09_c011_df = pd.DataFrame(exp09_c011_rows)

elapsed = time.time() - start_time

print(f"Frames: {exp09_c011_df['frame'].max()}")
print(f"Rows: {len(exp09_c011_df)}")
print(f"Unique IDs: {exp09_c011_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {exp09_c011_df['frame'].max() / elapsed:.2f}")

Frames: 776
Rows: 10146
Unique IDs: 67
Time: 68.27s
FPS: 11.37


In [71]:
exp09_c011_metrics = evaluate_mot_sequence(
    "c011_match_0.9",
    exp09_c011_df,
    SPORTS_MOT_VAL_DIR / "v_ITo3sCnpw_k_c011" / "gt" / "gt.txt"
)

exp09_c011_metrics

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c011_match_0.9,776,9736,9332,54,760,350,0.880444,0.170569,0.644704,0.631678,0.658279


In [72]:
exp09_c067_model = YOLO(PLAYER_MODEL_PATH)

exp09_c067_rows = []
start_time = time.time()

c067_img_dir = (
    SPORTS_MOT_VAL_DIR /
    "v_dw7LOz17Omg_c067" /
    "img1"
)

results = exp09_c067_model.track(
    source=str(c067_img_dir),
    tracker=str(BYTE_TRACK_09),
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):
        x1, y1, x2, y2 = box

        exp09_c067_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

exp09_c067_df = pd.DataFrame(exp09_c067_rows)

elapsed = time.time() - start_time

print(f"Frames: {exp09_c067_df['frame'].max()}")
print(f"Rows: {len(exp09_c067_df)}")
print(f"Unique IDs: {exp09_c067_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {exp09_c067_df['frame'].max() / elapsed:.2f}")

Frames: 450
Rows: 7055
Unique IDs: 68
Time: 37.16s
FPS: 12.11


In [73]:
exp09_c067_metrics = evaluate_mot_sequence(
    "c067_match_0.9",
    exp09_c067_df,
    SPORTS_MOT_VAL_DIR / "v_dw7LOz17Omg_c067" / "gt" / "gt.txt"
)

exp09_c067_metrics

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c067_match_0.9,450,6959,6652,32,371,275,0.902572,0.161202,0.777936,0.772644,0.783302


In [74]:
exp09_c010_model = YOLO(PLAYER_MODEL_PATH)

exp09_c010_rows = []
start_time = time.time()

c010_img_dir = (
    SPORTS_MOT_VAL_DIR /
    "v_i2_L4qquVg0_c010" /
    "img1"
)

results = exp09_c010_model.track(
    source=str(c010_img_dir),
    tracker=str(BYTE_TRACK_09),
    persist=True,
    stream=True,
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for frame_idx, result in enumerate(results, start=1):
    if result.boxes is None or result.boxes.id is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, ids, confs):
        x1, y1, x2, y2 = box

        exp09_c010_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "confidence": conf
        })

exp09_c010_df = pd.DataFrame(exp09_c010_rows)

elapsed = time.time() - start_time

print(f"Frames: {exp09_c010_df['frame'].max()}")
print(f"Rows: {len(exp09_c010_df)}")
print(f"Unique IDs: {exp09_c010_df['track_id'].nunique()}")
print(f"Time: {elapsed:.2f}s")
print(f"FPS: {exp09_c010_df['frame'].max() / elapsed:.2f}")

Frames: 758
Rows: 7810
Unique IDs: 85
Time: 61.95s
FPS: 12.24


In [75]:
exp09_c010_metrics = evaluate_mot_sequence(
    "c010_match_0.9",
    exp09_c010_df,
    SPORTS_MOT_VAL_DIR / "v_i2_L4qquVg0_c010" / "gt" / "gt.txt"
)

exp09_c010_metrics

,num_frames,num_objects,num_matches,num_switches,num_false_positives,num_misses,mota,motp,idf1,idp,idr
c010_match_0.9,758,8262,7523,71,216,668,0.884411,0.169786,0.50784,0.522535,0.493948


In [76]:
# Final 5-sequence comparison
# Baseline values recorded earlier
baseline_data = {
    "c017": {
        "mota": 0.856841,
        "idf1": 0.759399,
        "switches": 67,
    },
    "c600": {
        "mota": 0.753387,
        "idf1": 0.558405,
        "switches": 760,
    },
    "c011": {
        "mota": 0.863804,
        "idf1": 0.586631,
        "switches": 107,
    },
    "c067": {
        "mota": 0.892801,
        "idf1": 0.693623,
        "switches": 257,
    },
    "c010": {
        "mota": 0.845437,
        "idf1": 0.449950,
        "switches": 866,
    },
}

# match_thresh = 0.9 values from our evaluations
match09_data = {
    "c017": {
        "mota": 0.863787,
        "idf1": 0.786401,
        "switches": 39,
    },
    "c600": {
        "mota": 0.777005,
        "idf1": 0.576487,
        "switches": 470,
    },
    "c011": {
        "mota": 0.880444,
        "idf1": 0.644704,
        "switches": 54,
    },
    "c067": {
        "mota": 0.902572,
        "idf1": 0.777936,
        "switches": 32,
    },
    "c010": {
        "mota": 0.884411,
        "idf1": 0.507840,
        "switches": 668,
    },
}

comparison_rows = []

for seq in baseline_data:
    b = baseline_data[seq]
    e = match09_data[seq]

    comparison_rows.append({
        "Sequence": seq,
        "Baseline MOTA (%)": b["mota"] * 100,
        "0.9 MOTA (%)": e["mota"] * 100,
        "MOTA Δ (pp)": (e["mota"] - b["mota"]) * 100,
        "Baseline IDF1 (%)": b["idf1"] * 100,
        "0.9 IDF1 (%)": e["idf1"] * 100,
        "IDF1 Δ (pp)": (e["idf1"] - b["idf1"]) * 100,
        "Baseline Switches": b["switches"],
        "0.9 Switches": e["switches"],
        "Switch Reduction": b["switches"] - e["switches"],
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.round(3)

,Sequence,Baseline MOTA (%),0.9 MOTA (%),MOTA Δ (pp),Baseline IDF1 (%),0.9 IDF1 (%),IDF1 Δ (pp),Baseline Switches,0.9 Switches,Switch Reduction
0,c017,85.684,86.379,0.695,75.940,78.640,2.700,67,39,28
1,c600,75.339,77.700,2.362,55.841,57.649,1.808,760,470,290
2,c011,86.380,88.044,1.664,58.663,64.470,5.807,107,54,53
3,c067,89.280,90.257,0.977,69.362,77.794,8.431,257,32,225
4,c010,84.544,88.441,3.897,44.995,50.784,5.789,866,668,198


In [77]:
from pathlib import Path

BYTE_TRACK_FINAL = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/bytetrack_final.yaml"
)

BYTE_TRACK_FINAL.write_text(
    """tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 30
match_thresh: 0.9
fuse_score: True
""",
    encoding="utf-8"
)

print(f"Saved: {BYTE_TRACK_FINAL}")
print()
print(BYTE_TRACK_FINAL.read_text())

Saved: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/notebooks/bytetrack_final.yaml

tracker_type: bytetrack
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 30
match_thresh: 0.9
fuse_score: True



In [78]:
from pathlib import Path

possible_mixsort_paths = [
    Path("/mnt/d/My Projects/AthletiQ/AthletiQ/MixSort"),
    Path("/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/MixSort"),
    Path("/mnt/d/My Projects/AthletiQ/AthletiQ/third_party/MixSort"),
]

for p in possible_mixsort_paths:
    print(f"{p} -> {'FOUND' if p.exists() else 'not found'}")

/mnt/d/My Projects/AthletiQ/AthletiQ/MixSort -> not found
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/MixSort -> not found
/mnt/d/My Projects/AthletiQ/AthletiQ/third_party/MixSort -> not found


In [79]:
import subprocess
from pathlib import Path

MIXSORT_DIR = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort"
)

MIXSORT_DIR.parent.mkdir(parents=True, exist_ok=True)

if not MIXSORT_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--recursive",
            "https://github.com/MCG-NJU/MixSort.git",
            str(MIXSORT_DIR),
        ],
        check=True
    )
    print("MixSort cloned successfully.")
else:
    print("MixSort directory already exists.")

print("\nLocation:", MIXSORT_DIR)
print("Exists:", MIXSORT_DIR.exists())
print("Files:", len(list(MIXSORT_DIR.iterdir())))

MixSort directory already exists.

Location: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort
Exists: True
Files: 18


In [80]:
from pathlib import Path

for p in sorted(MIXSORT_DIR.iterdir()):
    print(f"{'[DIR] ' if p.is_dir() else '[FILE]'} {p.name}")

[DIR]  .git
[FILE] .gitignore
[FILE] .gitmodules
[FILE] Dockerfile
[FILE] LICENSE
[DIR]  MixViT
[FILE] README.md
[DIR]  TrackEval
[DIR]  assets
[DIR]  deploy
[DIR]  exps
[DIR]  pretrained
[FILE] requirements.txt
[FILE] setup.cfg
[FILE] setup.py
[DIR]  tools
[DIR]  videos
[DIR]  yolox


In [81]:
from pathlib import Path

# Find files mentioning SportsMOT / MixSort / MOT
patterns = ["*sportsmot*", "*mixsort*", "*mot*"]

for pattern in patterns:
    print(f"\n===== {pattern} =====")
    matches = list(MIXSORT_DIR.rglob(pattern))
    
    for p in matches[:80]:
        print(p.relative_to(MIXSORT_DIR))
    
    if len(matches) > 80:
        print(f"... and {len(matches) - 80} more")


===== *sportsmot* =====
tools/convert_sportsmot_anno.py
tools/convert_sportsmot_mix_anno.py
tools/convert_sportsmot_to_coco.py
tools/mix_sportsmot.py
exps/example/mot/yolox_x_sportsmot.py
exps/example/mot/yolox_x_sportsmot_mix.py
MixViT/lib/train/dataset/sportsmot.py
MixViT/lib/train/dataset/__pycache__/sportsmot.cpython-312.pyc

===== *mixsort* =====
tools/track_mixsort.py
tools/track_mixsort_oc.py
yolox/mixsort_oc_tracker
yolox/mixsort_tracker
yolox/mixsort_oc_tracker/mixsort_oc_tracker.py
yolox/mixsort_tracker/mixsort_iou_tracker.py
yolox/mixsort_tracker/mixsort_tracker.py
yolox/mixsort_tracker/__pycache__/mixsort_tracker.cpython-312.pyc

===== *mot* =====
tools/convert_mot17_anno.py
tools/convert_mot17_to_coco.py
tools/convert_sportsmot_anno.py
tools/convert_sportsmot_mix_anno.py
tools/convert_sportsmot_to_coco.py
tools/mix_sportsmot.py
tools/mota.py
tools/track_motdt.py
yolox/motdt_tracker
.git/refs/remotes
.git/logs/refs/remotes
.git/modules/TrackEval/refs/remotes
.git/modules/T

In [82]:
track_mixsort_file = MIXSORT_DIR / "tools" / "track_mixsort.py"

print(track_mixsort_file.read_text(encoding="utf-8"))

from loguru import logger

import torch
import torch.backends.cudnn as cudnn
from torch.nn.parallel import DistributedDataParallel as DDP
import sys, os
sys.path.append(os.path.join(os.path.dirname(__file__), '../MixViT'))
from yolox.core import launch
from yolox.exp import get_exp
from yolox.utils import configure_nccl, fuse_model, get_local_rank, get_model_info, setup_logger
from yolox.evaluators import MOTEvaluator

import argparse
import random
import warnings
import glob
import motmetrics as mm
from collections import OrderedDict
from pathlib import Path


def make_parser():
    parser = argparse.ArgumentParser("YOLOX Eval")
    parser.add_argument("-expn", "--experiment-name", type=str, default=None)
    parser.add_argument("-n", "--name", type=str, default=None, help="model name")

    # distributed
    parser.add_argument(
        "--dist-backend", default="nccl", type=str, help="distributed backend"
    )
    parser.add_argument(
        "--dist-url",
        default=None,
   

In [83]:
mixsort_tracker_file = (
    MIXSORT_DIR
    / "yolox"
    / "mixsort_tracker"
    / "mixsort_tracker.py"
)

print(mixsort_tracker_file.read_text(encoding="utf-8"))

import numpy as np
from collections import deque
import os
import os.path as osp
import copy
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import resized_crop, normalize
import math
from torch.utils.tensorboard import SummaryWriter

from .kalman_filter import KalmanFilter
from . import matching
from .basetrack import BaseTrack, TrackState
from MixViT.lib.models.mixformer_vit import build_mixformer_deit
from MixViT.lib.train.data.processing import MixformerProcessing as MP
from MixViT.lib.train.data.transforms import Transform, ToTensor, Normalize
import MixViT.lib.train.data.processing_utils as prutils
import MixViT.lib.train.admin.settings as ws_settings
import importlib
from MixViT.lib.train.base_functions import update_settings
import MixViT.lib.train.data.transforms as tfm
from typing import List, Union, Tuple


class STrack(BaseTrack):
    shared_kalman = KalmanFilter()

    def __init__(self, tlwh, score, iou):

        # wait activate
      

In [84]:
TRACK_YAML = (
    MIXSORT_DIR
    / "MixViT"
    / "experiments"
    / "mixformer_deit"
    / "track.yaml"
)

print(TRACK_YAML)
print()
print(TRACK_YAML.read_text(encoding="utf-8"))

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/experiments/mixformer_deit/track.yaml

DATA:
  MAX_SAMPLE_INTERVAL: 10
  MEAN:
  - 0.485
  - 0.456
  - 0.406
  SEARCH:
    CENTER_JITTER: 3
    FACTOR: 4.0
    SCALE_JITTER: 0.3
    SIZE: 224
  STD:
  - 0.229
  - 0.224
  - 0.225
  TEMPLATE:
    CENTER_JITTER: 0
    FACTOR: 1.5
    SCALE_JITTER: 0
    SIZE: 96
    NUMBER: 1
  TRAIN:
    # DATASETS_NAME: "SoccerNet_train","SoccerNet_test","DanceTrack_train","DanceTrack_val","SportsMOT_train","SportsMOT_val","SportsMOT_mix","SportsMOT_test","MOT17-train","MOT17-train_half","MOT20-train","MOT17-val_half"
    DATASETS_NAME:
    - SportsMOT_train
    DATASETS_RATIO:
    - 1
    SAMPLE_PER_EPOCH: 60000
  VAL:
    DATASETS_NAME:
    - SportsMOT_val
    DATASETS_RATIO:
    - 1
    SAMPLE_PER_EPOCH: 10000
MODEL:
  BACKBONE:
    PRETRAINED: True 
    PRETRAINED_PATH: '/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.ta

In [85]:
MIXSORT_PRETRAINED = (
    MIXSORT_DIR
    / "pretrained"
    / "MixFormer_sports_train.pth.tar"
)

print("Checkpoint:")
print(MIXSORT_PRETRAINED)
print("Exists:", MIXSORT_PRETRAINED.exists())

if MIXSORT_PRETRAINED.exists():
    print(f"Size: {MIXSORT_PRETRAINED.stat().st_size / (1024**2):.2f} MB")

Checkpoint:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar
Exists: True
Size: 375.89 MB


In [86]:
README = MIXSORT_DIR / "README.md"

text = README.read_text(encoding="utf-8")

for i, line in enumerate(text.splitlines(), start=1):
    if any(x in line.lower() for x in [
        "google",
        "baidu",
        "mixformer_sports",
        "model zoo",
        "pretrained models",
    ]):
        print(f"{i}: {line}")

99: ## Model zoo
101: The trained models are provided in [[google]](https://drive.google.com/drive/folders/1pQs1gFC_jG0TlGIUMgf3E0I3OztCvgxI?usp=sharing), [[baidu (code:7438)]](https://pan.baidu.com/s/1YAP1zKtx-M_ay6uZINoCHg). For YOLOX model trained on MOT17 or DanceTrack, you can refer to [ByteTrack](https://github.com/ifzhang/ByteTrack#model-zoo) and [DanceTrack](https://github.com/DanceTrack/DanceTrack/blob/main/ByteTrack/README.md). 
103: After downloading the pretrained models, you can put them under `<MixSort_HOME>/pretrained`.
109: The COCO pretrained YOLOX model (`yolox_x.pth`) can be downloaded from their [model zoo](https://github.com/Megvii-BaseDetection/YOLOX/tree/0.1.0). 


In [87]:
from pathlib import Path

MIXSORT_DIR = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort"
)

MIXSORT_PRETRAINED = (
    MIXSORT_DIR / "pretrained" / "MixFormer_sports_train.pth.tar"
)

print("Checkpoint:")
print(MIXSORT_PRETRAINED)
print("Exists:", MIXSORT_PRETRAINED.exists())

if MIXSORT_PRETRAINED.exists():
    print(f"Size: {MIXSORT_PRETRAINED.stat().st_size / (1024**2):.2f} MB")

Checkpoint:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar
Exists: True
Size: 375.89 MB


In [88]:
from pathlib import Path

for p in [
    MIXSORT_DIR / "MixViT",
    MIXSORT_DIR / "MixViT" / "lib",
    MIXSORT_DIR / "yolox",
]:
    print(f"\n{p}")
    if p.exists():
        for x in list(p.iterdir())[:20]:
            print("  ", x.name)


/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT
   experiments
   external
   install_pytorch17.sh
   lib
   LICENSE
   README.md
   requirements.txt
   segmentation
   tracking

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib
   config
   models
   test
   train
   utils
   __init__.py
   __pycache__

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/yolox
   byte_tracker
   core
   data
   deepsort_tracker
   evaluators
   exp
   layers
   mixsort_oc_tracker
   mixsort_tracker
   models
   motdt_tracker
   ocsort_tracker
   sort_tracker
   tracking_utils
   utils
   __init__.py
   __pycache__


In [89]:
CONFIG_DIR = MIXSORT_DIR / "MixViT" / "lib" / "config"

print("Config directory:", CONFIG_DIR)
print("Exists:", CONFIG_DIR.exists())
print()

for p in CONFIG_DIR.iterdir():
    print(p.name)

Config directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/config
Exists: True

mixformer
mixformerpp_vit_multi
mixformer_deit
mixformer_deit_multi
mixformer_online
mixformer_vit
mixformer_vit_decoder
mixformer_vit_learn_pos
mixformer_vit_multi
__init__.py
__pycache__


In [90]:
MIXFORMER_CONFIG_DIR = (
    MIXSORT_DIR / "MixViT" / "lib" / "config" / "mixformer_deit"
)

print("Directory:", MIXFORMER_CONFIG_DIR)
print("Exists:", MIXFORMER_CONFIG_DIR.exists())
print()

for p in MIXFORMER_CONFIG_DIR.iterdir():
    print(p.name)

Directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/config/mixformer_deit
Exists: True

config.py
__pycache__


In [91]:
CONFIG_FILE = MIXFORMER_CONFIG_DIR / "config.py"

print(CONFIG_FILE.read_text(encoding="utf-8"))

from easydict import EasyDict as edict
import yaml

"""
Add default config for MixFormer.
"""
cfg = edict()

# MODEL
cfg.MODEL = edict()
cfg.MODEL.HEAD_TYPE = "Heatmap"
cfg.MODEL.HIDDEN_DIM = 384
cfg.MODEL.NUM_OBJECT_QUERIES = 1
cfg.MODEL.POSITION_EMBEDDING = 'sine'  # sine or learned
cfg.MODEL.PREDICT_MASK = False
cfg.MODEL.VIT_TYPE = 'deit3_4layers'
# MODEL.BACKBONE
cfg.MODEL.BACKBONE = edict()
cfg.MODEL.BACKBONE.PRETRAINED = True
cfg.MODEL.BACKBONE.PRETRAINED_PATH = ''
cfg.MODEL.BACKBONE.INIT = 'trunc_norm'
cfg.MODEL.BACKBONE.NUM_STAGES = 3
cfg.MODEL.BACKBONE.PATCH_SIZE = [ 7, 3, 3 ]
cfg.MODEL.BACKBONE.PATCH_STRIDE = [ 4, 2, 2 ]
cfg.MODEL.BACKBONE.PATCH_PADDING = [ 2, 1, 1 ]
cfg.MODEL.BACKBONE.DIM_EMBED = [ 64, 192, 384 ]
cfg.MODEL.BACKBONE.NUM_HEADS = [ 1, 3, 6 ]
cfg.MODEL.BACKBONE.DEPTH = [ 1, 2, 10 ]
cfg.MODEL.BACKBONE.MLP_RATIO = [ 4.0, 4.0, 4.0 ]
cfg.MODEL.BACKBONE.ATTN_DROP_RATE = [ 0.0, 0.0, 0.0 ]
cfg.MODEL.BACKBONE.DROP_RATE = [ 0.0, 0.0, 0.0 ]
cfg.MODEL.BACKBONE.DROP_PATH_R

In [92]:
import sys
from pathlib import Path

MIXVIT_DIR = Path(
    "/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT"
)

# Put MixViT FIRST in Python's module search path
sys.path.insert(0, str(MIXVIT_DIR))

print("MixViT exists:", MIXVIT_DIR.exists())
print("lib exists:", (MIXVIT_DIR / "lib").exists())
print("sys.path[0]:", sys.path[0])

# Check what Python actually resolves
import importlib.util

spec = importlib.util.find_spec("lib")
print("lib spec:", spec)

MixViT exists: True
lib exists: True
sys.path[0]: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT
lib spec: ModuleSpec(name='lib', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7d7dccd6ac30>, origin='/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/__init__.py', submodule_search_locations=['/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib'])


In [93]:
import sys

# Remove the conflicting already-loaded "lib" package
for name in list(sys.modules):
    if name == "lib" or name.startswith("lib."):
        del sys.modules[name]

# Make MixViT the first search location
MIXVIT_DIR = MIXSORT_DIR / "MixViT"

if str(MIXVIT_DIR) in sys.path:
    sys.path.remove(str(MIXVIT_DIR))

sys.path.insert(0, str(MIXVIT_DIR))

# Check what Python now resolves
import importlib.util

spec = importlib.util.find_spec("lib")

print("Resolved lib:")
print(spec)
print()
print("Expected:")
print(MIXVIT_DIR / "lib" / "__init__.py")

Resolved lib:
ModuleSpec(name='lib', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7d7dcc8de930>, origin='/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/__init__.py', submodule_search_locations=['/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib'])

Expected:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/__init__.py


In [94]:
!pip install easydict


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [95]:
from lib.config.mixformer_deit.config import cfg, update_config_from_file

TRACK_CONFIG = (
    MIXSORT_DIR
    / "MixViT"
    / "experiments"
    / "mixformer_deit"
    / "track.yaml"
)

update_config_from_file(str(TRACK_CONFIG))

print("Configuration loaded successfully")
print("VIT_TYPE:", cfg.MODEL.VIT_TYPE)
print("HEAD_TYPE:", cfg.MODEL.HEAD_TYPE)
print("PRETRAINED:", cfg.MODEL.BACKBONE.PRETRAINED)
print("PRETRAINED_PATH:", cfg.MODEL.BACKBONE.PRETRAINED_PATH)
print("HIDDEN_DIM:", cfg.MODEL.HIDDEN_DIM)

Configuration loaded successfully
VIT_TYPE: deit3_4layers
HEAD_TYPE: Heatmap
PRETRAINED: True
PRETRAINED_PATH: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar
HIDDEN_DIM: 768


In [96]:
MODELS_DIR = MIXSORT_DIR / "MixViT" / "lib" / "models"

print("Models directory:", MODELS_DIR)
print("Exists:", MODELS_DIR.exists())
print()

for p in MODELS_DIR.iterdir():
    print(p.name)

Models directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models
Exists: True

losses
mixformer
mixformer_vit
__init__.py
__pycache__


In [97]:
MIXFORMER_DIR = MODELS_DIR / "mixformer"

print("Directory:", MIXFORMER_DIR)
print("Exists:", MIXFORMER_DIR.exists())
print()

for p in MIXFORMER_DIR.iterdir():
    print(p.name)

Directory: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer
Exists: True

head.py
mixformer.py
mixformer_online.py
mixformer_vit_multi_stage_ablation.py
utils.py
__init__.py
__pycache__


In [98]:
MIXFORMER_FILE = MIXFORMER_DIR / "mixformer.py"

text = MIXFORMER_FILE.read_text(encoding="utf-8")

for i, line in enumerate(text.splitlines(), start=1):
    if (
        "def build_" in line
        or "class Mix" in line
        or "def build" in line
    ):
        print(f"{i}: {line}")

765: class MixFormer(nn.Module):
818: def build_mixformer(cfg):


In [99]:
print("\n".join(
    text.splitlines()[800:850]
))

        self.backbone.set_online(template, online_template)

    def forward_box_head(self, search):
        """
        :param search: (b, c, h, w)
        :return:
        """
        if self.head_type == "CORNER":
            # run the corner head
            b = search.size(0)
            outputs_coord = box_xyxy_to_cxcywh(self.box_head(search))
            outputs_coord_new = outputs_coord.view(b, 1, 4)
            out = {'pred_boxes': outputs_coord_new}
            return out, outputs_coord_new
        else:
            raise KeyError

def build_mixformer(cfg):
    backbone = get_mixformer_model(cfg)  # backbone without positional encoding and attention mask
    box_head = build_box_head(cfg)  # a simple corner head
    model = MixFormer(
        backbone,
        box_head,
        head_type=cfg.MODEL.HEAD_TYPE
    )

    return model


In [100]:
# Find where get_mixformer_model is defined
import re

for root in [
    MIXSORT_DIR / "MixViT" / "lib" / "models",
]:
    for file in root.rglob("*.py"):
        content = file.read_text(encoding="utf-8", errors="ignore")
        if "def get_mixformer_model" in content:
            print("Found in:")
            print(file)
            print()

            lines = content.splitlines()
            for i, line in enumerate(lines):
                if "def get_mixformer_model" in line:
                    print("\n".join(
                        f"{j+1}: {lines[j]}"
                        for j in range(max(0, i-5), min(len(lines), i+40))
                    ))

Found in:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer/mixformer.py

735:     def set_online(self, template, online_template):
736:         for i in range(self.num_stages):
737:             template, online_template = getattr(self, f'stage{i}').set_online(template, online_template)
738: 
739: 
740: def get_mixformer_model(config, **kwargs):
741:     msvit_spec = config.MODEL.BACKBONE
742:     msvit = ConvolutionalVisionTransformer(
743:         in_chans=3,
744:         act_layer=QuickGELU,
745:         norm_layer=partial(LayerNorm, eps=1e-5),
746:         init=getattr(msvit_spec, 'INIT', 'trunc_norm'),
747:         spec=msvit_spec
748:     )
749: 
750:     if config.MODEL.BACKBONE.PRETRAINED:
751:         try:
752:             ckpt_path = config.MODEL.BACKBONE.PRETRAINED_PATH
753:             ckpt = torch.load(ckpt_path, map_location='cpu')
754:             missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)
755:  

In [101]:
cfg.MODEL.BACKBONE.PRETRAINED_PATH = str(MIXSORT_PRETRAINED)

print("Checkpoint configured as:")
print(cfg.MODEL.BACKBONE.PRETRAINED_PATH)
print()
print("Exists:", MIXSORT_PRETRAINED.exists())

Checkpoint configured as:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar

Exists: True


In [102]:
!pip install einops
!pip install timm


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [103]:
from lib.models.mixformer.mixformer import build_mixformer

print("build_mixformer imported successfully")
print(build_mixformer)

/home/jatin/tf312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


build_mixformer imported successfully
<function build_mixformer at 0x7d7db9f43920>


In [104]:
import time

print("Building MixFormer...")
t0 = time.time()

network = build_mixformer(cfg)

elapsed = time.time() - t0

print(f"\nBuild finished in {elapsed:.2f} seconds")
print("Model type:", type(network))

Building MixFormer...
head channel: 384

Build finished in 5.87 seconds
Model type: <class 'lib.models.mixformer.mixformer.MixFormer'>


In [105]:
import torch

print("Loading trusted MixSort checkpoint...")

checkpoint = torch.load(
    str(MIXSORT_PRETRAINED),
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully!")
print("Type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Keys:", list(checkpoint.keys())[:20])

Loading trusted MixSort checkpoint...


I0000 00:00:1791018564.131975     888 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791018569.520735     888 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Checkpoint loaded successfully!
Type: <class 'dict'>
Keys: ['epoch', 'actor_type', 'net_type', 'net', 'net_info', 'constructor', 'optimizer', 'stats', 'settings']


In [106]:
net_state = checkpoint["net"]

print("net type:", type(net_state))

if isinstance(net_state, dict):
    print("Number of parameters:", len(net_state))
    print("\nFirst 15 keys:")
    for key in list(net_state.keys())[:15]:
        value = net_state[key]
        print(f"{key:70s} {tuple(value.shape) if hasattr(value, 'shape') else type(value)}")

net type: <class 'collections.OrderedDict'>
Number of parameters: 88

First 15 keys:
backbone.cls_token                                                     (1, 1, 768)
backbone.pos_embed                                                     (1, 197, 768)
backbone.pos_embed_s                                                   (1, 196, 768)
backbone.pos_embed_t                                                   (1, 36, 768)
backbone.patch_embed.proj.weight                                       (768, 3, 16, 16)
backbone.patch_embed.proj.bias                                         (768,)
backbone.blocks.0.norm1.weight                                         (768,)
backbone.blocks.0.norm1.bias                                           (768,)
backbone.blocks.0.attn.qkv.weight                                      (2304, 768)
backbone.blocks.0.attn.qkv.bias                                        (2304,)
backbone.blocks.0.attn.proj.weight                                     (768, 768)
backbone.blo

In [107]:
print("net_type:")
print(checkpoint["net_type"])

print("\nactor_type:")
print(checkpoint["actor_type"])

print("\nconstructor:")
print(checkpoint["constructor"])

print("\nnet_info:")
print(checkpoint["net_info"])

print("\nsettings:")
print(checkpoint["settings"])

net_type:
MixFormer

actor_type:
MixFormerActor

constructor:
None

net_info:
None

settings:


In [108]:
settings = checkpoint["settings"]

print("Settings attributes:")
for name in dir(settings):
    if not name.startswith("_"):
        try:
            value = getattr(settings, name)
            if not callable(value):
                print(f"{name}: {value}")
        except Exception:
            pass

Settings attributes:
batchsize: 96
center_jitter_factor: {'template': 0, 'search': 3}
cfg: {'MODEL': {'HEAD_TYPE': 'Heatmap', 'HIDDEN_DIM': 768, 'NUM_OBJECT_QUERIES': 1, 'POSITION_EMBEDDING': 'sine', 'PREDICT_MASK': False, 'VIT_TYPE': 'deit3_4layers', 'BACKBONE': {'PRETRAINED': True, 'PRETRAINED_PATH': '/data3/zck/MixTrack/pretrained/MixFormerOnlineScore_ep0600.pth.tar', 'INIT': 'trunc_norm', 'NUM_STAGES': 3, 'PATCH_SIZE': [7, 3, 3], 'PATCH_STRIDE': [4, 2, 2], 'PATCH_PADDING': [2, 1, 1], 'DIM_EMBED': [64, 192, 768], 'NUM_HEADS': [1, 3, 12], 'DEPTH': [1, 2, 9], 'MLP_RATIO': [4.0, 4.0, 4.0], 'ATTN_DROP_RATE': [0.0, 0.0, 0.0], 'DROP_RATE': [0.0, 0.0, 0.0], 'DROP_PATH_RATE': [0.0, 0.0, 0.1], 'QKV_BIAS': [True, True, True], 'CLS_TOKEN': [False, False, False], 'POS_EMBED': [False, False, False], 'QKV_PROJ_METHOD': ['dw_bn', 'dw_bn', 'dw_bn'], 'KERNEL_QKV': [3, 3, 3], 'PADDING_KV': [1, 1, 1], 'STRIDE_KV': [2, 2, 2], 'PADDING_Q': [1, 1, 1], 'STRIDE_Q': [1, 1, 1], 'FREEZE_BN': False}}, 'TRAIN':

In [109]:
# Keys expected by the MixSort model we constructed
model_keys = list(network.state_dict().keys())

# Keys stored in the downloaded checkpoint
checkpoint_keys = list(checkpoint["net"].keys())

print("Constructed model parameters:", len(model_keys))
print("Checkpoint parameters:", len(checkpoint_keys))

print("\nConstructed model — first 20 keys:")
for k in model_keys[:20]:
    print(k)

print("\nCheckpoint — first 20 keys:")
for k in checkpoint_keys[:20]:
    print(k)

Constructed model parameters: 346
Checkpoint parameters: 88

Constructed model — first 20 keys:
backbone.stage0.patch_embed.proj.weight
backbone.stage0.patch_embed.proj.bias
backbone.stage0.patch_embed.norm.weight
backbone.stage0.patch_embed.norm.bias
backbone.stage0.blocks.0.norm1.weight
backbone.stage0.blocks.0.norm1.bias
backbone.stage0.blocks.0.attn.conv_proj_q.conv.weight
backbone.stage0.blocks.0.attn.conv_proj_q.bn.weight
backbone.stage0.blocks.0.attn.conv_proj_q.bn.bias
backbone.stage0.blocks.0.attn.conv_proj_k.conv.weight
backbone.stage0.blocks.0.attn.conv_proj_k.bn.weight
backbone.stage0.blocks.0.attn.conv_proj_k.bn.bias
backbone.stage0.blocks.0.attn.conv_proj_v.conv.weight
backbone.stage0.blocks.0.attn.conv_proj_v.bn.weight
backbone.stage0.blocks.0.attn.conv_proj_v.bn.bias
backbone.stage0.blocks.0.attn.proj_q.weight
backbone.stage0.blocks.0.attn.proj_q.bias
backbone.stage0.blocks.0.attn.proj_k.weight
backbone.stage0.blocks.0.attn.proj_k.bias
backbone.stage0.blocks.0.attn.proj

In [110]:
BASELINE_CONFIG = (
    MIXSORT_DIR
    / "MixViT"
    / "experiments"
    / "mixformer_deit"
    / "baseline.yaml"
)

print("Baseline config:")
print(BASELINE_CONFIG)
print()
print(BASELINE_CONFIG.read_text(encoding="utf-8"))

Baseline config:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/experiments/mixformer_deit/baseline.yaml

DATA:
  MAX_SAMPLE_INTERVAL: 10
  MEAN:
  - 0.485
  - 0.456
  - 0.406
  SEARCH:
    CENTER_JITTER: 3
    FACTOR: 4.0
    SCALE_JITTER: 0.3
    SIZE: 224
  STD:
  - 0.229
  - 0.224
  - 0.225
  TEMPLATE:
    CENTER_JITTER: 0
    FACTOR: 1.5
    SCALE_JITTER: 0
    SIZE: 96
    NUMBER: 1
  TRAIN:
    # DATASETS_NAME: "SoccerNet_train","SoccerNet_test","DanceTrack_train","DanceTrack_val","SportsMOT_train","SportsMOT_val","SportsMOT_mix","MOT17-train","MOT17-train_half","MOT17-val_half"
    DATASETS_NAME:
    - SportsMOT_train
    DATASETS_RATIO:
    - 1
    SAMPLE_PER_EPOCH: 60000
  VAL:
    DATASETS_NAME:
    - SportsMOT_val
    DATASETS_RATIO:
    - 1
    SAMPLE_PER_EPOCH: 10000
MODEL:
  BACKBONE:
    PRETRAINED: True 
    PRETRAINED_PATH: 'pretrained/MixFormerOnlineScore.pth.tar'
    INIT: 'trunc_norm'
    NUM_STAGES: 3
    PATCH_SIZE: [ 7, 3, 3 ]
    PATC

In [111]:
PRETRAINED_DIR = MIXSORT_DIR / "pretrained"

print("Pretrained directory:")
print(PRETRAINED_DIR)
print()

for p in sorted(PRETRAINED_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name:45s} {p.stat().st_size / (1024**2):8.2f} MB")

Pretrained directory:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained

MixFormerOnlineScore.pth.tar                    439.43 MB
MixFormer_sports_train.pth.tar                  375.89 MB


In [112]:
net_state = checkpoint["net"]

print("Total checkpoint tensors:", len(net_state))
print()

for i, (key, value) in enumerate(net_state.items()):
    print(f"{i:3d}  {key:65s}  {tuple(value.shape)}")

Total checkpoint tensors: 88

  0  backbone.cls_token                                                 (1, 1, 768)
  1  backbone.pos_embed                                                 (1, 197, 768)
  2  backbone.pos_embed_s                                               (1, 196, 768)
  3  backbone.pos_embed_t                                               (1, 36, 768)
  4  backbone.patch_embed.proj.weight                                   (768, 3, 16, 16)
  5  backbone.patch_embed.proj.bias                                     (768,)
  6  backbone.blocks.0.norm1.weight                                     (768,)
  7  backbone.blocks.0.norm1.bias                                       (768,)
  8  backbone.blocks.0.attn.qkv.weight                                  (2304, 768)
  9  backbone.blocks.0.attn.qkv.bias                                    (2304,)
 10  backbone.blocks.0.attn.proj.weight                                 (768, 768)
 11  backbone.blocks.0.attn.proj.bias                   

In [113]:
ONLINE_CKPT = (
    MIXSORT_DIR
    / "pretrained"
    / "MixFormerOnlineScore.pth.tar"
)

print("Checkpoint:")
print(ONLINE_CKPT)
print("Exists:", ONLINE_CKPT.exists())

if ONLINE_CKPT.exists():
    print(f"Size: {ONLINE_CKPT.stat().st_size / (1024**2):.2f} MB")

Checkpoint:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormerOnlineScore.pth.tar
Exists: True
Size: 439.43 MB


In [114]:
ONLINE_CKPT = (
    MIXSORT_DIR
    / "pretrained"
    / "MixFormerOnlineScore.pth.tar"
)

cfg.MODEL.BACKBONE.PRETRAINED = True
cfg.MODEL.BACKBONE.PRETRAINED_PATH = str(ONLINE_CKPT)

print("Configured checkpoint:")
print(cfg.MODEL.BACKBONE.PRETRAINED_PATH)
print("Exists:", ONLINE_CKPT.exists())

Configured checkpoint:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormerOnlineScore.pth.tar
Exists: True


In [115]:
from lib.config.mixformer_deit.config import cfg, update_config_from_file

BASELINE_CONFIG = (
    MIXSORT_DIR
    / "MixViT"
    / "experiments"
    / "mixformer_deit"
    / "baseline.yaml"
)

update_config_from_file(str(BASELINE_CONFIG))

# Override the relative checkpoint path with our actual absolute path
cfg.MODEL.BACKBONE.PRETRAINED = True
cfg.MODEL.BACKBONE.PRETRAINED_PATH = str(ONLINE_CKPT)

print("Baseline configuration loaded")
print("VIT_TYPE:", cfg.MODEL.VIT_TYPE)
print("HIDDEN_DIM:", cfg.MODEL.HIDDEN_DIM)
print("Checkpoint:", cfg.MODEL.BACKBONE.PRETRAINED_PATH)

Baseline configuration loaded
VIT_TYPE: deit3_4layers
HIDDEN_DIM: 768
Checkpoint: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormerOnlineScore.pth.tar


In [116]:
import time

print("Building MixFormer with MixFormerOnlineScore...")
t0 = time.time()

network = build_mixformer(cfg)

elapsed = time.time() - t0

print(f"\nBuild finished in {elapsed:.2f} seconds")
print("Model type:", type(network))

Building MixFormer with MixFormerOnlineScore...
head channel: 384

Build finished in 8.00 seconds
Model type: <class 'lib.models.mixformer.mixformer.MixFormer'>


In [117]:
import torch

print("Loading MixFormerOnlineScore directly...")

online_checkpoint = torch.load(
    str(ONLINE_CKPT),
    map_location="cpu",
    weights_only=False
)

print("Loaded successfully!")
print("Type:", type(online_checkpoint))

if isinstance(online_checkpoint, dict):
    print("Keys:")
    print(list(online_checkpoint.keys()))

Loading MixFormerOnlineScore directly...
Loaded successfully!
Type: <class 'dict'>
Keys:
['epoch', 'actor_type', 'net_type', 'net', 'net_info', 'constructor', 'optimizer', 'stats', 'settings']


In [118]:
online_net = online_checkpoint["net"]

print("Number of tensors:", len(online_net))
print()

for i, (key, value) in enumerate(online_net.items()):
    if i >= 20:
        break

    shape = tuple(value.shape) if hasattr(value, "shape") else type(value)
    print(f"{i:2d}  {key:65s} {shape}")

Number of tensors: 155

 0  backbone.cls_token                                                (1, 1, 768)
 1  backbone.pos_embed                                                (1, 197, 768)
 2  backbone.pos_embed_s                                              (1, 324, 768)
 3  backbone.pos_embed_t                                              (1, 64, 768)
 4  backbone.patch_embed.proj.weight                                  (768, 3, 16, 16)
 5  backbone.patch_embed.proj.bias                                    (768,)
 6  backbone.blocks.0.norm1.weight                                    (768,)
 7  backbone.blocks.0.norm1.bias                                      (768,)
 8  backbone.blocks.0.attn.qkv.weight                                 (2304, 768)
 9  backbone.blocks.0.attn.qkv.bias                                   (2304,)
10  backbone.blocks.0.attn.proj.weight                                (768, 768)
11  backbone.blocks.0.attn.proj.bias                                  (768,)
12  bac

In [119]:
import re

SEARCH_ROOT = MIXSORT_DIR / "MixViT" / "lib"

patterns = [
    "checkpoint['net']",
    'checkpoint["net"]',
    "load_state_dict",
]

for file in SEARCH_ROOT.rglob("*.py"):
    text = file.read_text(encoding="utf-8", errors="ignore")

    if "load_state_dict" in text and (
        "checkpoint" in text or "ckpt" in text
    ):
        print("\nFILE:", file)

        for i, line in enumerate(text.splitlines(), start=1):
            if "load_state_dict" in line:
                print(f"{i}: {line.strip()}")


FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer/mixformer.py
754: missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)

FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer/mixformer_online.py
810: missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)
913: missing_keys, unexpected_keys = model.load_state_dict(ckpt['net'], strict=False)

FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer/mixformer_vit_multi_stage_ablation.py
662: missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)

FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer_vit/mixformerpp_vit_multi_score.py
324: missing_keys, unexpected_keys = vit.load_state_dict(new_dict, strict=False)
448: # missing_keys, unexpected_keys = model.load_state_dict(ckpt, strict=False)
4

In [120]:
ONLINE_MODEL_FILE = (
    MIXSORT_DIR
    / "MixViT"
    / "lib"
    / "models"
    / "mixformer"
    / "mixformer_online.py"
)

text_online = ONLINE_MODEL_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

for i, line in enumerate(text_online.splitlines(), start=1):
    if "def build_" in line or "load_state_dict" in line:
        print(f"{i}: {line}")

810:             missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)
898: def build_mixformer_online_score(cfg, settings=None, train=True):
913:             missing_keys, unexpected_keys = model.load_state_dict(ckpt['net'], strict=False)


In [121]:
lines = text_online.splitlines()

print("\n".join(
    f"{i+1}: {lines[i]}"
    for i in range(890, 921)
))

891:             outputs_coord = box_xyxy_to_cxcywh(self.box_head(search))
892:             outputs_coord_new = outputs_coord.view(b, 1, 4)
893:             out = {'pred_boxes': outputs_coord_new}
894:             return out, outputs_coord_new
895:         else:
896:             raise KeyError
897: 
898: def build_mixformer_online_score(cfg, settings=None, train=True):
899:     backbone = get_mixformer_online_model(cfg)  # backbone without positional encoding and attention mask
900:     box_head = build_box_head(cfg)  # a simple corner head
901:     score_branch = ScoreDecoder(cfg, pool_size=4) # the proposed score prediction module (SPM)
902:     model = MixFormerOnlineScore(
903:         backbone,
904:         box_head,
905:         score_branch,
906:         head_type=cfg.MODEL.HEAD_TYPE
907:     )
908: 
909:     if cfg.MODEL.PRETRAINED_STAGE1 and train:
910:         try:
911:             ckpt_path = settings.stage1_model
912:             ckpt = torch.load(ckpt_path, map_location='c

In [122]:
# Find the online backbone builder
for i, line in enumerate(text_online.splitlines(), start=1):
    if "def get_mixformer_online_model" in line:
        print(f"Found at line {i}")
        print("\n".join(
            f"{j+1}: {text_online.splitlines()[j]}"
            for j in range(i-1, min(len(text_online.splitlines()), i+45))
        ))

Found at line 796
796: def get_mixformer_online_model(config, **kwargs):
797:     msvit_spec = config.MODEL.BACKBONE
798:     msvit = ConvolutionalVisionTransformer(
799:         in_chans=3,
800:         act_layer=QuickGELU,
801:         norm_layer=partial(LayerNorm, eps=1e-5),
802:         init=getattr(msvit_spec, 'INIT', 'trunc_norm'),
803:         spec=msvit_spec
804:     )
805: 
806:     if config.MODEL.BACKBONE.PRETRAINED:
807:         try:
808:             ckpt_path = config.MODEL.BACKBONE.PRETRAINED_PATH
809:             ckpt = torch.load(ckpt_path, map_location='cpu')
810:             missing_keys, unexpected_keys = msvit.load_state_dict(ckpt, strict=False)
811:             if is_main_process():
812:                 print("missing keys:", missing_keys)
813:                 print("unexpected keys:", unexpected_keys)
814:                 print("Loading pretrained CVT done.")
815:         except:
816:             print("Warning: Pretrained CVT weights are not loaded")
817: 
818:  

In [123]:
SEARCH_ROOT = MIXSORT_DIR / "MixViT"

for file in SEARCH_ROOT.rglob("*.py"):
    text_file = file.read_text(encoding="utf-8", errors="ignore")

    if "NLAYER_HEAD" in text_file:
        print(f"\nFILE: {file}")

        for i, line in enumerate(text_file.splitlines(), start=1):
            if "NLAYER_HEAD" in line:
                print(f"{i}: {line.strip()}")


FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/config/mixformer_online/config.py
43: cfg.MODEL.NLAYER_HEAD = 3

FILE: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer/mixformer_online.py
743: self.score_head = MLP(cfg.MODEL.HIDDEN_DIM, cfg.MODEL.HIDDEN_DIM, 1, cfg.MODEL.NLAYER_HEAD)


In [124]:
cfg.MODEL.NLAYER_HEAD = 3

print("NLAYER_HEAD:", cfg.MODEL.NLAYER_HEAD)
print("HIDDEN_DIM:", cfg.MODEL.HIDDEN_DIM)
print("HEAD_TYPE:", cfg.MODEL.HEAD_TYPE)

NLAYER_HEAD: 3
HIDDEN_DIM: 768
HEAD_TYPE: Heatmap


In [125]:
DEIT_FILES = list((MIXSORT_DIR / "MixViT/lib/models").rglob("*deit*.py"))

for f in DEIT_FILES:
    print(f)

/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer_vit/mixformer_deit.py
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/MixViT/lib/models/mixformer_vit/mixformer_deit_multi_score.py


In [126]:
for file in [
    MIXSORT_DIR / "MixViT/lib/models/mixformer_vit/mixformer_deit.py",
    MIXSORT_DIR / "MixViT/lib/models/mixformer_vit/mixformer_deit_multi_score.py",
]:
    print(f"\n{'='*80}")
    print(file.name)
    print(f"{'='*80}")

    lines = file.read_text(encoding="utf-8", errors="ignore").splitlines()

    for i, line in enumerate(lines, start=1):
        if any(x in line for x in [
            "class MixFormer",
            "def build",
            "def get_mixformer",
            "load_state_dict",
            "VIT_TYPE",
        ]):
            print(f"{i}: {line.strip()}")


mixformer_deit.py
197: def get_mixformer_vit(config):
200: if config.MODEL.VIT_TYPE == 'large_patch16':
205: elif config.MODEL.VIT_TYPE == 'base_patch16':
210: elif config.MODEL.VIT_TYPE == 'deit3_4layers':
216: raise KeyError(f"VIT_TYPE shoule set to 'large_patch16' or 'base_patch16'")
221: class MixFormer(nn.Module):
271: def build_mixformer_deit(cfg):
287: missing_keys, unexpected_keys = model.load_state_dict(new_dict, strict=False)

mixformer_deit_multi_score.py
287: def get_mixformer_vit(config, train):
290: if config.MODEL.VIT_TYPE == 'large_patch16':
295: elif config.MODEL.VIT_TYPE == 'base_patch16':
301: raise KeyError(f"VIT_TYPE shoule set to 'large_patch16' or 'base_patch16'")
338: missing_keys, unexpected_keys = vit.load_state_dict(new_dict, strict=False)
349: class MixFormerOnlineScore(nn.Module):
427: def build_mixformer_deit_multi_score(cfg, settings=None, train=True):
462: # missing_keys, unexpected_keys = model.load_state_dict(ckpt, strict=False)
472: missing_keys, une

In [127]:
from pathlib import Path

file = MIXSORT_DIR / "MixViT/lib/models/mixformer_vit/mixformer_deit.py"
lines = file.read_text(encoding="utf-8", errors="ignore").splitlines()

for start, end in [(197, 220), (271, 290)]:
    print(f"\n--- lines {start}-{end} ---")
    for i in range(start, end + 1):
        print(f"{i}: {lines[i-1]}")


--- lines 197-220 ---
197: def get_mixformer_vit(config):
198:     img_size_s = config.DATA.SEARCH.SIZE
199:     img_size_t = config.DATA.TEMPLATE.SIZE
200:     if config.MODEL.VIT_TYPE == 'large_patch16':
201:         vit = VisionTransformer(
202:             img_size_s=img_size_s, img_size_t=img_size_t,
203:             patch_size=16, embed_dim=1024, depth=24, num_heads=16, mlp_ratio=4, qkv_bias=True,
204:             norm_layer=partial(nn.LayerNorm, eps=1e-6), drop_path_rate=0.1)
205:     elif config.MODEL.VIT_TYPE == 'base_patch16':
206:         vit = VisionTransformer(
207:             img_size_s=img_size_s, img_size_t=img_size_t,
208:             patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True,
209:             norm_layer=partial(nn.LayerNorm, eps=1e-6), drop_path_rate=0.1)
210:     elif config.MODEL.VIT_TYPE == 'deit3_4layers':
211:         vit = VisionTransformer(
212:             img_size_s=img_size_s, img_size_t=img_size_t,
213:             p

In [128]:
# Use the exact pretrained MixFormer checkpoint
cfg.MODEL.VIT_TYPE = "deit3_4layers"
cfg.MODEL.BACKBONE.PRETRAINED = True
cfg.MODEL.BACKBONE.PRETRAINED_PATH = str(
    MIXSORT_DIR / "pretrained/MixFormerOnlineScore.pth.tar"
)

print("VIT_TYPE:", cfg.MODEL.VIT_TYPE)
print("Checkpoint:", cfg.MODEL.BACKBONE.PRETRAINED_PATH)

VIT_TYPE: deit3_4layers
Checkpoint: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormerOnlineScore.pth.tar


In [129]:
import numpy as np

# Compatibility for old MixSort code with NumPy 2.x
if not hasattr(np, "float"):
    np.float = float

print("np.float compatibility:", np.float)

np.float compatibility: <class 'float'>


In [130]:
import torch

_original_torch_load = torch.load

def torch_load_legacy(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = torch_load_legacy

print("torch.load patched: weights_only=False")

torch.load patched: weights_only=False


In [137]:
from lib.models.mixformer_vit.mixformer_deit import build_mixformer_deit

print("build_mixformer_deit imported successfully")

build_mixformer_deit imported successfully


In [138]:
model = build_mixformer_deit(cfg)

print("\nModel loaded successfully.")
print("Parameters:", sum(p.numel() for p in model.parameters()))

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t', 'box_head.conv1.0.weight', 'box_head.conv1.0.bias', 'box_head.conv1.1.weight', 'box_head.conv1.1.bias', 'box_head.conv1.1.running_mean', 'box_head.conv1.1.running_var', 'box_head.conv2.0.weight', 'box_head.conv2.0.bias', 'box_head.conv2.1.weight', 'box_head.conv2.1.bias', 'box_head.conv2.1.running_mean', 'box_head.conv2.1.running_var', 'box_head.conv3.0.weight', 'box_head.conv3.0.bias', 'box_head.conv3.1.weight', 'box_head.conv3.1.bias', 'box_head.conv3.1.running_mean', 'box_head.conv3.1.running_var', 'box_head.conv4.0.weight', 'box_head.conv4.0.bias', 'box_head.conv4.1.weight', 'box_head.conv4.1.bias', 'box_head.conv4.1.running_mean', 'box_head.conv4.1.running_var', 'box_head.conv5.weight', 'box_head.conv5.bias']
unexpected keys: ['score_branch.score_token', 'score_branch.score_head.layers.0.weight', 'score_branch.score_head.layers.0.bias', 'score_branch.score_head.layers.1.weight', 

In [139]:
import inspect

model = model.cuda().eval()

print("Device:", next(model.parameters()).device)
print("\nForward signature:")
print(inspect.signature(model.forward))

Device: cuda:0

Forward signature:
(template, search)


In [140]:
import torch

template = torch.randn(1, 3, cfg.DATA.TEMPLATE.SIZE, cfg.DATA.TEMPLATE.SIZE, device="cuda")
search = torch.randn(1, 3, cfg.DATA.SEARCH.SIZE, cfg.DATA.SEARCH.SIZE, device="cuda")

with torch.no_grad():
    output = model(template, search)

print("Template:", template.shape)
print("Search:", search.shape)
print("Output type:", type(output))

if isinstance(output, dict):
    print("Output keys:", output.keys())
else:
    print("Output shape:", getattr(output, "shape", None))

Template: torch.Size([1, 3, 96, 96])
Search: torch.Size([1, 3, 224, 224])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([1, 1, 14, 14])


In [134]:
!pip install lmdb


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [142]:
import sys

# Add MixSort root so Python can find its local yolox package
if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

print("MixSort root:", MIXSORT_DIR)
print("In sys.path:", str(MIXSORT_DIR) in sys.path)

from yolox.mixsort_tracker.mixsort_tracker import MIXTracker

print("MIXTracker imported successfully")

MixSort root: /mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort
In sys.path: True
MIXTracker imported successfully


In [143]:
tracker_file = MIXSORT_DIR / "yolox/mixsort_tracker/mixsort_tracker.py"
lines = tracker_file.read_text(encoding="utf-8", errors="ignore").splitlines()

for i, line in enumerate(lines, start=1):
    if "args." in line:
        print(f"{i}: {line.strip()}")

183: # self.det_thresh = args.track_thresh
184: self.det_thresh = args.track_thresh + 0.1
185: self.buffer_size = int(frame_rate / 30.0 * args.track_buffer)
190: self.alpha = args.alpha
191: self.radius = args.radius
192: self.iou_thresh = args.iou_thresh
197: self.settings.script_name = args.script
198: self.settings.config_name = args.config
203: prj_dir, f"experiments/{args.script}/{args.config}.yaml"
214: self.network = network.cuda(torch.device(f"cuda:{args.local_rank}"))
225: # self.det_thresh = args.track_thresh
226: self.det_thresh = args.track_thresh + 0.1
227: self.buffer_size = int(frame_rate / 30.0 * args.track_buffer)
232: self.alpha = args.alpha
233: self.radius = args.radius
234: self.iou_thresh = args.iou_thresh
377: # if iou.min()<self.args.fuse_iou_thresh:
384: #         if iou[i][j]>self.args.fuse_iou_thresh and vit[i][j]<self.args.fuse_vit_thresh:
416: remain_inds = scores > self.args.track_thresh
418: inds_high = scores < self.args.track_thresh
449: # if not self.a

In [144]:
from types import SimpleNamespace

args = SimpleNamespace(
    track_thresh=0.6,
    track_buffer=30,
    match_thresh=0.9,
    iou_thresh=0.3,
    alpha=0.6,
    radius=0,
    script="mixformer_deit",
    config="track",
    local_rank=0,
    mot20=False,
)

print(args)

namespace(track_thresh=0.6, track_buffer=30, match_thresh=0.9, iou_thresh=0.3, alpha=0.6, radius=0, script='mixformer_deit', config='track', local_rank=0, mot20=False)


In [145]:
track_cfg = MIXSORT_DIR / "MixViT/experiments/mixformer_deit/track.yaml"

text = track_cfg.read_text(encoding="utf-8")

for i, line in enumerate(text.splitlines(), start=1):
    if "PRETRAINED_PATH" in line:
        print(f"{i}: {line}")

38:     PRETRAINED_PATH: '/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar'


In [146]:
track_cfg = MIXSORT_DIR / "MixViT/experiments/mixformer_deit/track.yaml"

old_path = "pretrained/MixFormer_sports_train.pth.tar"
new_path = str(MIXSORT_DIR / "pretrained/MixFormer_sports_train.pth.tar")

text = track_cfg.read_text(encoding="utf-8")

assert old_path in text, "Expected checkpoint path was not found."

text = text.replace(old_path, new_path)

track_cfg.write_text(text, encoding="utf-8")

print("Updated checkpoint path to:")
print(new_path)

Updated checkpoint path to:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar


In [148]:
track_cfg = MIXSORT_DIR / "MixViT/experiments/mixformer_deit/track.yaml"

text = track_cfg.read_text(encoding="utf-8")

for line in text.splitlines():
    if "PRETRAINED_PATH" in line:
        print("Config line:")
        print(line)

        path = line.split(":", 1)[1].strip().strip("'\"")
        checkpoint_path = Path(path)

        print("\nParsed path:")
        print(checkpoint_path)

        print("\nExists exactly as written:", checkpoint_path.exists())

        if not checkpoint_path.is_absolute():
            checkpoint_path = track_cfg.parent / checkpoint_path
            print("Resolved relative to config:", checkpoint_path)
            print("Exists:", checkpoint_path.exists())

Config line:
    PRETRAINED_PATH: '/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort//mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar'

Parsed path:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar

Exists exactly as written: False


In [149]:
from pathlib import Path

track_cfg = MIXSORT_DIR / "MixViT/experiments/mixformer_deit/track.yaml"

correct_checkpoint = MIXSORT_DIR / "pretrained/MixFormer_sports_train.pth.tar"

text = track_cfg.read_text(encoding="utf-8")

lines = text.splitlines()

for i, line in enumerate(lines):
    if "PRETRAINED_PATH:" in line:
        lines[i] = f"    PRETRAINED_PATH: '{correct_checkpoint}'"
        break
else:
    raise RuntimeError("PRETRAINED_PATH not found")

track_cfg.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Fixed path:")
print(correct_checkpoint)
print("Exists:", correct_checkpoint.exists())

Fixed path:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar
Exists: True


In [150]:
tracker = MIXTracker(args, frame_rate=25)

print("MIXTracker initialized successfully.")
print("Tracker:", type(tracker).__name__)

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MIXTracker initialized successfully.
Tracker: MIXTracker


In [152]:
from pathlib import Path

SEQ_DIR = (
    MIXSORT_DIR.parent.parent.parent
    / "datasets/football/benchmarks/sportsmot/extracted/val"
    / "v_2QhNRucNC7E_c017"
)

print("Sequence directory:")
print(SEQ_DIR)

print("\nExists:", SEQ_DIR.exists())

print("\nFirst 30 entries:")
for x in sorted(SEQ_DIR.iterdir())[:30]:
    print(x.name)

Sequence directory:
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017

Exists: True

First 30 entries:
gt
img1
seqinfo.ini


In [154]:
import cv2
import numpy as np
import torch
import inspect
from pathlib import Path

In [155]:

IMG_DIR = SEQ_DIR / "img1"

frames = sorted(IMG_DIR.glob("*.jpg"))

print("Image directory:", IMG_DIR)
print("Frames:", len(frames))
print("First 5:", [f.name for f in frames[:5]])

frame = cv2.imread(str(frames[0]))
print("Frame shape:", frame.shape)

Image directory: /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017/img1
Frames: 450
First 5: ['000001.jpg', '000002.jpg', '000003.jpg', '000004.jpg', '000005.jpg']
Frame shape: (720, 1280, 3)


In [156]:
from ultralytics import YOLO
import numpy as np

detector = YOLO(
    str(MIXSORT_DIR.parent.parent / "models/football_player_detector.pt")
)

frame = cv2.imread(str(frames[0]))

result = detector.predict(
    frame,
    conf=0.40,
    iou=0.40,
    device=0,
    verbose=False
)[0]

boxes = result.boxes.xyxy.cpu().numpy()
scores = result.boxes.conf.cpu().numpy()

output_results = np.column_stack([boxes, scores]).astype(np.float32)

print("Detections:", len(output_results))
print("Shape:", output_results.shape)
print("First detection:", output_results[0] if len(output_results) else None)

Detections: 22
Shape: (22, 5)
First detection: [     776.61      385.22      816.67      467.82     0.95881]


In [157]:
if not hasattr(np, "int"):
    np.int = int

print("NumPy compatibility aliases:")
print("np.float →", np.float)
print("np.int   →", np.int)

NumPy compatibility aliases:
np.float → <class 'float'>
np.int   → <class 'int'>


In [158]:
img_info = (frame.shape[0], frame.shape[1])
img_size = (frame.shape[1], frame.shape[0])

online_targets = tracker.update(
    output_results,
    img_info,
    img_size,
    frame
)

print("Active MixSort targets:", len(online_targets))

for t in online_targets[:10]:
    print(
        "track_id:", t.track_id,
        "tlwh:", np.round(t.tlwh, 2),
        "score:", round(float(t.score), 4)
    )

Active MixSort targets: 20
track_id: 1 tlwh: [     1380.6      684.83       71.22      146.85] score: 0.9588
track_id: 2 tlwh: [     1695.3      427.33       69.35      118.76] score: 0.9553
track_id: 3 tlwh: [     1908.5      538.98       48.77      129.48] score: 0.9519
track_id: 4 tlwh: [     905.64      816.65        62.6      161.12] score: 0.9503
track_id: 5 tlwh: [     898.48      673.43       54.77      153.37] score: 0.9436
track_id: 6 tlwh: [     1552.2      708.03       60.64      147.27] score: 0.9405
track_id: 7 tlwh: [     1649.9      369.04       50.48      104.03] score: 0.9369
track_id: 8 tlwh: [     1559.4      370.61       44.36      112.36] score: 0.9345
track_id: 9 tlwh: [      350.5       555.9       90.34      117.27] score: 0.9345
track_id: 10 tlwh: [      13.18      834.85       69.99      186.93] score: 0.9342


In [159]:
print("Detection scores:")
print(np.round(scores, 4))

print("\nScore statistics:")
print("min :", scores.min())
print("max :", scores.max())
print("mean:", scores.mean())
print(">0.7:", np.sum(scores > 0.7))
print(">0.6:", np.sum(scores > 0.6))

Detection scores:
[     0.9588      0.9553      0.9519      0.9503      0.9436      0.9405      0.9369      0.9345      0.9345      0.9342      0.9304      0.9299      0.9269      0.9266      0.9203      0.9081      0.9073       0.785      0.7691      0.7383      0.6311      0.4427]

Score statistics:
min : 0.44268554
max : 0.9588117
mean: 0.8752914
>0.7: 20
>0.6: 21


In [160]:
tracker_file = MIXSORT_DIR / "yolox/mixsort_tracker/mixsort_tracker.py"
lines = tracker_file.read_text(encoding="utf-8", errors="ignore").splitlines()

for i, line in enumerate(lines, start=1):
    if "def activate" in line or "track.activate" in line:
        print(f"{i}: {line.strip()}")

65: def activate(self, kalman_filter, frame_id, template):
540: track.activate(


In [161]:
for i in range(55, 90):
    print(f"{i}: {lines[i-1]}")

55:             for i, st in enumerate(stracks):
56:                 if st.state != TrackState.Tracked:
57:                     multi_mean[i][7] = 0
58:             multi_mean, multi_covariance = STrack.shared_kalman.multi_predict(
59:                 multi_mean, multi_covariance
60:             )
61:             for i, (mean, cov) in enumerate(zip(multi_mean, multi_covariance)):
62:                 stracks[i].mean = mean
63:                 stracks[i].covariance = cov
64: 
65:     def activate(self, kalman_filter, frame_id, template):
66:         """Start a new tracklet"""
67:         self.kalman_filter = kalman_filter
68:         self.track_id = self.next_id()
69:         self.mean, self.covariance = self.kalman_filter.initiate(
70:             self.tlwh_to_xyah(self._tlwh)
71:         )
72:         self.template = template
73: 
74:         self.tracklet_len = 0
75:         self.state = TrackState.Tracked
76:         if frame_id == 1:
77:             self.is_activated = True
78:     

In [162]:
for i in range(520, 570):
    print(f"{i}: {lines[i-1]}")

520:         for itracked, idet in matches:
521:             det = detections[idet]
522:             template = (
523:                 self.crop_and_resize(img, det.tlwh, "template")
524:                 if det._iou < self.iou_thresh
525:                 else None
526:             )
527:             unconfirmed[itracked].update(detections[idet], self.frame_id, template)
528:             activated_starcks.append(unconfirmed[itracked])
529:         for it in u_unconfirmed:
530:             track = unconfirmed[it]
531:             track.mark_removed()
532:             removed_stracks.append(track)
533: 
534:         """ Step 4: Init new stracks"""
535:         for inew in u_detection:
536:             track = detections[inew]
537:             if track.score < self.det_thresh:
538:                 continue
539:             # do not consider iou constraint
540:             track.activate(
541:                 self.kalman_filter,
542:                 self.frame_id,
543:                 self.

In [163]:
print("frame_id:", tracker.frame_id)
print("tracked_stracks:", len(tracker.tracked_stracks))
print("lost_stracks:", len(tracker.lost_stracks))
print("removed_stracks:", len(tracker.removed_stracks))

print("\nTrack states:")
for t in tracker.tracked_stracks[:10]:
    print(
        "ID:", t.track_id,
        "activated:", t.is_activated,
        "state:", t.state,
        "score:", round(float(t.score), 4)
    )

frame_id: 1
tracked_stracks: 20
lost_stracks: 0
removed_stracks: 0

Track states:
ID: 1 activated: True state: 1 score: 0.9588
ID: 2 activated: True state: 1 score: 0.9553
ID: 3 activated: True state: 1 score: 0.9519
ID: 4 activated: True state: 1 score: 0.9503
ID: 5 activated: True state: 1 score: 0.9436
ID: 6 activated: True state: 1 score: 0.9405
ID: 7 activated: True state: 1 score: 0.9369
ID: 8 activated: True state: 1 score: 0.9345
ID: 9 activated: True state: 1 score: 0.9345
ID: 10 activated: True state: 1 score: 0.9342


In [164]:
for i in range(389, 430):
    print(f"{i}: {lines[i-1]}")

389:         self.frame_id += 1
390:         activated_starcks = []
391:         refind_stracks = []
392:         lost_stracks = []
393:         removed_stracks = []
394: 
395:         if output_results.shape[1] == 5:
396:             scores = output_results[:, 4]
397:             bboxes = output_results[:, :4]
398:         else:
399:             output_results = output_results.cpu().numpy()
400:             scores = output_results[:, 4] * output_results[:, 5]
401:             bboxes = output_results[:, :4]  # x1y1x2y2
402:         img_h, img_w = img_info[0], img_info[1]
403:         scale = min(img_size[0] / float(img_h), img_size[1] / float(img_w))
404:         bboxes /= scale
405: 
406:         # compute all ious
407:         all_dets = [x for x in bboxes]
408:         iou = matching.ious(all_dets, all_dets)
409:         # compute max iou for every det
410:         max_iou = []
411:         for i in range(len(all_dets)):
412:             iou[i][i] = 0
413:             max_iou.append

In [165]:
tracker = MIXTracker(args, frame_rate=25)

print("Fresh tracker frame_id:", tracker.frame_id)

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
Fresh tracker frame_id: 0


In [166]:
online_targets = tracker.update(
    output_results,
    img_info,
    img_size,
    frame
)

print("frame_id:", tracker.frame_id)
print("Active MixSort targets:", len(online_targets))

for t in online_targets[:10]:
    print(
        "ID:", t.track_id,
        "score:", round(float(t.score), 4),
        "tlwh:", np.round(t.tlwh, 2)
    )

frame_id: 1
Active MixSort targets: 20
ID: 21 score: 0.9588 tlwh: [     2454.5      1217.5      126.61      261.07]
ID: 22 score: 0.9553 tlwh: [     3013.9       759.7      123.29      211.13]
ID: 23 score: 0.9519 tlwh: [     3392.9      958.19       86.71      230.19]
ID: 24 score: 0.9503 tlwh: [       1610      1451.8      111.29      286.44]
ID: 25 score: 0.9436 tlwh: [     1597.3      1197.2       97.37      272.65]
ID: 26 score: 0.9405 tlwh: [     2759.4      1258.7       107.8      261.81]
ID: 27 score: 0.9369 tlwh: [     2933.1      656.07       89.75      184.95]
ID: 28 score: 0.9345 tlwh: [     2772.3      658.86       78.87      199.76]
ID: 29 score: 0.9345 tlwh: [     623.11      988.27      160.61      208.48]
ID: 30 score: 0.9342 tlwh: [      23.43      1484.2      124.43      332.32]


In [167]:
# Correct dimensions: (height, width)
img_info = (frame.shape[0], frame.shape[1])
img_size = (frame.shape[0], frame.shape[1])

tracker = MIXTracker(args, frame_rate=25)

online_targets = tracker.update(
    output_results,
    img_info,
    img_size,
    frame
)

print("Active MixSort targets:", len(online_targets))

for t in online_targets[:10]:
    print(
        "ID:", t.track_id,
        "score:", round(float(t.score), 4),
        "tlwh:", np.round(t.tlwh, 2)
    )

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
Active MixSort targets: 20
ID: 41 score: 0.9588 tlwh: [     2454.5      1217.5      126.61      261.07]
ID: 42 score: 0.9553 tlwh: [     3013.9       759.7      123.29      211.13]
ID: 43 score: 0.9519 tlwh: [     3392.9      958.19       86.71      230.19]
ID: 44 score: 0.9503 tlwh: [       1610      1451.8      111.29      286.44]
ID: 45 score: 0.9436 tlwh: [     1597.3      1197.2       97.37      272.65]
ID: 46 score: 0.9405 tlwh: [     2759.4      1258.7       107.8      261.81]
ID: 47 score: 0.9369 tlwh: [     2933.1      656.07       89.75      184.95]
ID: 48 score: 0.9345 tlwh: [     2772.3      658.86       78.87      199.76]
ID: 49 score: 0.9345 tlwh: [     623.11      988.27      160.61      208.48]
ID: 50 score: 0.9342 tlwh: [      23.43      1484.2      124.43      332.32]


In [168]:
# Re-run detector so output_results contains the ORIGINAL pixel coordinates
result = detector.predict(
    frame,
    conf=0.40,
    iou=0.40,
    device=0,
    verbose=False
)[0]

boxes = result.boxes.xyxy.cpu().numpy()
scores = result.boxes.conf.cpu().numpy()

output_results = np.column_stack([boxes, scores]).astype(np.float32)

# SportsMOT frame dimensions: H=720, W=1280
img_info = (frame.shape[0], frame.shape[1])
img_size = (frame.shape[0], frame.shape[1])

# Fresh tracker
tracker = MIXTracker(args, frame_rate=25)

online_targets = tracker.update(
    output_results.copy(),
    img_info,
    img_size,
    frame
)

print("Original detection:", output_results[0])
print("Active targets:", len(online_targets))

for t in online_targets[:10]:
    print(
        "ID:", t.track_id,
        "tlwh:", np.round(t.tlwh, 2),
        "score:", round(float(t.score), 4)
    )

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
Original detection: [     776.61      385.22      816.67      467.82     0.95881]
Active targets: 20
ID: 61 tlwh: [     776.61      385.22       40.06        82.6] score: 0.9588
ID: 62 tlwh: [     953.61      240.38       39.01        66.8] score: 0.9553
ID: 63 tlwh: [     1073.5      303.18       27.43       72.83] score: 0.9519
ID: 64 tlwh: [     509.42      459.37       35.21       90.63] score: 0.9503
ID: 65 tlwh: [     505.39       378.8       30.81       86.27] score: 0.9436
ID: 66 tlwh: [     873.09      398.27       34.11       82.84] score: 0.9405
ID: 67 tlwh: [     928.06      207.58        28.4       58.52] score: 0.9369
ID: 68 tlwh: [     877.17      208.47       24.95        63.2] score: 0.9345
ID: 69 tlwh: [     197.16       312.7       50.82       65.97] score: 0.9345
ID: 70 tlwh: [       7.41       469.6       39.37     

In [170]:
# Inspect what MixSort's crop_and_resize actually returns
test_track = tracker.tracked_stracks[0]

template_result = tracker.crop_and_resize(
    frame,
    test_track._tlwh,
    "template"
)

print("Return type:", type(template_result))

if isinstance(template_result, tuple):
    print("Tuple length:", len(template_result))
    for i, item in enumerate(template_result):
        print(
            f"  [{i}] type={type(item)}, "
            f"shape={getattr(item, 'shape', None)}"
        )
else:
    print("Shape:", getattr(template_result, "shape", None))

Return type: <class 'tuple'>
Tuple length: 2
  [0] type=<class 'torch.Tensor'>, shape=torch.Size([3, 96, 96])
  [1] type=<class 'list'>, shape=None


In [171]:
for i in range(290, 350):
    print(f"{i}: {lines[i-1]}")

290:     def compute_mix_dist(
291:         self,
292:         stracks: List[STrack],
293:         dets: List[STrack],
294:         img: torch.Tensor,
295:         fuse: bool = False,
296:     ) -> np.ndarray:
297:         """compute mix distance between stracks and dets.
298: 
299:         Args:
300:             stracks (List[STrack]): len = m
301:             dets (List[STrack]): len = n
302:             img (torch.Tensor): current image
303:             fuse (bool, optional): whether to fuse det score into iou. Defaults to False.
304: 
305:         Returns:
306:             np.ndarray: m x n
307:         """
308: 
309:         # Predict the current location with KF
310:         STrack.multi_predict(stracks)
311:         # compute iou dist
312:         iou = matching.iou_distance(stracks, dets)
313:         if fuse:
314:             iou = matching.fuse_score(iou, dets)
315: 
316:         if len(stracks) * len(dets) == 0:
317:             return iou
318: 
319:         # for every stra

In [172]:
tracker_file = MIXSORT_DIR / "yolox/mixsort_tracker/mixsort_tracker.py"

text = tracker_file.read_text(encoding="utf-8")

old = "        template_imgs = [s.template for s in stracks]"
new = """        template_imgs = [
            s.template[0] if isinstance(s.template, tuple) else s.template
            for s in stracks
        ]"""

if old not in text:
    print("Original line not found — patch may already be applied.")
else:
    text = text.replace(old, new, 1)
    tracker_file.write_text(text, encoding="utf-8")
    print("Patch applied successfully.")

Patch applied successfully.


In [173]:
import importlib
import yolox.mixsort_tracker.mixsort_tracker as mixsort_module

importlib.reload(mixsort_module)

MIXTracker = mixsort_module.MIXTracker

tracker = MIXTracker(args, frame_rate=25)

print("Tracker reloaded and initialized.")

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
Tracker reloaded and initialized.


In [174]:
import time
from collections import defaultdict

# Fresh tracker
tracker = MIXTracker(args, frame_rate=25)

track_history = defaultdict(list)

start = time.time()

for frame_idx, frame_path in enumerate(frames[:50], start=1):
    frame = cv2.imread(str(frame_path))

    result = detector.predict(
        frame,
        conf=0.40,
        iou=0.40,
        device=0,
        verbose=False
    )[0]

    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()

    detections = np.column_stack([boxes, scores]).astype(np.float32)

    img_info = (frame.shape[0], frame.shape[1])
    img_size = (frame.shape[0], frame.shape[1])

    targets = tracker.update(
        detections.copy(),
        img_info,
        img_size,
        frame
    )

    for t in targets:
        track_history[t.track_id].append(frame_idx)

elapsed = time.time() - start

print("Frames processed:", 50)
print("Unique track IDs:", len(track_history))
print("Active IDs at final frame:", len(targets))
print("Total time:", round(elapsed, 2), "sec")
print("FPS:", round(50 / elapsed, 2))

print("\nFirst 15 track histories:")
for track_id, history in list(track_history.items())[:15]:
    print(track_id, "→", len(history), "frames")

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
Frames processed: 50
Unique track IDs: 21
Active IDs at final frame: 19
Total time: 10.67 sec
FPS: 4.69

First 15 track histories:
101 → 50 frames
102 → 48 frames
103 → 50 frames
104 → 50 frames
105 → 49 frames
106 → 50 frames
107 → 50 frames
108 → 50 frames
109 → 49 frames
110 → 50 frames
111 → 45 frames
112 → 50 frames
113 → 50 frames
114 → 50 frames
115 → 2 frames


In [177]:
# Inspect the exact structure of the 5 evaluation sequences

for seq_name in SEQUENCES:
    seq_dir = VAL_DIR / seq_name

    print("\n" + "=" * 70)
    print(seq_name)
    print("=" * 70)

    if not seq_dir.exists():
        print("SEQUENCE DIRECTORY NOT FOUND")
        continue

    print("Sequence:", seq_dir)

    # Show immediate directories/files
    for p in sorted(seq_dir.iterdir()):
        if p.is_dir():
            print(f"[DIR]  {p.name}")
        else:
            print(f"[FILE] {p.name}")

    # Search for possible GT files
    gt_candidates = list(seq_dir.rglob("gt.txt"))

    print("\nGT candidates:")
    if gt_candidates:
        for p in gt_candidates:
            print(" ", p)
    else:
        print("  NONE")

    # Search for sequence metadata
    seqinfo_candidates = list(seq_dir.rglob("seqinfo.ini"))

    print("\nseqinfo.ini:")
    if seqinfo_candidates:
        for p in seqinfo_candidates:
            print(" ", p)
    else:
        print("  NONE")


v_2QhNRucNC7E_c017
Sequence: /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017
[DIR]  gt
[DIR]  img1
[FILE] seqinfo.ini

GT candidates:
  /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017/gt/gt.txt

seqinfo.ini:
  /mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_2QhNRucNC7E_c017/seqinfo.ini

v_2QhNRucNC7E_c600
SEQUENCE DIRECTORY NOT FOUND

v_2QhNRucNC7E_c011
SEQUENCE DIRECTORY NOT FOUND

v_2QhNRucNC7E_c067
SEQUENCE DIRECTORY NOT FOUND

v_2QhNRucNC7E_c010
SEQUENCE DIRECTORY NOT FOUND


In [178]:
# Find the actual locations of the 4 missing SportsMOT sequences

SEARCH_ROOT = ROOT / "datasets/football/benchmarks/sportsmot"

missing_sequences = [
    "v_2QhNRucNC7E_c600",
    "v_2QhNRucNC7E_c011",
    "v_2QhNRucNC7E_c067",
    "v_2QhNRucNC7E_c010",
]

print("=" * 70)
print("SEARCHING SPORTS MOT DATASET")
print("=" * 70)

for seq_name in missing_sequences:

    matches = list(SEARCH_ROOT.rglob(seq_name))

    print(f"\n{seq_name}")

    if matches:
        for p in matches:
            print("FOUND:", p)
    else:
        print("NOT FOUND")

SEARCHING SPORTS MOT DATASET

v_2QhNRucNC7E_c600
NOT FOUND

v_2QhNRucNC7E_c011
NOT FOUND

v_2QhNRucNC7E_c067
NOT FOUND

v_2QhNRucNC7E_c010
NOT FOUND


In [179]:
# Correct SportsMOT evaluation sequences

SEQUENCES = [
    "v_2QhNRucNC7E_c017",
    "v_G-vNjfx1GGc_c600",
    "v_ITo3sCnpw_k_c011",
    "v_dw7LOz17Omg_c067",
    "v_i2_I4qquVg0_c010",
]

print("=" * 70)
print("CHECKING MIXSORT EVALUATION SEQUENCES")
print("=" * 70)

for i, seq_name in enumerate(SEQUENCES, 1):

    seq_dir = VAL_DIR / seq_name
    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    print(f"\n{i}. {seq_name}")
    print("   Directory:", seq_dir.exists())
    print("   Images:   ", img_dir.exists())
    print("   GT:       ", gt_path.exists())

    if img_dir.exists():
        print("   Frames:   ", len(list(img_dir.glob("*.jpg"))))

print("\n" + "=" * 70)
print("READY FOR MIXSORT EVALUATION")
print("=" * 70)

CHECKING MIXSORT EVALUATION SEQUENCES

1. v_2QhNRucNC7E_c017
   Directory: True
   Images:    True
   GT:        True
   Frames:    450

2. v_G-vNjfx1GGc_c600
   Directory: True
   Images:    True
   GT:        True
   Frames:    500

3. v_ITo3sCnpw_k_c011
   Directory: True
   Images:    True
   GT:        True
   Frames:    776

4. v_dw7LOz17Omg_c067
   Directory: True
   Images:    True
   GT:        True
   Frames:    450

5. v_i2_I4qquVg0_c010
   Directory: False
   Images:    False
   GT:        False

READY FOR MIXSORT EVALUATION


In [180]:
# Find the exact location of the SportsMOT c010 sequence

SEARCH_ROOT = ROOT / "datasets/football/benchmarks/sportsmot"

matches = []

for p in SEARCH_ROOT.rglob("*c010"):
    if p.is_dir():
        matches.append(p)

print("=" * 70)
print("SEARCH RESULTS FOR c010")
print("=" * 70)

if matches:
    for p in matches:
        print(p)
else:
    print("No c010 directory found.")

SEARCH RESULTS FOR c010
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/train/v_HdiyOtliFiw_c010
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_i2_L4qquVg0_c010
/mnt/d/My Projects/AthletiQ/AthletiQ/datasets/football/benchmarks/sportsmot/extracted/val/v_ITo3sCnpw_k_c010


In [181]:
# ============================================================
# ATHLETIQ — MixSort 5-Sequence MOT Evaluation
# Same detector + SportsMOT GT + IoU=0.50
# ============================================================

import sys
import time
import cv2
import numpy as np
import pandas as pd
import motmetrics as mm

from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

from ultralytics import YOLO


# ============================================================
# 1. NUMPY 2.x COMPATIBILITY
# ============================================================

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)


# ============================================================
# 2. PROJECT PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

MIXSORT_DIR = (
    ROOT
    / "ai-engine/third_party/MixSort"
)

VAL_DIR = (
    ROOT
    / "datasets/football/benchmarks/sportsmot/extracted/val"
)

MODEL_PATH = (
    ROOT
    / "ai-engine/models/football_player_detector.pt"
)


# ============================================================
# 3. EXACT SPORTS MOT VALIDATION SEQUENCES
# ============================================================

SEQUENCES = [
    "v_2QhNRucNC7E_c017",
    "v_G-vNjfx1GGc_c600",
    "v_ITo3sCnpw_k_c011",
    "v_dw7LOz17Omg_c067",
    "v_i2_L4qquVg0_c010",
]


# ============================================================
# 4. CHECK SEQUENCES BEFORE STARTING
# ============================================================

print("=" * 70)
print("MIXSORT 5-SEQUENCE MOT EVALUATION")
print("=" * 70)

for seq_name in SEQUENCES:

    seq_dir = VAL_DIR / seq_name
    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    print(f"\n{seq_name}")
    print("  Directory:", seq_dir.exists())
    print("  Images:   ", img_dir.exists())
    print("  GT:       ", gt_path.exists())

    if not seq_dir.exists():
        raise FileNotFoundError(
            f"Sequence directory not found:\n{seq_dir}"
        )

    if not img_dir.exists():
        raise FileNotFoundError(
            f"img1 directory not found:\n{img_dir}"
        )

    if not gt_path.exists():
        raise FileNotFoundError(
            f"GT file not found:\n{gt_path}"
        )

print("\nAll 5 sequences verified.")


# ============================================================
# 5. IMPORT MIXSORT
# ============================================================

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

from yolox.mixsort_tracker.mixsort_tracker import MIXTracker


# ============================================================
# 6. MIXSORT CONFIGURATION
# ============================================================

args = SimpleNamespace(
    track_thresh=0.6,
    track_buffer=30,
    match_thresh=0.9,
    iou_thresh=0.3,
    alpha=0.6,
    radius=0,
    script="mixformer_deit",
    config="track",
    local_rank=0,
    mot20=False,
)


# ============================================================
# 7. LOAD FOOTBALL PLAYER DETECTOR
# ============================================================

detector = YOLO(str(MODEL_PATH))

print("\nDetector loaded:")
print(MODEL_PATH)

print("\nEvaluation settings:")
print("  Detection confidence :", 0.40)
print("  Detection IoU        :", 0.40)
print("  MOT matching IoU     :", 0.50)
print("  Tracker              : MixSort")
print("  Tracker threshold    :", args.track_thresh)
print("  Match threshold      :", args.match_thresh)


# ============================================================
# 8. SPORTS MOT GT LOADER
# ============================================================

def load_gt(gt_path):

    gt = defaultdict(list)

    with open(gt_path, "r") as f:

        for line in f:

            parts = line.strip().split(",")

            if len(parts) < 6:
                continue

            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))

            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])

            gt[frame_id].append(
                (track_id, x, y, w, h)
            )

    return gt


# ============================================================
# 9. EVALUATE ALL FIVE SEQUENCES
# ============================================================

all_results = []

total_frames = 0
total_objects = 0
total_predictions = 0
total_matches = 0
total_switches = 0
total_fp = 0
total_fn = 0

for seq_number, seq_name in enumerate(SEQUENCES, start=1):

    print("\n")
    print("=" * 70)
    print(f"SEQUENCE {seq_number}/5: {seq_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    seq_dir = VAL_DIR / seq_name
    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    frames = sorted(img_dir.glob("*.jpg"))

    gt = load_gt(gt_path)

    gt_count = sum(
        len(v) for v in gt.values()
    )

    print("Frames:", len(frames))
    print("GT rows:", gt_count)

    # --------------------------------------------------------
    # Fresh MixSort tracker
    # --------------------------------------------------------

    tracker = MIXTracker(
        args,
        frame_rate=25
    )

    # MOT accumulator for this sequence only
    acc = mm.MOTAccumulator(
        auto_id=True
    )

    unique_ids = set()

    sequence_predictions = 0

    start_time = time.time()

    # --------------------------------------------------------
    # FRAME LOOP
    # --------------------------------------------------------

    for frame_idx, frame_path in enumerate(
        frames,
        start=1
    ):

        frame = cv2.imread(
            str(frame_path)
        )

        if frame is None:
            raise RuntimeError(
                f"Could not read frame:\n{frame_path}"
            )

        # ----------------------------------------------------
        # PLAYER DETECTION
        # ----------------------------------------------------

        result = detector.predict(
            frame,
            conf=0.40,
            iou=0.40,
            device=0,
            verbose=False
        )[0]

        boxes = (
            result
            .boxes
            .xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result
            .boxes
            .conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [
                boxes,
                scores
            ]
        ).astype(
            np.float32
        )

        # ----------------------------------------------------
        # MIXSORT INPUT
        # Important: H,W — not W,H
        # ----------------------------------------------------

        img_info = (
            frame.shape[0],
            frame.shape[1]
        )

        img_size = (
            frame.shape[0],
            frame.shape[1]
        )

        # MixSort modifies detection array internally,
        # therefore pass a copy.
        targets = tracker.update(
            detections.copy(),
            img_info,
            img_size,
            frame
        )

        # ----------------------------------------------------
        # MIXSORT PREDICTIONS
        # ----------------------------------------------------

        pred_ids = []
        pred_boxes = []

        for target in targets:

            track_id = int(
                target.track_id
            )

            x, y, w, h = map(
                float,
                target.tlwh
            )

            pred_ids.append(
                track_id
            )

            pred_boxes.append(
                [
                    x,
                    y,
                    w,
                    h
                ]
            )

            unique_ids.add(
                track_id
            )

        sequence_predictions += len(
            pred_ids
        )

        # ----------------------------------------------------
        # GROUND TRUTH
        # ----------------------------------------------------

        gt_entries = gt.get(
            frame_idx,
            []
        )

        gt_ids = []
        gt_boxes = []

        for (
            track_id,
            x,
            y,
            w,
            h
        ) in gt_entries:

            gt_ids.append(
                int(track_id)
            )

            gt_boxes.append(
                [
                    x,
                    y,
                    w,
                    h
                ]
            )

        # ----------------------------------------------------
        # IoU DISTANCE MATRIX
        #
        # max_iou=0.5 means:
        # IoU >= 0.5 -> valid match
        # ----------------------------------------------------

        if (
            len(gt_boxes) == 0
            and len(pred_boxes) == 0
        ):

            distance = np.empty(
                (0, 0)
            )

        elif len(gt_boxes) == 0:

            distance = np.empty(
                (
                    0,
                    len(pred_boxes)
                )
            )

        elif len(pred_boxes) == 0:

            distance = np.empty(
                (
                    len(gt_boxes),
                    0
                )
            )

        else:

            distance = mm.distances.iou_matrix(
                np.asarray(
                    gt_boxes,
                    dtype=float
                ),
                np.asarray(
                    pred_boxes,
                    dtype=float
                ),
                max_iou=0.5
            )

        # ----------------------------------------------------
        # UPDATE MOTMETRICS
        # ----------------------------------------------------

        acc.update(
            gt_ids,
            pred_ids,
            distance
        )

    # ========================================================
    # SEQUENCE TIMING
    # ========================================================

    elapsed = time.time() - start_time

    fps = (
        len(frames) / elapsed
        if elapsed > 0
        else 0
    )

    # ========================================================
    # COMPUTE METRICS
    # ========================================================

    mh = mm.metrics.create()

    summary = mh.compute(
        acc,
        metrics=[
            "num_frames",
            "num_objects",
            "num_predictions",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "idf1"
        ],
        name=seq_name
    )

    row = summary.iloc[0]

    mota = float(
        row["mota"]
    )

    idf1 = float(
        row["idf1"]
    )

    switches = int(
        row["num_switches"]
    )

    fp = int(
        row["num_false_positives"]
    )

    fn = int(
        row["num_misses"]
    )

    matches = int(
        row["num_matches"]
    )

    objects = int(
        row["num_objects"]
    )

    predictions = int(
        row["num_predictions"]
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    all_results.append(
        {
            "Sequence": seq_name,
            "Frames": len(frames),
            "GT Objects": objects,
            "Predictions": predictions,
            "Matches": matches,
            "ID Switches": switches,
            "FP": fp,
            "FN": fn,
            "MOTA": mota,
            "IDF1": idf1,
            "Unique IDs": len(unique_ids),
            "FPS": fps
        }
    )

    # --------------------------------------------------------
    # Accumulate totals
    # --------------------------------------------------------

    total_frames += len(frames)
    total_objects += objects
    total_predictions += predictions
    total_matches += matches
    total_switches += switches
    total_fp += fp
    total_fn += fn

    # ========================================================
    # PRINT SEQUENCE RESULT
    # ========================================================

    print("\nRESULTS")

    print(
        f"MOTA:          {mota:.6f}"
    )

    print(
        f"IDF1:          {idf1:.6f}"
    )

    print(
        f"ID Switches:   {switches}"
    )

    print(
        f"FP:            {fp}"
    )

    print(
        f"FN:            {fn}"
    )

    print(
        f"Matches:       {matches}"
    )

    print(
        f"Unique IDs:    {len(unique_ids)}"
    )

    print(
        f"FPS:           {fps:.2f}"
    )


# ============================================================
# 10. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    all_results
)

print("\n")
print("=" * 100)
print("MIXSORT — FINAL 5-SEQUENCE RESULTS")
print("=" * 100)

display(
    results_df
)


# ============================================================
# 11. ADDITIVE OVERALL MOTA
#
# MOTA = 1 - (FN + FP + IDSW) / GT
#
# This is valid to aggregate across independent sequences.
# ============================================================

overall_mota = (
    1
    - (
        total_fn
        + total_fp
        + total_switches
    )
    / total_objects
)


# ============================================================
# 12. MACRO-AVERAGE IDF1
#
# IDF1 is NOT safely obtained by simply concatenating
# independent sequences with overlapping track IDs.
# Therefore report the mean of the five sequence IDF1 values.
# ============================================================

macro_idf1 = results_df[
    "IDF1"
].mean()

macro_mota = results_df[
    "MOTA"
].mean()

average_fps = results_df[
    "FPS"
].mean()


# ============================================================
# 13. OVERALL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("MIXSORT OVERALL SUMMARY")
print("=" * 70)

print(
    f"Total frames:       {total_frames}"
)

print(
    f"Total GT objects:   {total_objects}"
)

print(
    f"Total predictions:  {total_predictions}"
)

print(
    f"Total matches:      {total_matches}"
)

print(
    f"Total ID switches:  {total_switches}"
)

print(
    f"Total FP:            {total_fp}"
)

print(
    f"Total FN:            {total_fn}"
)

print(
    f"Overall MOTA:       {overall_mota:.6f}"
)

print(
    f"Macro MOTA:         {macro_mota:.6f}"
)

print(
    f"Macro IDF1:         {macro_idf1:.6f}"
)

print(
    f"Average FPS:        {average_fps:.2f}"
)

print("\nEvaluation complete.")

MIXSORT 5-SEQUENCE MOT EVALUATION

v_2QhNRucNC7E_c017
  Directory: True
  Images:    True
  GT:        True

v_G-vNjfx1GGc_c600
  Directory: True
  Images:    True
  GT:        True

v_ITo3sCnpw_k_c011
  Directory: True
  Images:    True
  GT:        True

v_dw7LOz17Omg_c067
  Directory: True
  Images:    True
  GT:        True

v_i2_L4qquVg0_c010
  Directory: True
  Images:    True
  GT:        True

All 5 sequences verified.

Detector loaded:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/models/football_player_detector.pt

Evaluation settings:
  Detection confidence : 0.4
  Detection IoU        : 0.4
  MOT matching IoU     : 0.5
  Tracker              : MixSort
  Tracker threshold    : 0.6
  Match threshold      : 0.9


SEQUENCE 1/5: v_2QhNRucNC7E_c017
Frames: 450
GT rows: 6622
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.

RESULTS
MOTA:          0.843250
IDF1:          0.759869

,Sequence,Frames,GT Objects,Predictions,Matches,ID Switches,FP,FN,MOTA,IDF1,Unique IDs,FPS
0,v_2QhNRucNC7E_c017,450,6622,6525,6033,43,449,546,0.843250,0.759869,41,4.991787
1,v_G-vNjfx1GGc_c600,500,5462,4533,4187,40,306,1235,0.710546,0.566483,59,5.519453
2,v_ITo3sCnpw_k_c011,776,9736,9625,8952,60,613,724,0.856512,0.596457,55,5.214271
3,v_dw7LOz17Omg_c067,450,6959,6700,6435,31,234,493,0.891076,0.792445,55,4.854335
4,v_i2_L4qquVg0_c010,758,8262,7077,6908,55,114,1299,0.822319,0.480866,72,5.297228




MIXSORT OVERALL SUMMARY
Total frames:       2934
Total GT objects:   37041
Total predictions:  34460
Total matches:      32515
Total ID switches:  229
Total FP:            1716
Total FN:            4297
Overall MOTA:       0.831484
Macro MOTA:         0.824741
Macro IDF1:         0.639224
Average FPS:        5.18

Evaluation complete.


In [182]:
# ============================================================
# ATHLETIQ — MixSort Hyperparameter Tuning
# Phase 1: alpha × match_thresh
# Validation sequences: c600 + c010
# ============================================================

import sys
import time
import cv2
import numpy as np
import pandas as pd
import motmetrics as mm

from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

from ultralytics import YOLO


# ============================================================
# 1. NUMPY COMPATIBILITY
# ============================================================

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)


# ============================================================
# 2. PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

MIXSORT_DIR = (
    ROOT / "ai-engine/third_party/MixSort"
)

VAL_DIR = (
    ROOT
    / "datasets/football/benchmarks/sportsmot/extracted/val"
)

MODEL_PATH = (
    ROOT
    / "ai-engine/models/football_player_detector.pt"
)


# ============================================================
# 3. ONLY THE TWO DIFFICULT SEQUENCES
# ============================================================

SEQUENCES = {
    "c600": "v_G-vNjfx1GGc_c600",
    "c010": "v_i2_L4qquVg0_c010",
}


# ============================================================
# 4. TUNING GRID
# ============================================================

ALPHAS = [
    0.4,
    0.6,
    0.8
]

MATCH_THRESHES = [
    0.7,
    0.8,
    0.9
]

# Keep this fixed in Phase 1
IOU_THRESH = 0.3

TRACK_THRESH = 0.6
TRACK_BUFFER = 30


# ============================================================
# 5. IMPORT MIXSORT
# ============================================================

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

from yolox.mixsort_tracker.mixsort_tracker import MIXTracker


# ============================================================
# 6. LOAD DETECTOR
# ============================================================

detector = YOLO(
    str(MODEL_PATH)
)


# ============================================================
# 7. GT LOADER
# ============================================================

def load_gt(gt_path):

    gt = defaultdict(list)

    with open(gt_path, "r") as f:

        for line in f:

            parts = line.strip().split(",")

            if len(parts) < 6:
                continue

            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))

            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])

            gt[frame_id].append(
                (track_id, x, y, w, h)
            )

    return gt


# ============================================================
# 8. EVALUATION FUNCTION
# ============================================================

def evaluate_config(
    sequence_name,
    alpha,
    match_thresh
):

    seq_dir = VAL_DIR / sequence_name

    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    frames = sorted(
        img_dir.glob("*.jpg")
    )

    gt = load_gt(gt_path)

    # --------------------------------------------------------
    # Fresh tracker
    # --------------------------------------------------------

    args = SimpleNamespace(
        track_thresh=TRACK_THRESH,
        track_buffer=TRACK_BUFFER,
        match_thresh=match_thresh,
        iou_thresh=IOU_THRESH,
        alpha=alpha,
        radius=0,
        script="mixformer_deit",
        config="track",
        local_rank=0,
        mot20=False,
    )

    tracker = MIXTracker(
        args,
        frame_rate=25
    )

    acc = mm.MOTAccumulator(
        auto_id=True
    )

    unique_ids = set()

    start_time = time.time()

    # --------------------------------------------------------
    # Frame loop
    # --------------------------------------------------------

    for frame_idx, frame_path in enumerate(
        frames,
        start=1
    ):

        frame = cv2.imread(
            str(frame_path)
        )

        # ----------------------------------------------------
        # Detector
        # ----------------------------------------------------

        result = detector.predict(
            frame,
            conf=0.40,
            iou=0.40,
            device=0,
            verbose=False
        )[0]

        boxes = (
            result.boxes.xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result.boxes.conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [
                boxes,
                scores
            ]
        ).astype(
            np.float32
        )

        # ----------------------------------------------------
        # MixSort expects H,W
        # ----------------------------------------------------

        img_info = (
            frame.shape[0],
            frame.shape[1]
        )

        img_size = (
            frame.shape[0],
            frame.shape[1]
        )

        targets = tracker.update(
            detections.copy(),
            img_info,
            img_size,
            frame
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        pred_ids = []
        pred_boxes = []

        for target in targets:

            track_id = int(
                target.track_id
            )

            x, y, w, h = map(
                float,
                target.tlwh
            )

            pred_ids.append(
                track_id
            )

            pred_boxes.append(
                [x, y, w, h]
            )

            unique_ids.add(
                track_id
            )

        # ----------------------------------------------------
        # GT
        # ----------------------------------------------------

        gt_entries = gt.get(
            frame_idx,
            []
        )

        gt_ids = []
        gt_boxes = []

        for (
            track_id,
            x,
            y,
            w,
            h
        ) in gt_entries:

            gt_ids.append(
                int(track_id)
            )

            gt_boxes.append(
                [x, y, w, h]
            )

        # ----------------------------------------------------
        # IoU distance
        # ----------------------------------------------------

        if (
            len(gt_boxes) == 0
            and len(pred_boxes) == 0
        ):

            distance = np.empty(
                (0, 0)
            )

        elif len(gt_boxes) == 0:

            distance = np.empty(
                (
                    0,
                    len(pred_boxes)
                )
            )

        elif len(pred_boxes) == 0:

            distance = np.empty(
                (
                    len(gt_boxes),
                    0
                )
            )

        else:

            distance = mm.distances.iou_matrix(
                np.asarray(
                    gt_boxes,
                    dtype=float
                ),
                np.asarray(
                    pred_boxes,
                    dtype=float
                ),
                max_iou=0.5
            )

        acc.update(
            gt_ids,
            pred_ids,
            distance
        )

    # ========================================================
    # METRICS
    # ========================================================

    elapsed = time.time() - start_time

    fps = (
        len(frames) / elapsed
        if elapsed > 0
        else 0
    )

    mh = mm.metrics.create()

    summary = mh.compute(
        acc,
        metrics=[
            "num_objects",
            "num_predictions",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "idf1"
        ],
        name="result"
    )

    row = summary.iloc[0]

    return {
        "Sequence": sequence_name,
        "alpha": alpha,
        "match_thresh": match_thresh,
        "iou_thresh": IOU_THRESH,
        "MOTA": float(row["mota"]),
        "IDF1": float(row["idf1"]),
        "ID Switches": int(row["num_switches"]),
        "FP": int(row["num_false_positives"]),
        "FN": int(row["num_misses"]),
        "Matches": int(row["num_matches"]),
        "GT": int(row["num_objects"]),
        "Unique IDs": len(unique_ids),
        "FPS": fps,
    }


# ============================================================
# 9. RUN GRID
# ============================================================

results = []

total_runs = (
    len(SEQUENCES)
    * len(ALPHAS)
    * len(MATCH_THRESHES)
)

run_number = 0

print("=" * 80)
print("MIXSORT PHASE 1 HYPERPARAMETER TUNING")
print("=" * 80)

print(
    f"Configurations: {len(ALPHAS) * len(MATCH_THRESHES)}"
)

print(
    f"Sequences: {list(SEQUENCES.keys())}"
)

print(
    f"Total runs: {total_runs}"
)

print()


for seq_label, seq_name in SEQUENCES.items():

    print("\n" + "=" * 80)
    print(
        f"SEQUENCE: {seq_label} → {seq_name}"
    )
    print("=" * 80)

    for alpha in ALPHAS:

        for match_thresh in MATCH_THRESHES:

            run_number += 1

            print(
                f"\n[{run_number}/{total_runs}] "
                f"{seq_label} | "
                f"alpha={alpha} | "
                f"match_thresh={match_thresh}"
            )

            result = evaluate_config(
                seq_name,
                alpha,
                match_thresh
            )

            result["Sequence"] = seq_label

            results.append(
                result
            )

            print(
                f"MOTA={result['MOTA']:.4f} | "
                f"IDF1={result['IDF1']:.4f} | "
                f"IDSW={result['ID Switches']} | "
                f"FP={result['FP']} | "
                f"FN={result['FN']} | "
                f"FPS={result['FPS']:.2f}"
            )


# ============================================================
# 10. RESULTS TABLE
# ============================================================

tuning_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 100)
print("ALL TUNING RESULTS")
print("=" * 100)

display(
    tuning_df.sort_values(
        [
            "Sequence",
            "MOTA"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ============================================================
# 11. BEST CONFIG PER SEQUENCE
# ============================================================

print("\n")
print("=" * 100)
print("BEST CONFIGURATION BY MOTA")
print("=" * 100)

best_mota = (
    tuning_df
    .sort_values(
        "MOTA",
        ascending=False
    )
    .groupby(
        "Sequence",
        as_index=False
    )
    .first()
)

display(
    best_mota
)


# ============================================================
# 12. BEST CONFIG BY IDF1
# ============================================================

print("\n")
print("=" * 100)
print("BEST CONFIGURATION BY IDF1")
print("=" * 100)

best_idf1 = (
    tuning_df
    .sort_values(
        "IDF1",
        ascending=False
    )
    .groupby(
        "Sequence",
        as_index=False
    )
    .first()
)

display(
    best_idf1
)


# ============================================================
# 13. BEST CONFIG BY LOWEST ID SWITCHES
# ============================================================

print("\n")
print("=" * 100)
print("BEST CONFIGURATION BY ID SWITCHES")
print("=" * 100)

best_ids = (
    tuning_df
    .sort_values(
        [
            "ID Switches",
            "FN"
        ],
        ascending=[
            True,
            True
        ]
    )
    .groupby(
        "Sequence",
        as_index=False
    )
    .first()
)

display(
    best_ids
)


# ============================================================
# 14. COMBINED SCORE
#
# We don't want to optimize only MOTA or only IDF1.
#
# This score is ONLY for selecting candidates for Phase 2.
# It is NOT a standard MOT metric.
# ============================================================

candidate_df = (
    tuning_df
    .groupby(
        [
            "alpha",
            "match_thresh"
        ]
    )
    .agg(
        MOTA=("MOTA", "mean"),
        IDF1=("IDF1", "mean"),
        IDSW=("ID Switches", "sum"),
        FN=("FN", "sum"),
        FP=("FP", "sum"),
        FPS=("FPS", "mean")
    )
    .reset_index()
)

# Normalize MOTA and IDF1 between tested configurations
candidate_df["MOTA_norm"] = (
    (
        candidate_df["MOTA"]
        - candidate_df["MOTA"].min()
    )
    /
    (
        candidate_df["MOTA"].max()
        - candidate_df["MOTA"].min()
        + 1e-9
    )
)

candidate_df["IDF1_norm"] = (
    (
        candidate_df["IDF1"]
        - candidate_df["IDF1"].min()
    )
    /
    (
        candidate_df["IDF1"].max()
        - candidate_df["IDF1"].min()
        + 1e-9
    )
)

candidate_df["Score"] = (
    0.50 * candidate_df["MOTA_norm"]
    +
    0.50 * candidate_df["IDF1_norm"]
)

candidate_df = candidate_df.sort_values(
    "Score",
    ascending=False
)

print("\n")
print("=" * 100)
print("PHASE 2 CANDIDATES")
print("=" * 100)

display(
    candidate_df
)


print("\n")
print("=" * 100)
print("TUNING PHASE 1 COMPLETE")
print("=" * 100)

print(
    "Next: take the strongest configurations "
    "and test iou_thresh."
)

MIXSORT PHASE 1 HYPERPARAMETER TUNING
Configurations: 9
Sequences: ['c600', 'c010']
Total runs: 18


SEQUENCE: c600 → v_G-vNjfx1GGc_c600

[1/18] c600 | alpha=0.4 | match_thresh=0.7
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.2067 | IDF1=0.1201 | IDSW=403 | FP=66 | FN=3864 | FPS=2.90

[2/18] c600 | alpha=0.4 | match_thresh=0.8
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.5254 | IDF1=0.3780 | IDSW=66 | FP=147 | FN=2379 | FPS=5.52

[3/18] c600 | alpha=0.4 | match_thresh=0.9
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.6415 | IDF1=0.5425 | IDSW=30 | FP=191 | FN=1737 | FPS=5.44

[4/18] c600 | alpha=0.6 | match_thresh=0.7
head channel: 384
missing keys: [

[W1003 10:17:12.472113421 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 188743680 bytes (free: 0, total: 6441926656).


MOTA=0.2136 | IDF1=0.0860 | IDSW=746 | FP=8 | FN=5743 | FPS=3.03

[11/18] c010 | alpha=0.4 | match_thresh=0.8
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.5748 | IDF1=0.3578 | IDSW=83 | FP=19 | FN=3411 | FPS=5.19

[12/18] c010 | alpha=0.4 | match_thresh=0.9
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.6972 | IDF1=0.5028 | IDSW=37 | FP=43 | FN=2422 | FPS=5.83

[13/18] c010 | alpha=0.6 | match_thresh=0.7
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.7177 | IDF1=0.3472 | IDSW=234 | FP=57 | FN=2041 | FPS=4.95

[14/18] c010 | alpha=0.6 | match_thresh=0.8
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']


,Sequence,alpha,match_thresh,iou_thresh,MOTA,IDF1,ID Switches,FP,FN,Matches,GT,Unique IDs,FPS
17,c010,0.8,0.9,0.3,0.860809,0.512249,67,137,946,7249,8262,77,5.408747
16,c010,0.8,0.8,0.3,0.842653,0.491086,85,114,1101,7076,8262,103,5.741241
14,c010,0.6,0.9,0.3,0.822319,0.480866,55,114,1299,6908,8262,72,5.933971
15,c010,0.8,0.7,0.3,0.815783,0.456159,128,99,1295,6839,8262,151,5.489752
13,c010,0.6,0.8,0.3,0.780562,0.474963,77,95,1641,6544,8262,101,5.961474
12,c010,0.6,0.7,0.3,0.717744,0.347180,234,57,2041,5987,8262,244,4.954367
11,c010,0.4,0.9,0.3,0.697168,0.502793,37,43,2422,5803,8262,49,5.825693
10,c010,0.4,0.8,0.3,0.574800,0.357752,83,19,3411,4768,8262,100,5.193390
9,c010,0.4,0.7,0.3,0.213629,0.086014,746,8,5743,1773,8262,718,3.029420
8,c600,0.8,0.9,0.3,0.759795,0.580982,42,347,923,4497,5462,59,5.644904




BEST CONFIGURATION BY MOTA


...



BEST CONFIGURATION BY ID SWITCHES


,Sequence,alpha,match_thresh,iou_thresh,MOTA,IDF1,ID Switches,FP,FN,Matches,GT,Unique IDs,FPS
0,c010,0.4,0.9,0.3,0.697168,0.502793,37,43,2422,5803,8262,49,5.825693
1,c600,0.4,0.9,0.3,0.641523,0.542546,30,191,1737,3695,5462,51,5.444371




PHASE 2 CANDIDATES


,alpha,match_thresh,MOTA,IDF1,IDSW,FN,FP,FPS,MOTA_norm,IDF1_norm,Score
8,0.8,0.9,0.810302,0.546616,109,1869,484,5.526826,1.000000,1.000000,1.000000
7,0.8,0.8,0.793168,0.516751,137,2121,442,5.662347,0.971451,0.932668,0.952060
5,0.6,0.9,0.766432,0.523675,95,2534,420,5.706226,0.926901,0.948278,0.937590
2,0.4,0.9,0.669346,0.522669,67,4159,234,5.635032,0.765127,0.946012,0.855569
4,0.6,0.8,0.729900,0.466438,136,3049,380,5.656530,0.866028,0.819235,0.842631
6,0.8,0.7,0.765636,0.434173,224,2464,388,5.373720,0.925574,0.746491,0.836033
3,0.6,0.7,0.666635,0.349105,385,3740,307,4.880711,0.760610,0.554701,0.657655
1,0.4,0.8,0.550124,0.367900,149,5790,166,5.356362,0.566470,0.597077,0.581774
0,0.4,0.7,0.210165,0.103069,1149,9607,74,2.964346,0.000000,0.000000,0.000000




TUNING PHASE 1 COMPLETE
Next: take the strongest configurations and test iou_thresh.


In [183]:
# ============================================================
# ATHLETIQ — MixSort Phase 2 Tuning
# Sweep iou_thresh only
#
# Fixed:
#   alpha        = 0.8
#   match_thresh = 0.9
#
# Test:
#   iou_thresh = 0.2, 0.3, 0.4
#
# Sequences:
#   c600, c010
#
# Total runs = 6
# ============================================================

import sys
import gc
import time
import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
import torch

from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

from ultralytics import YOLO


# ============================================================
# 1. NUMPY 2.x COMPATIBILITY
# ============================================================

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)


# ============================================================
# 2. PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

MIXSORT_DIR = (
    ROOT / "ai-engine/third_party/MixSort"
)

VAL_DIR = (
    ROOT
    / "datasets/football/benchmarks/sportsmot/extracted/val"
)

MODEL_PATH = (
    ROOT
    / "ai-engine/models/football_player_detector.pt"
)


# ============================================================
# 3. SEQUENCES
# ============================================================

SEQUENCES = {
    "c600": "v_G-vNjfx1GGc_c600",
    "c010": "v_i2_L4qquVg0_c010",
}


# ============================================================
# 4. FIXED PARAMETERS
# ============================================================

ALPHA = 0.8
MATCH_THRESH = 0.9

IOU_THRESHES = [
    0.2,
    0.3,
    0.4
]

TRACK_THRESH = 0.6
TRACK_BUFFER = 30


# ============================================================
# 5. IMPORT MIXSORT
# ============================================================

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

from yolox.mixsort_tracker.mixsort_tracker import MIXTracker


# ============================================================
# 6. LOAD DETECTOR ONCE
# ============================================================

detector = YOLO(
    str(MODEL_PATH)
)

print("=" * 80)
print("MIXSORT PHASE 2 — IOU THRESHOLD TUNING")
print("=" * 80)

print("Fixed alpha:        ", ALPHA)
print("Fixed match_thresh: ", MATCH_THRESH)
print("IoU thresholds:     ", IOU_THRESHES)
print("Sequences:          ", list(SEQUENCES.keys()))
print("Total runs:         ", len(SEQUENCES) * len(IOU_THRESHES))

print()


# ============================================================
# 7. GT LOADER
# ============================================================

def load_gt(gt_path):

    gt = defaultdict(list)

    with open(gt_path, "r") as f:

        for line in f:

            parts = line.strip().split(",")

            if len(parts) < 6:
                continue

            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))

            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])

            gt[frame_id].append(
                (track_id, x, y, w, h)
            )

    return gt


# ============================================================
# 8. SINGLE CONFIG EVALUATION
# ============================================================

def evaluate_config(
    seq_name,
    iou_thresh
):

    seq_dir = VAL_DIR / seq_name

    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    frames = sorted(
        img_dir.glob("*.jpg")
    )

    gt = load_gt(gt_path)

    # --------------------------------------------------------
    # Create fresh MixSort configuration
    # --------------------------------------------------------

    args = SimpleNamespace(
        track_thresh=TRACK_THRESH,
        track_buffer=TRACK_BUFFER,
        match_thresh=MATCH_THRESH,
        iou_thresh=iou_thresh,
        alpha=ALPHA,
        radius=0,
        script="mixformer_deit",
        config="track",
        local_rank=0,
        mot20=False,
    )

    # --------------------------------------------------------
    # Fresh tracker
    # --------------------------------------------------------

    tracker = MIXTracker(
        args,
        frame_rate=25
    )

    acc = mm.MOTAccumulator(
        auto_id=True
    )

    unique_ids = set()

    start_time = time.time()

    # --------------------------------------------------------
    # Frame loop
    # --------------------------------------------------------

    for frame_idx, frame_path in enumerate(
        frames,
        start=1
    ):

        frame = cv2.imread(
            str(frame_path)
        )

        # ----------------------------------------------------
        # Detection
        # ----------------------------------------------------

        result = detector.predict(
            frame,
            conf=0.40,
            iou=0.40,
            device=0,
            verbose=False
        )[0]

        boxes = (
            result
            .boxes
            .xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result
            .boxes
            .conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [
                boxes,
                scores
            ]
        ).astype(
            np.float32
        )

        # ----------------------------------------------------
        # MixSort expects H,W
        # ----------------------------------------------------

        img_info = (
            frame.shape[0],
            frame.shape[1]
        )

        img_size = (
            frame.shape[0],
            frame.shape[1]
        )

        # Important: MixSort modifies detections
        targets = tracker.update(
            detections.copy(),
            img_info,
            img_size,
            frame
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        pred_ids = []
        pred_boxes = []

        for target in targets:

            track_id = int(
                target.track_id
            )

            x, y, w, h = map(
                float,
                target.tlwh
            )

            pred_ids.append(
                track_id
            )

            pred_boxes.append(
                [x, y, w, h]
            )

            unique_ids.add(
                track_id
            )

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        gt_entries = gt.get(
            frame_idx,
            []
        )

        gt_ids = []
        gt_boxes = []

        for (
            track_id,
            x,
            y,
            w,
            h
        ) in gt_entries:

            gt_ids.append(
                int(track_id)
            )

            gt_boxes.append(
                [x, y, w, h]
            )

        # ----------------------------------------------------
        # IoU distance matrix
        # ----------------------------------------------------

        if (
            len(gt_boxes) == 0
            and len(pred_boxes) == 0
        ):

            distance = np.empty(
                (0, 0)
            )

        elif len(gt_boxes) == 0:

            distance = np.empty(
                (0, len(pred_boxes))
            )

        elif len(pred_boxes) == 0:

            distance = np.empty(
                (len(gt_boxes), 0)
            )

        else:

            distance = mm.distances.iou_matrix(
                np.asarray(
                    gt_boxes,
                    dtype=float
                ),
                np.asarray(
                    pred_boxes,
                    dtype=float
                ),
                max_iou=0.5
            )

        # ----------------------------------------------------
        # Update MOT accumulator
        # ----------------------------------------------------

        acc.update(
            gt_ids,
            pred_ids,
            distance
        )

    # ========================================================
    # METRICS
    # ========================================================

    elapsed = time.time() - start_time

    fps = (
        len(frames) / elapsed
        if elapsed > 0
        else 0
    )

    mh = mm.metrics.create()

    summary = mh.compute(
        acc,
        metrics=[
            "num_objects",
            "num_predictions",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "idf1"
        ],
        name="result"
    )

    row = summary.iloc[0]

    result = {
        "Sequence": seq_name,
        "alpha": ALPHA,
        "match_thresh": MATCH_THRESH,
        "iou_thresh": iou_thresh,
        "MOTA": float(row["mota"]),
        "IDF1": float(row["idf1"]),
        "ID Switches": int(row["num_switches"]),
        "FP": int(row["num_false_positives"]),
        "FN": int(row["num_misses"]),
        "Matches": int(row["num_matches"]),
        "GT": int(row["num_objects"]),
        "Predictions": int(row["num_predictions"]),
        "Unique IDs": len(unique_ids),
        "FPS": fps,
    }

    # ========================================================
    # IMPORTANT — FREE GPU MEMORY
    # ========================================================

    del tracker
    del acc
    del gt

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    return result


# ============================================================
# 9. RUN 6 CONFIGURATIONS
# ============================================================

results = []

run_number = 0
total_runs = (
    len(SEQUENCES)
    * len(IOU_THRESHES)
)

for seq_label, seq_name in SEQUENCES.items():

    print("\n" + "=" * 80)
    print(
        f"SEQUENCE: {seq_label}"
    )
    print("=" * 80)

    for iou_thresh in IOU_THRESHES:

        run_number += 1

        print(
            f"\n[{run_number}/{total_runs}] "
            f"{seq_label} | "
            f"alpha={ALPHA} | "
            f"match_thresh={MATCH_THRESH} | "
            f"iou_thresh={iou_thresh}"
        )

        result = evaluate_config(
            seq_name,
            iou_thresh
        )

        # Replace long sequence name with label
        result["Sequence"] = seq_label

        results.append(
            result
        )

        print(
            f"MOTA={result['MOTA']:.4f} | "
            f"IDF1={result['IDF1']:.4f} | "
            f"IDSW={result['ID Switches']} | "
            f"FP={result['FP']} | "
            f"FN={result['FN']} | "
            f"FPS={result['FPS']:.2f}"
        )

        # ----------------------------------------------------
        # GPU status
        # ----------------------------------------------------

        if torch.cuda.is_available():

            allocated = (
                torch.cuda.memory_allocated()
                / 1024**3
            )

            reserved = (
                torch.cuda.memory_reserved()
                / 1024**3
            )

            print(
                f"GPU memory after cleanup: "
                f"allocated={allocated:.2f} GB, "
                f"reserved={reserved:.2f} GB"
            )


# ============================================================
# 10. RESULTS TABLE
# ============================================================

phase2_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 100)
print("PHASE 2 — ALL RESULTS")
print("=" * 100)

display(
    phase2_df.sort_values(
        [
            "iou_thresh",
            "Sequence"
        ]
    )
)


# ============================================================
# 11. AVERAGE ACROSS c600 + c010
# ============================================================

phase2_summary = (
    phase2_df
    .groupby("iou_thresh")
    .agg(
        MOTA=("MOTA", "mean"),
        IDF1=("IDF1", "mean"),
        IDSW=("ID Switches", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum"),
        Matches=("Matches", "sum"),
        GT=("GT", "sum"),
        FPS=("FPS", "mean")
    )
    .reset_index()
)

print("\n")
print("=" * 100)
print("PHASE 2 — CONFIGURATION COMPARISON")
print("=" * 100)

display(
    phase2_summary.sort_values(
        "MOTA",
        ascending=False
    )
)


# ============================================================
# 12. BEST CONFIGURATIONS
# ============================================================

best_mota = phase2_summary.loc[
    phase2_summary["MOTA"].idxmax()
]

best_idf1 = phase2_summary.loc[
    phase2_summary["IDF1"].idxmax()
]

best_ids = phase2_summary.loc[
    phase2_summary["IDSW"].idxmin()
]

print("\n")
print("=" * 100)
print("PHASE 2 WINNERS")
print("=" * 100)

print(
    f"\nBest MOTA:"
    f"  iou_thresh={best_mota['iou_thresh']}"
    f"  → {best_mota['MOTA']:.6f}"
)

print(
    f"Best IDF1:"
    f"  iou_thresh={best_idf1['iou_thresh']}"
    f"  → {best_idf1['IDF1']:.6f}"
)

print(
    f"Fewest ID switches:"
    f"  iou_thresh={best_ids['iou_thresh']}"
    f"  → {int(best_ids['IDSW'])}"
)


# ============================================================
# 13. GPU CLEANUP
# ============================================================

del phase2_df
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("\n")
print("=" * 100)
print("PHASE 2 COMPLETE")
print("=" * 100)

MIXSORT PHASE 2 — IOU THRESHOLD TUNING
Fixed alpha:         0.8
Fixed match_thresh:  0.9
IoU thresholds:      [0.2, 0.3, 0.4]
Sequences:           ['c600', 'c010']
Total runs:          6


SEQUENCE: c600

[1/6] c600 | alpha=0.8 | match_thresh=0.9 | iou_thresh=0.2
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.7598 | IDF1=0.5810 | IDSW=42 | FP=347 | FN=923 | FPS=5.68
GPU memory after cleanup: allocated=0.99 GB, reserved=1.31 GB

[2/6] c600 | alpha=0.8 | match_thresh=0.9 | iou_thresh=0.3
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MOTA=0.7598 | IDF1=0.5810 | IDSW=42 | FP=347 | FN=923 | FPS=5.51
GPU memory after cleanup: allocated=0.99 GB, reserved=1.31 GB

[3/6] c600 | alpha=0.8 | match_thresh=0.9 | iou_thresh=0.4
head channel: 384
missing keys: ['backbone.pos_embed', 'ba

,Sequence,alpha,match_thresh,iou_thresh,MOTA,IDF1,ID Switches,FP,FN,Matches,GT,Predictions,Unique IDs,FPS
3,c010,0.8,0.9,0.2,0.860809,0.512249,67,137,946,7249,8262,7453,77,5.777769
0,c600,0.8,0.9,0.2,0.759795,0.580982,42,347,923,4497,5462,4886,59,5.682720
4,c010,0.8,0.9,0.3,0.860809,0.512249,67,137,946,7249,8262,7453,77,6.056907
1,c600,0.8,0.9,0.3,0.759795,0.580982,42,347,923,4497,5462,4886,59,5.511431
5,c010,0.8,0.9,0.4,0.860809,0.512249,67,137,946,7249,8262,7453,77,6.052204
2,c600,0.8,0.9,0.4,0.759795,0.580982,42,347,923,4497,5462,4886,59,5.837067




PHASE 2 — CONFIGURATION COMPARISON


,iou_thresh,MOTA,IDF1,IDSW,FP,FN,Matches,GT,FPS
0,0.2,0.810302,0.546616,109,484,1869,11746,13724,5.730244
1,0.3,0.810302,0.546616,109,484,1869,11746,13724,5.784169
2,0.4,0.810302,0.546616,109,484,1869,11746,13724,5.944635




PHASE 2 WINNERS

Best MOTA:  iou_thresh=0.2  → 0.810302
Best IDF1:  iou_thresh=0.2  → 0.546616
Fewest ID switches:  iou_thresh=0.2  → 109


PHASE 2 COMPLETE


In [184]:
# ============================================================
# ATHLETIQ — FINAL TUNED MIXSORT EVALUATION
# ============================================================
#
# Tuned configuration:
#   alpha        = 0.8
#   match_thresh = 0.9
#   iou_thresh   = 0.3
#   track_thresh = 0.6
#   track_buffer = 30
#
# Detection:
#   confidence = 0.40
#   NMS IoU    = 0.40
#
# MOT evaluation:
#   matching IoU = 0.50
#
# Sequences:
#   c017, c600, c011, c067, c010
# ============================================================

import sys
import gc
import time
import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
import torch

from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

from ultralytics import YOLO


# ============================================================
# 1. NUMPY 2.x COMPATIBILITY
# ============================================================

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)


# ============================================================
# 2. PATHS
# ============================================================

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

MIXSORT_DIR = (
    ROOT / "ai-engine/third_party/MixSort"
)

VAL_DIR = (
    ROOT
    / "datasets/football/benchmarks/sportsmot/extracted/val"
)

MODEL_PATH = (
    ROOT
    / "ai-engine/models/football_player_detector.pt"
)


# ============================================================
# 3. EXACT VALIDATION SEQUENCES
# ============================================================

SEQUENCES = [
    "v_2QhNRucNC7E_c017",
    "v_G-vNjfx1GGc_c600",
    "v_ITo3sCnpw_k_c011",
    "v_dw7LOz17Omg_c067",
    "v_i2_L4qquVg0_c010",
]


# ============================================================
# 4. FINAL TUNED MIXSORT CONFIG
# ============================================================

args = SimpleNamespace(
    track_thresh=0.6,
    track_buffer=30,
    match_thresh=0.9,
    iou_thresh=0.3,
    alpha=0.8,
    radius=0,
    script="mixformer_deit",
    config="track",
    local_rank=0,
    mot20=False,
)


# ============================================================
# 5. IMPORT MIXSORT
# ============================================================

if str(MIXSORT_DIR) not in sys.path:
    sys.path.insert(0, str(MIXSORT_DIR))

from yolox.mixsort_tracker.mixsort_tracker import MIXTracker


# ============================================================
# 6. LOAD DETECTOR ONCE
# ============================================================

detector = YOLO(
    str(MODEL_PATH)
)


# ============================================================
# 7. GT LOADER
# ============================================================

def load_gt(gt_path):

    gt = defaultdict(list)

    with open(gt_path, "r") as f:

        for line in f:

            parts = line.strip().split(",")

            if len(parts) < 6:
                continue

            frame_id = int(float(parts[0]))
            track_id = int(float(parts[1]))

            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])

            gt[frame_id].append(
                (track_id, x, y, w, h)
            )

    return gt


# ============================================================
# 8. VERIFY ALL SEQUENCES
# ============================================================

print("=" * 80)
print("FINAL TUNED MIXSORT EVALUATION")
print("=" * 80)

print("\nFinal tracker configuration:")
print("  track_thresh :", args.track_thresh)
print("  track_buffer :", args.track_buffer)
print("  match_thresh :", args.match_thresh)
print("  iou_thresh   :", args.iou_thresh)
print("  alpha        :", args.alpha)

print("\nDetector:")
print("  confidence   : 0.40")
print("  NMS IoU      : 0.40")

print("\nMOT evaluation:")
print("  matching IoU : 0.50")

print("\nSequence verification:")

for seq_name in SEQUENCES:

    seq_dir = VAL_DIR / seq_name
    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    if not seq_dir.exists():
        raise FileNotFoundError(
            f"Sequence not found:\n{seq_dir}"
        )

    if not img_dir.exists():
        raise FileNotFoundError(
            f"img1 not found:\n{img_dir}"
        )

    if not gt_path.exists():
        raise FileNotFoundError(
            f"GT not found:\n{gt_path}"
        )

    print(
        f"  ✓ {seq_name}"
    )


# ============================================================
# 9. RESULTS STORAGE
# ============================================================

all_results = []

total_frames = 0
total_objects = 0
total_predictions = 0
total_matches = 0
total_switches = 0
total_fp = 0
total_fn = 0

total_elapsed = 0.0


# ============================================================
# 10. EVALUATE EACH SEQUENCE
# ============================================================

for seq_number, seq_name in enumerate(
    SEQUENCES,
    start=1
):

    print("\n")
    print("=" * 80)
    print(
        f"SEQUENCE {seq_number}/5"
    )
    print(seq_name)
    print("=" * 80)

    seq_dir = VAL_DIR / seq_name

    img_dir = seq_dir / "img1"
    gt_path = seq_dir / "gt" / "gt.txt"

    frames = sorted(
        img_dir.glob("*.jpg")
    )

    gt = load_gt(
        gt_path
    )

    print(
        "Frames:",
        len(frames)
    )

    print(
        "GT rows:",
        sum(len(v) for v in gt.values())
    )

    # --------------------------------------------------------
    # Fresh tracker
    # --------------------------------------------------------

    tracker = MIXTracker(
        args,
        frame_rate=25
    )

    acc = mm.MOTAccumulator(
        auto_id=True
    )

    unique_ids = set()

    start_time = time.time()

    # --------------------------------------------------------
    # Frame loop
    # --------------------------------------------------------

    for frame_idx, frame_path in enumerate(
        frames,
        start=1
    ):

        frame = cv2.imread(
            str(frame_path)
        )

        if frame is None:
            raise RuntimeError(
                f"Failed to read:\n{frame_path}"
            )

        # ----------------------------------------------------
        # Player detector
        # ----------------------------------------------------

        result = detector.predict(
            frame,
            conf=0.40,
            iou=0.40,
            device=0,
            verbose=False
        )[0]

        boxes = (
            result
            .boxes
            .xyxy
            .cpu()
            .numpy()
        )

        scores = (
            result
            .boxes
            .conf
            .cpu()
            .numpy()
        )

        detections = np.column_stack(
            [
                boxes,
                scores
            ]
        ).astype(
            np.float32
        )

        # ----------------------------------------------------
        # MixSort expects H,W
        # ----------------------------------------------------

        img_info = (
            frame.shape[0],
            frame.shape[1]
        )

        img_size = (
            frame.shape[0],
            frame.shape[1]
        )

        # MixSort modifies detections in-place
        targets = tracker.update(
            detections.copy(),
            img_info,
            img_size,
            frame
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        pred_ids = []
        pred_boxes = []

        for target in targets:

            track_id = int(
                target.track_id
            )

            x, y, w, h = map(
                float,
                target.tlwh
            )

            pred_ids.append(
                track_id
            )

            pred_boxes.append(
                [
                    x,
                    y,
                    w,
                    h
                ]
            )

            unique_ids.add(
                track_id
            )

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        gt_entries = gt.get(
            frame_idx,
            []
        )

        gt_ids = []
        gt_boxes = []

        for (
            track_id,
            x,
            y,
            w,
            h
        ) in gt_entries:

            gt_ids.append(
                int(track_id)
            )

            gt_boxes.append(
                [
                    x,
                    y,
                    w,
                    h
                ]
            )

        # ----------------------------------------------------
        # IoU distance matrix
        # ----------------------------------------------------

        if (
            len(gt_boxes) == 0
            and len(pred_boxes) == 0
        ):

            distance = np.empty(
                (0, 0)
            )

        elif len(gt_boxes) == 0:

            distance = np.empty(
                (
                    0,
                    len(pred_boxes)
                )
            )

        elif len(pred_boxes) == 0:

            distance = np.empty(
                (
                    len(gt_boxes),
                    0
                )
            )

        else:

            distance = mm.distances.iou_matrix(
                np.asarray(
                    gt_boxes,
                    dtype=float
                ),
                np.asarray(
                    pred_boxes,
                    dtype=float
                ),
                max_iou=0.5
            )

        # ----------------------------------------------------
        # MOT accumulator
        # ----------------------------------------------------

        acc.update(
            gt_ids,
            pred_ids,
            distance
        )

    # ========================================================
    # TIMING
    # ========================================================

    elapsed = time.time() - start_time

    fps = (
        len(frames) / elapsed
        if elapsed > 0
        else 0
    )

    total_elapsed += elapsed

    # ========================================================
    # METRICS
    # ========================================================

    summary = mm.metrics.create().compute(
        acc,
        metrics=[
            "num_frames",
            "num_objects",
            "num_predictions",
            "num_matches",
            "num_switches",
            "num_false_positives",
            "num_misses",
            "mota",
            "idf1"
        ],
        name=seq_name
    )

    row = summary.iloc[0]

    num_objects = int(
        row["num_objects"]
    )

    num_predictions = int(
        row["num_predictions"]
    )

    num_matches = int(
        row["num_matches"]
    )

    num_switches = int(
        row["num_switches"]
    )

    fp = int(
        row["num_false_positives"]
    )

    fn = int(
        row["num_misses"]
    )

    mota = float(
        row["mota"]
    )

    idf1 = float(
        row["idf1"]
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    all_results.append(
        {
            "Sequence": seq_name,
            "Frames": len(frames),
            "GT Objects": num_objects,
            "Predictions": num_predictions,
            "Matches": num_matches,
            "ID Switches": num_switches,
            "FP": fp,
            "FN": fn,
            "MOTA": mota,
            "IDF1": idf1,
            "Unique IDs": len(unique_ids),
            "FPS": fps,
        }
    )

    # --------------------------------------------------------
    # Accumulate
    # --------------------------------------------------------

    total_frames += len(frames)
    total_objects += num_objects
    total_predictions += num_predictions
    total_matches += num_matches
    total_switches += num_switches
    total_fp += fp
    total_fn += fn

    # ========================================================
    # PRINT
    # ========================================================

    print("\nRESULTS")

    print(
        f"MOTA:          {mota:.6f}"
    )

    print(
        f"IDF1:          {idf1:.6f}"
    )

    print(
        f"ID Switches:   {num_switches}"
    )

    print(
        f"FP:            {fp}"
    )

    print(
        f"FN:            {fn}"
    )

    print(
        f"Matches:       {num_matches}"
    )

    print(
        f"Unique IDs:    {len(unique_ids)}"
    )

    print(
        f"FPS:           {fps:.2f}"
    )

    # --------------------------------------------------------
    # Free tracker memory before next sequence
    # --------------------------------------------------------

    del tracker
    del acc
    del gt

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        print(
            f"GPU memory after cleanup: "
            f"allocated={allocated:.2f} GB, "
            f"reserved={reserved:.2f} GB"
        )


# ============================================================
# 11. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    all_results
)

print("\n")
print("=" * 110)
print("FINAL TUNED MIXSORT — 5 SEQUENCE RESULTS")
print("=" * 110)

display(
    results_df
)


# ============================================================
# 12. MACRO METRICS
# ============================================================

macro_mota = results_df[
    "MOTA"
].mean()

macro_idf1 = results_df[
    "IDF1"
].mean()

average_fps = results_df[
    "FPS"
].mean()


# ============================================================
# 13. ADDITIVE OVERALL MOTA
# ============================================================

overall_mota = (
    1
    - (
        total_fn
        + total_fp
        + total_switches
    )
    / total_objects
)


# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("FINAL TUNED MIXSORT SUMMARY")
print("=" * 80)

print(
    f"Total frames:       {total_frames}"
)

print(
    f"Total GT objects:   {total_objects}"
)

print(
    f"Total predictions:  {total_predictions}"
)

print(
    f"Total matches:      {total_matches}"
)

print(
    f"Total ID switches:  {total_switches}"
)

print(
    f"Total FP:           {total_fp}"
)

print(
    f"Total FN:           {total_fn}"
)

print(
    f"Overall MOTA:       {overall_mota:.6f}"
)

print(
    f"Macro MOTA:         {macro_mota:.6f}"
)

print(
    f"Macro IDF1:         {macro_idf1:.6f}"
)

print(
    f"Average FPS:        {average_fps:.2f}"
)

print("\n")
print("=" * 80)
print("FINAL MIXSORT CONFIG")
print("=" * 80)

print("track_thresh =", args.track_thresh)
print("track_buffer =", args.track_buffer)
print("match_thresh =", args.match_thresh)
print("iou_thresh   =", args.iou_thresh)
print("alpha        =", args.alpha)

print("\nFinal evaluation complete.")

FINAL TUNED MIXSORT EVALUATION

Final tracker configuration:
  track_thresh : 0.6
  track_buffer : 30
  match_thresh : 0.9
  iou_thresh   : 0.3
  alpha        : 0.8

Detector:
  confidence   : 0.40
  NMS IoU      : 0.40

MOT evaluation:
  matching IoU : 0.50

Sequence verification:
  ✓ v_2QhNRucNC7E_c017
  ✓ v_G-vNjfx1GGc_c600
  ✓ v_ITo3sCnpw_k_c011
  ✓ v_dw7LOz17Omg_c067
  ✓ v_i2_L4qquVg0_c010


SEQUENCE 1/5
v_2QhNRucNC7E_c017
Frames: 450
GT rows: 6622
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.

RESULTS
MOTA:          0.854727
IDF1:          0.786027
ID Switches:   42
FP:            465
FN:            455
Matches:       6125
Unique IDs:    42
FPS:           5.27
GPU memory after cleanup: allocated=0.77 GB, reserved=0.95 GB


SEQUENCE 2/5
v_G-vNjfx1GGc_c600
Frames: 500
GT rows: 5462
head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embe

,Sequence,Frames,GT Objects,Predictions,Matches,ID Switches,FP,FN,MOTA,IDF1,Unique IDs,FPS
0,v_2QhNRucNC7E_c017,450,6622,6632,6125,42,465,455,0.854727,0.786027,42,5.269293
1,v_G-vNjfx1GGc_c600,500,5462,4886,4497,42,347,923,0.759795,0.580982,59,5.967711
2,v_ITo3sCnpw_k_c011,776,9736,9853,9143,59,651,534,0.872227,0.609219,53,5.327613
3,v_dw7LOz17Omg_c067,450,6959,6843,6538,28,277,393,0.899698,0.824953,52,4.626475
4,v_i2_L4qquVg0_c010,758,8262,7453,7249,67,137,946,0.860809,0.512249,77,5.488540




FINAL TUNED MIXSORT SUMMARY
Total frames:       2934
Total GT objects:   37041
Total predictions:  35667
Total matches:      33552
Total ID switches:  238
Total FP:           1877
Total FN:           3251
Overall MOTA:       0.855134
Macro MOTA:         0.849451
Macro IDF1:         0.662686
Average FPS:        5.34


FINAL MIXSORT CONFIG
track_thresh = 0.6
track_buffer = 30
match_thresh = 0.9
iou_thresh   = 0.3
alpha        = 0.8

Final evaluation complete.


In [185]:
from pathlib import Path
import yaml

ROOT = Path("/mnt/d/My Projects/AthletiQ/AthletiQ")

MODELS_DIR = ROOT / "ai-engine" / "models"
MIXSORT_DIR = ROOT / "ai-engine" / "third_party" / "MixSort"
CHECKPOINT = MIXSORT_DIR / "pretrained" / "MixFormer_sports_train.pth.tar"

CONFIG_PATH = MODELS_DIR / "mixsort_final.yaml"

config = {
    "tracker": {
        "name": "MixSort",
        "track_thresh": 0.6,
        "track_buffer": 30,
        "match_thresh": 0.9,
        "iou_thresh": 0.3,
        "alpha": 0.8,
    },
    "detector": {
        "confidence": 0.40,
        "nms_iou": 0.40,
    },
    "evaluation": {
        "matching_iou": 0.50,
    },
    "model": {
        "architecture": "mixformer_deit",
        "checkpoint": str(CHECKPOINT),
    },
}

MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Config saved:")
print(CONFIG_PATH)

print("\nCheckpoint:")
print(CHECKPOINT)
print("Exists:", CHECKPOINT.exists())

Config saved:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/models/mixsort_final.yaml

Checkpoint:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/pretrained/MixFormer_sports_train.pth.tar
Exists: True


In [186]:
from pathlib import Path

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

wrapper_path = TRACKING_DIR / "mixsort_tracker.py"

wrapper_code = r'''
from pathlib import Path
from types import SimpleNamespace
import sys
import gc

import numpy as np
import torch
import yaml


class AthletiQMixSort:
    """
    Reusable MixSort tracker for AthletiQ.

    Final configuration:
        track_thresh = 0.6
        track_buffer = 30
        match_thresh = 0.9
        iou_thresh = 0.3
        alpha = 0.8
    """

    def __init__(
        self,
        project_root,
        frame_rate=25,
        config_path=None,
    ):
        self.project_root = Path(project_root)
        self.frame_rate = frame_rate

        self.mixsort_dir = (
            self.project_root
            / "ai-engine"
            / "third_party"
            / "MixSort"
        )

        if config_path is None:
            config_path = (
                self.project_root
                / "ai-engine"
                / "models"
                / "mixsort_final.yaml"
            )

        self.config_path = Path(config_path)

        # ------------------------------------------------------------------
        # NumPy compatibility for older MixSort code
        # ------------------------------------------------------------------
        if not hasattr(np, "float"):
            np.float = float

        if not hasattr(np, "int"):
            np.int = int

        # ------------------------------------------------------------------
        # Add MixSort to Python path
        # ------------------------------------------------------------------
        mixsort_path = str(self.mixsort_dir)

        if mixsort_path not in sys.path:
            sys.path.insert(0, mixsort_path)

        # ------------------------------------------------------------------
        # Load final AthletiQ configuration
        # ------------------------------------------------------------------
        with open(self.config_path, "r") as f:
            cfg = yaml.safe_load(f)

        tracker_cfg = cfg["tracker"]

        self.track_thresh = tracker_cfg["track_thresh"]
        self.track_buffer = tracker_cfg["track_buffer"]
        self.match_thresh = tracker_cfg["match_thresh"]
        self.iou_thresh = tracker_cfg["iou_thresh"]
        self.alpha = tracker_cfg["alpha"]

        # ------------------------------------------------------------------
        # Import MixSort
        # ------------------------------------------------------------------
        from yolox.mixsort_tracker.mixsort_tracker import MIXTracker

        self.MIXTracker = MIXTracker

        # ------------------------------------------------------------------
        # MixSort arguments
        # ------------------------------------------------------------------
        args = SimpleNamespace(
            track_thresh=self.track_thresh,
            track_buffer=self.track_buffer,
            match_thresh=self.match_thresh,
            iou_thresh=self.iou_thresh,
            alpha=self.alpha,
            radius=0,
            script="mixformer_deit",
            config="track",
            local_rank=0,
            mot20=False,
        )

        self.args = args

        # ------------------------------------------------------------------
        # Create tracker
        # ------------------------------------------------------------------
        self.tracker = MIXTracker(
            self.args,
            frame_rate=self.frame_rate
        )

        self.frame_id = 0

    # ----------------------------------------------------------------------
    # Update tracker
    # ----------------------------------------------------------------------
    def update(self, detections, frame):
        """
        Update MixSort with detector outputs.

        Parameters
        ----------
        detections : np.ndarray
            Shape: (N, 5)
            [x1, y1, x2, y2, confidence]

        frame : np.ndarray
            BGR image.

        Returns
        -------
        list of dict
            [
                {
                    "track_id": int,
                    "bbox": [x1, y1, x2, y2],
                    "score": float
                },
                ...
            ]
        """

        if detections is None:
            detections = np.empty((0, 5), dtype=np.float32)

        detections = np.asarray(
            detections,
            dtype=np.float32
        ).copy()

        h, w = frame.shape[:2]

        img_info = (h, w)
        img_size = (h, w)

        online_targets = self.tracker.update(
            detections,
            img_info,
            img_size,
            frame
        )

        results = []

        for target in online_targets:

            tlwh = target.tlwh

            x, y, bw, bh = tlwh

            x1 = float(x)
            y1 = float(y)
            x2 = float(x + bw)
            y2 = float(y + bh)

            results.append({
                "track_id": int(target.track_id),
                "bbox": [x1, y1, x2, y2],
                "score": float(target.score),
            })

        self.frame_id += 1

        return results

    # ----------------------------------------------------------------------
    # Reset tracker
    # ----------------------------------------------------------------------
    def reset(self):
        """
        Create a fresh tracker.
        """

        self.tracker = self.MIXTracker(
            self.args,
            frame_rate=self.frame_rate
        )

        self.frame_id = 0

    # ----------------------------------------------------------------------
    # Cleanup
    # ----------------------------------------------------------------------
    def cleanup(self):
        """
        Release tracker and GPU memory.
        """

        self.tracker = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    # ----------------------------------------------------------------------
    # Configuration
    # ----------------------------------------------------------------------
    def get_config(self):

        return {
            "track_thresh": self.track_thresh,
            "track_buffer": self.track_buffer,
            "match_thresh": self.match_thresh,
            "iou_thresh": self.iou_thresh,
            "alpha": self.alpha,
            "frame_rate": self.frame_rate,
        }
'''

with open(wrapper_path, "w") as f:
    f.write(wrapper_code)

print("Wrapper created:")
print(wrapper_path)

Wrapper created:
/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/core/tracking/mixsort_tracker.py


In [187]:
import sys
import importlib

TRACKING_DIR = ROOT / "ai-engine" / "core" / "tracking"

if str(TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(TRACKING_DIR))

from mixsort_tracker import AthletiQMixSort

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=25
)

print("MixSort loaded successfully.")
print()
print("Configuration:")
print(tracker.get_config())

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.
MixSort loaded successfully.

Configuration:
{'track_thresh': 0.6, 'track_buffer': 30, 'match_thresh': 0.9, 'iou_thresh': 0.3, 'alpha': 0.8, 'frame_rate': 25}


In [188]:
from mixsort_tracker import AthletiQMixSort

tracker = AthletiQMixSort(
    project_root=ROOT,
    frame_rate=25
)

tracks = tracker.update(
    detections,
    frame
)

head channel: 384
missing keys: ['backbone.pos_embed', 'backbone.pos_embed_s', 'backbone.pos_embed_t']
unexpected keys: []
Loading pretrained ViT done.


In [192]:
!grep -n -A5 -B2 "template_imgs =" \
"/mnt/d/My Projects/AthletiQ/AthletiQ/ai-engine/third_party/MixSort/yolox/mixsort_tracker/mixsort_tracker.py"

329-        # self.visualize_box(self.logger,img,dets,"dets")
330-        vit = np.zeros((len(stracks), len(dets)), dtype=np.float64)
331:        template_imgs = [
332-            s.template[0] if isinstance(s.template, tuple) else s.template
333-            for s in stracks
334-        ]
335-        for strack in stracks:
336-            # centered at predicted position
--
342-
343-        # img transform & compute
344:        template_imgs = normalize(
345-            torch.stack(template_imgs).float().div(255),
346-            self.cfg.DATA.MEAN,
347-            self.cfg.DATA.STD,
348-        )
349-        search_imgs = normalize(
